# BUS-BRA: classification under lesion-boundary uncertainty (submission notebook)

Mostafa Bdeir and Hadi Fardan. This notebook walks through the whole workflow: data audit and patient splits, feature extraction, the Logistic Regression and RBF SVM baselines with nested tuning, the boundary-edit stress tests, stability feature selection, patient-level calibration, the saved held-out results, failure analysis, complexity analysis and loading the trained model.

**Three run modes** (set `RUN_MODE` in the next cell):

| Mode | What it does | Needs | Time |
|---|---|---|---|
| `saved` | Shows the saved result tables, recomputes held-out metrics from saved patient probabilities, verifies and loads the trained model | This repository only | seconds |
| `derived` | Also retrains everything from the saved derived feature files (`derived_features/`): baselines, stress tests, stability selection, calibration | This repository only | about 3 minutes |
| `full` | Starts from the raw archive: audits it, extracts features and perturbed features, then runs every stage | `data/raw/BUSBRA.zip` ([Zenodo](https://zenodo.org/records/8231412)) | about 8 minutes |

This copy was executed in `full` mode from the raw archive. Set RUN_MODE = "saved" to run it without the archive.

**Held-out safety.** No mode runs the locked held-out evaluation (`src.heldout_evaluation`) or the final-report script. The held-out numbers here are recomputed from the saved patient probabilities only, and the notebook asserts this at the end. In `full` mode the archive audit (`src.data`) opens every image-mask pair, test ones included, but only to check that the files are intact; it computes no features or predictions. The development stages write to a temporary working copy, never to the repository.

## 1. Setup

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, inspect, time
from pathlib import Path
from importlib.metadata import version
from tempfile import mkdtemp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML, Code, Markdown
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, brier_score_loss, log_loss, confusion_matrix

RUN_MODE = "full"          # "saved" | "derived" | "full"   <-- executed in full mode; use "saved" without the archive
RUN_BENCHMARK = False       # development-only timing benchmark (about 1 minute); needs RUN_MODE != "saved"

# Locate the repository root (the folder that contains src/features.py)
here = Path.cwd().resolve()
root = Path(os.environ.get("BUS_BRA_ROOT", next((p for p in (here, *here.parents) if (p / "src/features.py").is_file()), here))).resolve()
sys.path.insert(0, str(root))
tables = root / "reports/submission_v2/tables"
published = root / "reports/final_step22_run2"
derived = root / "derived_features"
archive = root / "data/raw/BUSBRA.zip"
assert RUN_MODE in ("saved", "derived", "full")
if RUN_MODE == "full" and not archive.is_file():
    raise FileNotFoundError(f"RUN_MODE='full' needs {archive} (download BUSBRA.zip from https://zenodo.org/records/8231412)")

pinned = {"numpy": "2.5.4", "pandas": "3.0.6", "scikit-learn": "1.9.1", "scipy": "1.18.1", "scikit-image": "0.26.0", "joblib": "1.6.0"}
installed = {p: version(p) for p in pinned}
print("Python", sys.version.split()[0], "| mode:", RUN_MODE)
for p, v in pinned.items():
    print(f"  {p:14s} installed {installed[p]:8s} pinned {v}", "" if installed[p] == v else "  <-- differs from the original run")

def show_table(name):
    "Display a saved result table from reports/submission_v2/tables."
    df = pd.read_csv(tables / name)
    display(HTML(df.to_html(index=False)))
    return df

## 2. Data audit and patient-level splits

Every split is by **patient**, never by image, because a patient can have several views. The committed split files are locked: `src.splits` refuses to overwrite them if the dataset disagrees. Development = 851 patients (1,514 images) with five locked outer folds; test = 213 patients (361 images), used only for the one-shot evaluation (the archive audit also checks that their files open).

In [ ]:
images = pd.read_csv(root / "splits/image_splits.csv")
patients = pd.read_csv(root / "splits/patient_splits.csv")

# Integrity checks: unique IDs, one split and one fold per patient, no patient in both development and test
assert images.ID.is_unique and patients.Case.is_unique
assert images.groupby("Case").split.nunique().max() == 1
assert images.groupby("Case").dev_fold.nunique().max() == 1
assert set(patients.loc[patients.split == "development", "Case"]).isdisjoint(patients.loc[patients.split == "test", "Case"])
display(patients.groupby(["split", "Pathology"]).agg(patients=("Case", "size"), images=("n_images", "sum")))
print("Development patients per outer fold:\n", patients[patients.split == "development"].groupby("dev_fold").size().to_string())

## 3. Working copy for the development stages

The stages below run as separate processes (`python -m src.<module>`) in a temporary copy of `src/` and `splits/`, so the repository is never modified. In `derived` mode the saved derived feature files are copied in; in `full` mode the raw archive is copied in and the extraction stages run first.

In [ ]:
work = None
def run(module, *args):
    "Run one pipeline stage in the working copy; print its run time and the stage's own summary lines."
    assert module not in ("src.heldout_evaluation", "src.final_report"), "held-out stages are never run here"
    t0 = time.time()
    proc = subprocess.run([sys.executable, "-m", module, *args], cwd=work, capture_output=True, text=True)
    keys = ("Overall", "MATCHED", "NOT USED", "FROZEN", "Blocking issues", "Patient leakage", "Valid variants", "Extracted")
    summary = [l.strip() for l in proc.stdout.splitlines() if any(k in l for k in keys) and "Saved" not in l]
    print(f"{' '.join([module, *args])}: finished in {time.time()-t0:.0f} s")
    for line in summary[-3:]:
        print("   ", line)
    if proc.returncode:
        print(proc.stderr[-1500:])
        raise RuntimeError(f"{module} failed")

if RUN_MODE != "saved":
    work = Path(mkdtemp(prefix="busbra-work-"))
    shutil.copytree(root / "src", work / "src", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copytree(root / "splits", work / "splits")
    (work / "data/processed").mkdir(parents=True)
    (work / "results/robustness").mkdir(parents=True)
    print("Created a temporary working copy for the stages below (the repository is not modified).")
    if RUN_MODE == "full":
        (work / "data/raw").mkdir(parents=True)
        shutil.copy2(archive, work / "data/raw/BUSBRA.zip")
        run("src.data")
        run("src.splits")                                   # verifies the locked split files against the archive
        run("src.features", "--all-development")            # 23 clean features per development image
        run("src.perturbations", "--all-development")       # original eight-condition masks (geometry audit)
        run("src.perturbed_features", "--all-development")  # features for those masks
        run("src.coherent_perturbations", "--all-development")
        run("src.coherent_features", "--all-development")
    else:
        for name in ("development_features.csv", "development_perturbed_features.csv", "development_coherent_features.csv"):
            shutil.copy2(derived / name, work / "data/processed" / name)
        shutil.copy2(derived / "coherent_geometry_development.csv", work / "results/robustness/coherent_geometry_development.csv")
        print("Derived feature files copied in; extraction stages are skipped.")
else:
    print("RUN_MODE='saved': stages are not run; saved tables are shown instead.")

## 4. Features and the two boundary-edit constructions

Each image and its lesion mask give 23 handcrafted features: six shape, seven intensity and ten masked texture features. The figure compares the **original** edit construction (boundary pixels chosen by distance to the contour, pseudo-random tie-break, which can scatter isolated pixel changes) with the **coherent** construction (one connected patch near a contour anchor). Both target an overlap (IoU) of about 0.95 with the original mask.

In [ ]:
from src.features import read_pair, preprocess_pair, extract_features, FEATURE_COLUMNS
from src.perturbations import partial_boundary_change, mask_iou
from src.coherent_perturbations import coherent_boundary_change

if archive.is_file():
    from zipfile import ZipFile
    image_id = images.loc[images.split.eq("development"), "ID"].sort_values().iloc[3]
    with ZipFile(archive) as z:
        gray, mask = read_pair(z, image_id)
    print("Real development image:", image_id)
else:
    yy, xx = np.ogrid[:160, :200]
    mask = ((yy - 80) / 32) ** 2 + ((xx - 100) / 44) ** 2 <= 1
    gray = (120 + 40 * np.sin(xx / 7.0) + 20 * np.cos(yy / 5.0)).clip(0, 255).astype(np.uint8) * np.ones_like(yy, dtype=np.uint8)
    print("Raw archive not present: using a synthetic ellipse so the demo still runs.")
gray, mask = preprocess_pair(gray, mask)          # grayscale, longest side 256, nearest-neighbor mask
orig_erosion = partial_boundary_change(mask, 0.95, "erosion")
coh_erosion = coherent_boundary_change(mask, 0.95, "erosion")
print(f"IoU original construction {mask_iou(mask, orig_erosion):.3f} | coherent construction {mask_iou(mask, coh_erosion):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, m, title in zip(axes, [mask, orig_erosion, coh_erosion], ["Original mask", "Original-construction erosion", "Coherent erosion"]):
    ax.imshow(gray, cmap="gray"); ax.contour(m, levels=[.5], colors="orange", linewidths=1.2)
    ax.contour(mask, levels=[.5], colors="cyan", linewidths=.6, linestyles="dashed"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

demo = pd.DataFrame([extract_features(gray, mask), extract_features(gray, orig_erosion), extract_features(gray, coh_erosion)],
                    index=["clean", "original erosion", "coherent erosion"]).T
demo["abs change (orig)"] = (demo["original erosion"] - demo["clean"]).abs()
demo["abs change (coherent)"] = (demo["coherent erosion"] - demo["clean"]).abs()
display(demo.round(4))

## 5. Baselines: Logistic Regression and RBF SVM with nested tuning

Outer loop: five locked patient folds. Inner loop: three patient-stratified folds choose the hyperparameters by patient balanced accuracy (ties broken by AUC, then the simpler setting). Scaling and fitting use training folds only, and each patient contributes total weight one across their views. The next cell shows the tuning code itself.

In [ ]:
import src.models as M
# Show the exact source the pipeline runs, read from src/models.py, with syntax highlighting
for fn in (M.patient_image_weights, M.choose_settings):
    display(Markdown(f"Source of `{fn.__name__}` from `src/models.py`:"))
    display(Code(inspect.getsource(fn), language="python"))

In [ ]:
if RUN_MODE != "saved":
    run("src.models")      # Logistic Regression: nested CV, C in {0.01..100}, class balancing off/on
    run("src.svm_model")   # RBF SVM: nested CV, C in {0.1,1,10}, gamma in {scale,0.01,0.1}, balancing off/on
    base = {n: json.loads((work / f"results/modeling/{n}_nested_cv_summary.json").read_text()) for n in ("logreg", "svm")}
    print("Baselines from this run (development, out-of-fold):")
else:
    base = {n: json.loads((tables / f"baseline_summary_{n}.json").read_text()) for n in ("logreg", "svm")}
    print("Saved baselines (development, out-of-fold):")
for n, v in base.items():
    print(f"  {'Logistic Regression' if n == 'logreg' else 'RBF SVM':20s} balanced accuracy {100 * v['balanced_accuracy']:.2f}%   AUC {v['roc_auc']:.4f}")
print()
print("Hyperparameters chosen in each outer fold (saved table):")
_ = show_table("table_chosen_hyperparameters.csv")

## 6. Boundary-sensitivity stress tests (development data, models fixed)

The trained models and thresholds stay fixed, and only the lesion mask changes. Eight conditions: mild and moderate erosion and dilation, and four shifts. Then the same erosion and dilation conditions with the coherent construction.

In [ ]:
if RUN_MODE != "saved":
    run("src.robustness_eval")      # original construction, eight conditions
    run("src.coherent_eval")        # coherent construction, four conditions (post-hoc, labelled as such)
print("Original construction (development, paired changes vs each model's own clean baseline):")
orig = show_table("table_original_eight_conditions.csv")
print("Coherent construction:")
_ = show_table("table_coherent_development.csv")
display(Image(filename=str(root / "reports/submission_v2/figures/06_original_and_coherent.png")))

## 7. Stability feature selection (the mitigation experiment)

Inside every training partition, each feature's stability is its largest mean absolute change under the four coherent edits, divided by its clean standard deviation (lower is better). Subsets of 5, 10 and 15 features are candidates; the model grids are tuned with an objective that weights clean balanced accuracy and the four perturbed balanced accuracies equally. A matched all-23 comparator uses the same grids and objective.

In [ ]:
import src.stable_selection as SS
# The feature-stability ranking, shown from src/stable_selection.py
display(Markdown("Source of `rank_training_features` from `src/stable_selection.py`:"))
display(Code(inspect.getsource(SS.rank_training_features), language="python"))

In [ ]:
if RUN_MODE != "saved":
    run("src.stable_selection")     # freezes the design and audits patient-disjoint training scopes
    run("src.stable_evaluation")    # nested evaluation of the subset arm vs the matched all-23 arm
print("Matched nested development comparison (saved table):")
_ = show_table("clean_summary.csv")
print("Features selected by the stability procedure, number of outer folds (of 5):")
_ = show_table("table_stability_selection_frequency.csv")

## 8. Patient-level probability calibration and the final models

Each outer training pool is split into three patient folds. Each of the three ensemble members keeps one fold aside for calibration: its base model is tuned and fitted on the other two folds, and a sigmoid is then fitted on the clean scores of the patients kept aside. So the calibration patients never take part in tuning or fitting that member, and the outer-validation patients take part in none of it. The three members' probabilities are averaged, and the decision threshold is fixed at 0.5.

In [ ]:
if RUN_MODE != "saved":
    run("src.patient_calibration")   # calibrated nested CV + final development-only ensembles (no held-out access)
    for leftover in ("held_out_evaluation_started.json",):
        assert not (work / "results/robustness/calibration_step20" / leftover).exists()
print("Calibrated outer-fold development performance (saved table):")
_ = show_table("calibrated_clean_summary.csv")
print("Fold-to-fold variation (calibrated development predictions):")
_ = show_table("table_fold_variation.csv")

## 9. Reproduction check

When a mode that retrains is used, the rebuilt numbers must equal the published development numbers. In `saved` mode the same constants are checked against the saved tables.

In [ ]:
published_numbers = {
    "LR baseline BA": 0.8689, "SVM baseline BA": 0.8707,
    "LR all-23 clean BA": 0.8735, "LR subset clean BA": 0.8455, "SVM all-23 clean BA": 0.8707, "SVM subset clean BA": 0.8687,
    "LR calibrated BA": 0.8416, "SVM calibrated BA": 0.8473,
}
src_dir = work / "results/robustness" if work else None
def tbl(path_work, path_saved):
    return pd.read_csv((src_dir / path_work) if work else (tables / path_saved))
cs = tbl("selection_step19/clean_summary.csv", "clean_summary.csv")
cal = tbl("calibration_step20/calibrated_clean_summary.csv", "calibrated_clean_summary.csv")
get = lambda df, m, arm=None: float(df[(df.model == m) & ((df.arm == arm) if arm else True)].balanced_accuracy.iloc[0])
observed = {
    # In retraining modes the baseline comes from the run just made; in saved mode from the saved summaries
    "LR baseline BA": base["logreg"]["balanced_accuracy"] if work else float(json.loads((tables / "baseline_summary_logreg.json").read_text())["balanced_accuracy"]),
    "SVM baseline BA": base["svm"]["balanced_accuracy"] if work else float(json.loads((tables / "baseline_summary_svm.json").read_text())["balanced_accuracy"]),
    "LR all-23 clean BA": get(cs, "logreg", "all_23_matched_tuning"), "LR subset clean BA": get(cs, "logreg", "stability_topk"),
    "SVM all-23 clean BA": get(cs, "svm", "all_23_matched_tuning"), "SVM subset clean BA": get(cs, "svm", "stability_topk"),
    "LR calibrated BA": get(cal, "logreg"), "SVM calibrated BA": get(cal, "svm"),
}
check = pd.DataFrame({"published": published_numbers, "observed": {k: round(v, 4) for k, v in observed.items()}})
check["match"] = (check.published - check.observed).abs() < 5e-4
display(check)
assert check.match.all(), "a development number does not match the published one"
print("All eight checked balanced-accuracy values match the published values (within 0.0005).")

## 10. Locked held-out results, recomputed from saved patient probabilities

The one-shot held-out evaluation was run once during the study with frozen models. Here nothing is refitted or re-predicted: the metrics are recomputed from the saved per-patient probabilities (`patient_failure_audit.csv`), and the paired comparison counts patients.

In [ ]:
audit = pd.read_csv(published / "patient_failure_audit.csv")
rows = []
for name in ("lr", "svm"):
    y, p = audit.actual, audit[f"{name}_probability"]
    tn, fp, fn, tp = confusion_matrix(y, p >= .5).ravel()
    rows.append({"model": name.upper(), "BA %": 100 * balanced_accuracy_score(y, p >= .5), "sensitivity %": 100 * tp / (tp + fn),
                 "specificity %": 100 * tn / (tn + fp), "AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p),
                 "log loss": log_loss(y, p), "FN": fn, "FP": fp})
display(HTML(pd.DataFrame(rows).round(4).to_html(index=False)))
both = int((audit.lr_error & audit.svm_error).sum()); lr_only = int((audit.lr_error & ~audit.svm_error).sum())
svm_only = int((~audit.lr_error & audit.svm_error).sum()); neither = int((~audit.lr_error & ~audit.svm_error).sum())
print(f"misclassified by both {both} | LR only {lr_only} | SVM only {svm_only} | correct for both {neither} | patients {len(audit)}")
assert (both, lr_only, svm_only, neither) == (30, 3, 0, 180)
print("Held-out coherent boundary sensitivity (saved condition table):")
display(HTML(pd.read_csv(published / "heldout_robustness_table.csv")[["model", "condition", "delta_balanced_accuracy", "flip_rate", "new_false_negatives", "recovered_false_negatives", "delta_roc_auc"]].round(4).to_html(index=False)))

## 11. Failure analysis and complexity analysis (development data)

Errors by lesion size (tertiles of the mean area fraction of a patient's images) and by number of views, from calibrated out-of-fold predictions; and a timing comparison of one tuned LR fit against one tuned SVM fit on a development fold, with no tuning inside the timing.

In [ ]:
fb = pd.read_csv(tables / "table_failure_breakdown.csv")
fb["Group"] = fb["Group"].astype(str)
for factor, order in (("Lesion size", ["small", "middle", "large"]), ("Views per patient", ["1", "2"])):
    part = fb[fb["Factor"] == factor]
    wide = part.pivot_table(index=["Model", "Class"], columns="Group", values="Error rate (%)", aggfunc="first")[order]
    print(f"Error rate (%) by {factor.lower()}:")
    display(wide)
print("Complexity (single development fold; comparable single-model operations, not the three-member ensembles):")
_ = show_table("table_complexity.csv")
if RUN_BENCHMARK and RUN_MODE != "saved":
    shutil.copytree(root / "analysis", work / "analysis", dirs_exist_ok=True)
    proc = subprocess.run([sys.executable, "-m", "analysis.development_analysis", "--root", str(work), "--repeats", "10"],
                          cwd=work, capture_output=True, text=True)
    print(proc.stdout[-2500:] or proc.stderr[-1500:])

## 12. The trained model

`models/frozen_calibrated_models.joblib` holds the final three-member calibrated LR and SVM ensembles. The cell verifies its bytes and the library versions against the recorded protocol *before* loading it, then predicts for one **development** patient. (No test patients are used.)

In [ ]:
import joblib
from src.patient_calibration import predict_ensemble
from src.models import load_development

model_path = root / "models/frozen_calibrated_models.joblib"
protocol = json.loads((root / "models/final_evaluation_protocol.json").read_text())
summary = json.loads((root / "models/calibration_summary.json").read_text())
expected = "627ef9df92e4b12b475a12c844b53e82e9dd7fefe05a9f07c3b94ca7c77f3586"   # digest recorded in the original report manifest
with model_path.open("rb") as f:
    actual = hashlib.file_digest(f, "sha256").hexdigest()
assert actual == expected == protocol["model_artifact_sha256"], "model checksum mismatch"
print("SHA-256 matches the original frozen model:", actual)
if work:   # the full run regenerates the model in the working copy (src.patient_calibration writes it here)
    regenerated = work / "results/robustness/calibration_step20/frozen_calibrated_models.joblib"
    if regenerated.is_file():
        with regenerated.open("rb") as f:
            rebuilt = hashlib.file_digest(f, "sha256").hexdigest()
        print("Regenerated model in the working copy:", rebuilt)
        print("Equals the frozen model hash:", rebuilt == expected)
    else:
        print("No regenerated model bundle found in the working copy.")
mismatch = {p: (version(p), v) for p, v in summary["versions"].items() if version(p) != v}
print("Library versions differ from the recorded ones:", mismatch or "none")
if mismatch:
    print("(Loading may still work; install the pinned requirements to match exactly.)")

bundle = joblib.load(model_path)
dev = load_development(derived / "development_features.csv", root / "splits/image_splits.csv")
sample = dev.loc[dev.Case.eq(dev.Case.iloc[0])]
print("Development patient", int(sample.Case.iloc[0]), "| true label:", sample.Pathology.iloc[0], "| views:", len(sample))
for name, ensemble in bundle["models"].items():
    print(name, [m["settings"] for m in ensemble["members"]])
    display(predict_ensemble(ensemble, sample)[["actual", "n_images", "prob_malignant", "predicted"]])

## 12b. Interpretability: logistic regression coefficients

The three LR ensemble members each have a base logistic regression on the 23 standardized features. The table lists their coefficients (log-odds per standard deviation), the median across members, how many members agree with the median's sign, and how often the stability procedure selected the feature. It is built from the frozen model by `analysis/build_report_tables.py` (development artifacts only) and shown in the report as Section 4.5.1.

In [ ]:
# Positive values push towards malignant; rows are sorted by the absolute median coefficient.
_ = show_table("table_lr_coefficients.csv")

## 13. Safety check and summary

The notebook never ran the locked held-out evaluation. Summary of what it showed:

- In the retraining modes, all eight checked development balanced accuracies match the published values (section 9); in `saved` mode the same check runs against the saved tables.
- On the 213 held-out patients the calibrated SVM reached balanced accuracy 81.85% against 80.42% for LR; the paired difference is small (interval reaches zero), LR has the better AUC and probability losses, and the SVM missed 20 of 68 malignant patients at the fixed 0.5 threshold.
- Under the original boundary-edit construction the SVM lost less accuracy than LR; under the coherent construction both lost little. Stability selection did not help.

In [ ]:
# run() refuses the held-out modules; also confirm no held-out marker or output exists in the working copy
if work:
    assert not list(work.rglob("held_out_evaluation_started.json")), "a held-out marker was created"
    assert not (work / "results/robustness/heldout_step21").exists(), "held-out outputs exist"
print("No held-out stage was run. Mode used:", RUN_MODE)

## Appendix. Complete pipeline source

A static copy of every pipeline file, shown as text so it reads the same in any viewer. These cells are not run; the code runs through the stages above.

**`analysis/build_report.py`**

```python
"""Assemble the final report: fill table placeholders, render HTML, print a compact PDF.

    python analysis/build_report.py --dir reports/submission_v2

Needs the tables written by build_report_tables.py and the figures in <dir>/figures.
PDF rendering uses a locally installed Microsoft Edge or Google Chrome in headless mode (optional).
"""
import argparse
import base64
import re
import subprocess
from pathlib import Path

import markdown

TABLES = {
    "chosen_hyperparameters": "table_chosen_hyperparameters.md",
    "fold_variation": "table_fold_variation.md",
    "original_eight_conditions": "table_original_eight_conditions.md",
    "coherent_development": "table_coherent_development.md",
    "stability_frequency": "table_stability_selection_frequency.md",
    "failure_breakdown": "table_failure_breakdown.md",
    "complexity": "table_complexity.md",
    "threshold_fixed": "table_threshold_fixed.md",
    "threshold_policy": "table_threshold_policy.md",
    "lr_coefficients": "table_lr_coefficients.md",
}
AMERICAN = {"characterise": "characterize", "normalisation": "normalization", "quantisation": "quantization",
            "regularised": "regularized", "favourable": "favorable", "behaviour": "behavior",
            "generalisation": "generalization", "neighbour": "neighbor", "standardiser": "standardizer",
            "labelled": "labeled", "Regularised": "Regularized"}
CSS = """
@page { size: A4; margin: 16mm 15mm; }
body { font-family: 'Segoe UI', Arial, sans-serif; font-size: 10.2pt; line-height: 1.38; color: #1b2733; max-width: 100%; }
h1 { font-size: 20pt; margin: 0 0 4px; color: #14304a; }
h2 { font-size: 14pt; margin: 20px 0 6px; color: #14527a; border-bottom: 1px solid #c9d6e2; padding-bottom: 2px; }
h3 { font-size: 11.5pt; margin: 14px 0 4px; color: #14527a; }
p { margin: 5px 0 7px; }
h1, h2, h3, h4 { break-after: avoid; page-break-after: avoid; }
thead { display: table-header-group; }
tr { break-inside: avoid; page-break-inside: avoid; }
table { border-collapse: collapse; margin: 8px 0 10px; font-size: 8.8pt; width: 100%; }
th, td { border: 1px solid #c9d6e2; padding: 3px 6px; text-align: left; vertical-align: top; }
th { background: #eaf1f7; }
img { max-width: 78%; display: block; margin: 8px auto 2px; page-break-inside: avoid; }
code { font-family: Consolas, monospace; font-size: 8.8pt; background: #f1f4f7; padding: 0 2px; }
a { color: #14527a; }
table:last-of-type, table.keep { break-inside: avoid; page-break-inside: avoid; }
"""


def find_browser():
    for path in (r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
                 r"C:\Program Files\Microsoft\Edge\Application\msedge.exe",
                 r"C:\Program Files\Google\Chrome\Application\chrome.exe"):
        if Path(path).is_file():
            return path
    return None


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--dir", type=Path, default=Path("reports/submission_v2"))
    args = parser.parse_args()
    folder = args.dir.resolve()
    text = (folder / "BUS_BRA_Report_template.md").read_text(encoding="utf-8")
    for key, filename in TABLES.items():
        table = (folder / "tables" / filename).read_text(encoding="utf-8").strip()
        token = "{{T:" + key + "}}"
        if token not in text:
            raise SystemExit(f"Placeholder missing in template: {token}")
        text = text.replace(token, table)
    if "{{" in text:
        raise SystemExit("Unfilled placeholder remains: " + re.findall(r"\{\{[^}]*\}\}", text)[0])
    counter = iter(range(1, 1000))   # number table captions in order of appearance
    text = re.sub(r"(?m)^Table: (.*)$", lambda m: f"*Table {next(counter)}.* {m.group(1)}", text)
    for british, american in AMERICAN.items():
        text = text.replace(british, american)
    (folder / "BUS_BRA_Report.md").write_text(text, encoding="utf-8")
    body = markdown.markdown(text, extensions=["tables", "sane_lists"])

    def keep_small(match):  # short tables (10 body rows or fewer, little text) stay on one page; long ones may split
        table = match.group(0)
        rows = table.count("<tr>") - 1
        chars = len(re.sub(r"<[^>]+>", "", table))
        return table.replace("<table>", '<table class="keep">', 1) if rows <= 10 and chars <= 1000 else table

    body = re.sub(r"<table>.*?</table>", keep_small, body, flags=re.S)

    def embed(match):  # embed figures so the PDF never races image loading
        image = folder / match.group(1)
        data = base64.b64encode(image.read_bytes()).decode("ascii")
        return f'src="data:image/png;base64,{data}"'

    body = re.sub(r'src="(figures/[^"]+\.png)"', embed, body)
    html = f"<!doctype html><html><head><meta charset='utf-8'><title>BUS-BRA final report</title><style>{CSS}</style></head><body>{body}</body></html>"
    (folder / "BUS_BRA_Report.html").write_text(html, encoding="utf-8")
    browser = find_browser()
    if browser is None:
        print("No Edge/Chrome found; HTML written, print it to PDF manually.")
        return
    pdf = folder / "BUS_BRA_Report.pdf"
    subprocess.run([browser, "--headless", "--disable-gpu", "--no-pdf-header-footer",
                    f"--print-to-pdf={pdf}", (folder / "BUS_BRA_Report.html").as_uri()],
                   check=True, timeout=180)
    print("PDF:", pdf, round(pdf.stat().st_size / 1e6, 2), "MB")


if __name__ == "__main__":
    main()
```

**`analysis/build_report_tables.py`**

```python
"""Build the development-only report tables from saved rebuild outputs.

Run from the project root of a development rebuild (after src.patient_calibration):
    python analysis/build_report_tables.py --root . --out reports/submission_v2/tables

Reads only development outputs. It never touches held-out files. Writes CSV tables plus a JSON of
the numbers quoted in the report, so every figure in the text has one source.
"""
import argparse
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def md(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for row in df.itertuples(index=False):
        lines.append("| " + " | ".join(str(v) for v in row) + " |")
    return "\n".join(lines)


def load_dev_script(root: Path):
    path = Path(__file__).with_name("development_analysis.py")
    sys.path.insert(0, str(root))
    spec = importlib.util.spec_from_file_location("development_analysis", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def lr_coefficient_table(root: Path, tables: Path) -> pd.DataFrame:
    """Base LR coefficients (per SD of each feature) of the three frozen LR ensemble members.

    Reads only the frozen development-trained model file and the saved stability-selection table.
    The 1-input sigmoid calibrators are skipped (their slopes are checked to be positive).
    """
    import joblib
    sys.path.insert(0, str(root))
    bundle = joblib.load(root / "models/frozen_calibrated_models.joblib")
    members = bundle["models"]["logreg"]["members"]
    cols = {}
    for k, member in enumerate(members, start=1):
        pipe = member["base_model"].pipeline
        assert [name for name, _ in pipe.steps] == ["scaler", "classifier"], "StandardScaler must come before LR"
        assert list(pipe.named_steps["classifier"].classes_) == [0, 1]
        cols[f"Member {k}"] = pd.Series(pipe.named_steps["classifier"].coef_.ravel(), index=list(member["base_model"].columns))
    for k, member in enumerate(members, start=1):  # positive slope: signs carry through to the final probability
        sigmoid = member["calibrator"].named_steps["sigmoid"]
        scale = member["calibrator"].named_steps["scaler"].scale_[0]
        slope = float(sigmoid.coef_.ravel()[0] / scale)
        print(f"Member {k} calibrator slope on the raw score: {slope:.4f}")
        assert slope > 0, "calibrator slope must be positive"
    t = pd.DataFrame(cols)
    t.index.name = "Feature"
    members_only = list(t.columns)
    t["Median"] = t[members_only].median(axis=1)
    t["Members agreeing with the median's sign"] = (np.sign(t[members_only]).eq(np.sign(t["Median"]), axis=0)).sum(axis=1)
    stab = pd.read_csv(tables / "table_stability_selection_frequency.csv").set_index("Feature")
    t["Folds selected by stability (of 5)"] = stab["LR folds selected (of 5)"].reindex(t.index).astype(int)
    t = t.reindex(t["Median"].abs().sort_values(ascending=False).index).reset_index()
    return t.round({c: 4 for c in members_only + ["Median"]})


def write_lr_coefficient_tables(root: Path, out: Path) -> None:
    """Full table as CSV, and the top eight by |median| as a Markdown table for the report."""
    full = lr_coefficient_table(root, out)
    full.to_csv(out / "table_lr_coefficients.csv", index=False)
    top = full.head(8)
    short = pd.DataFrame({
        "Feature": top["Feature"],
        "Member 1": top["Member 1"].map("{:+.2f}".format), "Member 2": top["Member 2"].map("{:+.2f}".format),
        "Member 3": top["Member 3"].map("{:+.2f}".format), "Median": top["Median"].map("{:+.2f}".format),
        "Signs agreeing (of 3)": top["Members agreeing with the median's sign"],
        "Stability folds (of 5)": top["Folds selected by stability (of 5)"]})
    (out / "table_lr_coefficients.md").write_text(md(short), encoding="utf-8")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", type=Path, default=Path.cwd())
    parser.add_argument("--out", type=Path, default=None)
    parser.add_argument("--repeats", type=int, default=10)
    parser.add_argument("--lr-coefficients-only", action="store_true",
                        help="write only table_lr_coefficients.csv (needs just the frozen model file)")
    args = parser.parse_args()
    root = args.root.resolve()
    out = (args.out or root / "reports/submission_v2/tables").resolve()
    out.mkdir(parents=True, exist_ok=True)
    if args.lr_coefficients_only:
        write_lr_coefficient_tables(root, out)
        print("wrote", out / "table_lr_coefficients.csv")
        return
    rb = root / "results/robustness"
    nums = {}
    names = {"logreg": "LR", "svm": "SVM"}
    cond_names = {"erosion_mild": "Mild erosion", "dilation_mild": "Mild dilation",
                  "erosion_moderate": "Moderate erosion", "dilation_moderate": "Moderate dilation",
                  "shift_left_moderate": "Shift left", "shift_right_moderate": "Shift right",
                  "shift_up_moderate": "Shift up", "shift_down_moderate": "Shift down"}
    order = list(cond_names)

    # 1. Original eight-condition development stress test
    o = pd.read_csv(rb / "robustness_paired_condition_summary.csv")
    o["order"] = o.condition.map(order.index)
    o = o.sort_values(["model", "order"])
    t = pd.DataFrame({
        "Model": o.model.map(names), "Condition": o.condition.map(cond_names),
        "Patients": o.eligible_patients,
        "Delta BA (pp)": (o.delta_balanced_accuracy * 100).map("{:+.2f}".format),
        "Flip (%)": (o.flip_rate * 100).map("{:.1f}".format),
        "New FN": o.new_false_negatives, "Recovered FN": o.recovered_false_negatives})
    t.to_csv(out / "table_original_eight_conditions.csv", index=False)
    (out / "table_original_eight_conditions.md").write_text(md(t), encoding="utf-8")
    nums["original"] = {f"{r.model}|{r.condition}": {"delta_pp": round(r.delta_balanced_accuracy * 100, 2),
                                                      "flip_pct": round(r.flip_rate * 100, 1)}
                        for r in o.itertuples()}

    # 2. Coherent development comparison
    c = pd.read_csv(rb / "coherent_step16/coherent_condition_summary.csv")
    c["order"] = c.condition.map(order.index)
    c = c.sort_values(["model", "order"])
    tc = pd.DataFrame({
        "Model": c.model.map(names), "Condition": c.condition.map(cond_names),
        "Delta BA (pp)": (c.delta_balanced_accuracy * 100).map("{:+.2f}".format),
        "Flip (%)": (c.flip_rate * 100).map("{:.1f}".format),
        "New FN": c.new_false_negatives, "Recovered FN": c.recovered_false_negatives})
    tc.to_csv(out / "table_coherent_development.csv", index=False)
    (out / "table_coherent_development.md").write_text(md(tc), encoding="utf-8")
    nums["coherent"] = {f"{r.model}|{r.condition}": round(r.delta_balanced_accuracy * 100, 2) for r in c.itertuples()}

    # 3. Stability subset selection frequencies (union of the two classifiers' identical rankings)
    f = pd.read_csv(rb / "selection_step19/feature_selection_frequency.csv")
    s = f[f.arm == "stability_topk"].pivot(index="feature", columns="model", values="selected_outer_folds")
    s = s.reset_index().sort_values(["logreg", "svm", "feature"], ascending=[False, False, True])  # ties by name
    ts = s.rename(columns={"feature": "Feature", "logreg": "LR folds selected (of 5)", "svm": "SVM folds selected (of 5)"})
    ts.to_csv(out / "table_stability_selection_frequency.csv", index=False)
    (out / "table_stability_selection_frequency.md").write_text(md(ts), encoding="utf-8")
    nums["never_selected"] = s.loc[(s.logreg == 0) & (s.svm == 0), "feature"].tolist()

    # 4. Chosen hyperparameters of the original baselines (per outer fold)
    rows = []
    for m in ("logreg", "svm"):
        d = pd.read_csv(root / f"results/modeling/{m}_nested_cv_folds.csv")
        for r in d.itertuples():
            rows.append({"Model": names[m], "Outer fold": r.outer_fold, "C": f"{r.C:g}",
                         "gamma": getattr(r, "gamma", "-") if m == "svm" else "-",
                         "Class balancing": "on" if r.balance else "off",
                         "Outer-fold BA (%)": f"{r.balanced_accuracy * 100:.1f}"})
    th = pd.DataFrame(rows)
    th.to_csv(out / "table_chosen_hyperparameters.csv", index=False)
    (out / "table_chosen_hyperparameters.md").write_text(md(th), encoding="utf-8")

    # 5. Fold variation and failure breakdown on calibrated development out-of-fold predictions
    dev_mod = load_dev_script(root)
    development = dev_mod.load_development(root / "data/processed/development_features.csv",
                                           root / "splits/image_splits.csv")
    oof = pd.read_csv(rb / "calibration_step20/calibrated_clean_oof.csv")
    from sklearn.metrics import balanced_accuracy_score
    fold = pd.DataFrame([{"model": m, "fold": k, "BA": balanced_accuracy_score(g.actual, g.predicted)}
                         for (m, k), g in oof.groupby(["model", "outer_fold"])])
    fv = fold.groupby("model").BA.agg(["mean", "std", "min", "max"]).reset_index()
    tf = pd.DataFrame({"Model": fv.model.map(names), "Mean BA (%)": (fv["mean"] * 100).map("{:.2f}".format),
                       "SD across folds (pp)": (fv["std"] * 100).map("{:.2f}".format),
                       "Min (%)": (fv["min"] * 100).map("{:.1f}".format),
                       "Max (%)": (fv["max"] * 100).map("{:.1f}".format)})
    tf.to_csv(out / "table_fold_variation.csv", index=False)
    (out / "table_fold_variation.md").write_text(md(tf), encoding="utf-8")
    nums["fold"] = {r.model: {"mean": round(r.mean * 100, 2), "sd": round(r.std * 100, 2)} for r in fv.itertuples()}

    meta = development.groupby("Case").agg(area=("shape_area_fraction", "mean"), views=("ID", "size"),
                                           pathology=("Pathology", "first"))
    meta["Lesion size tertile"] = pd.qcut(meta.area, 3, labels=["small", "middle", "large"])
    frame = oof.merge(meta.reset_index(), on="Case", validate="many_to_one")
    assert len(frame) == len(oof) and (frame.actual == frame.pathology.eq("malignant")).all()
    frame["error"] = frame.actual.ne(frame.predicted)
    parts = []
    for group, label in (("Lesion size tertile", "Lesion size"), ("views", "Views per patient")):
        g = frame.groupby(["model", "pathology", group], observed=True).error.agg(["size", "sum", "mean"]).reset_index()
        g.columns = ["model", "pathology", "level", "patients", "errors", "rate"]
        g["Factor"] = label
        parts.append(g)
    fb = pd.concat(parts)
    tfb = pd.DataFrame({"Model": fb.model.map(names), "Class": fb.pathology.str.capitalize(),
                        "Factor": fb.Factor, "Group": fb.level.astype(str), "Patients": fb.patients,
                        "Errors": fb.errors, "Error rate (%)": (fb.rate * 100).map("{:.1f}".format)})
    tfb = tfb.sort_values(["Factor", "Model", "Class", "Group"], ascending=[True, True, True, True])
    tfb.to_csv(out / "table_failure_breakdown.csv", index=False)
    (out / "table_failure_breakdown.md").write_text(md(tfb), encoding="utf-8")

    # 6. Complexity benchmark (development fold 0, selected settings, no tuning)
    bench = dev_mod.benchmark(root, development, args.repeats, 0)
    bench.to_csv(out / "table_complexity_raw.csv", index=False)
    rows = []
    for m in ("logreg", "svm"):
        b = bench[bench.model == m]
        fit = b[b.stage.str.startswith("fit")].iloc[0]
        pred = b[b.stage.str.startswith("serial")].iloc[0]
        rows.append({"Model": names[m], "Fit, no tuning (ms, median [IQR])": f"{fit.median_ms:.1f} [{fit.q1_ms:.1f}-{fit.q3_ms:.1f}]",
                     "Predict one patient (ms)": f"{pred.median_ms:.2f} [{pred.q1_ms:.2f}-{pred.q3_ms:.2f}]",
                     "Compressed model (KB)": f"{fit.compressed_model_bytes / 1024:.1f}",
                     "Tuning candidates": 10 if m == "logreg" else 18})
    tcx = pd.DataFrame(rows)
    tcx.to_csv(out / "table_complexity.csv", index=False)
    (out / "table_complexity.md").write_text(md(tcx), encoding="utf-8")
    nums["complexity"] = {r["Model"]: r for r in rows}
    nums["benchmark_setup"] = {"fit_images": int(bench.fit_images.iloc[0]), "fit_patients": int(bench.fit_patients.iloc[0]),
                               "repeats": int(bench.repeats.iloc[0])}

    # 7. Feature extraction time per image (development, excludes ZIP reads)
    from time import perf_counter
    from zipfile import ZipFile
    from src.features import extract_features, preprocess_pair, read_pair
    ids = development.sort_values("ID").ID.head(40).tolist()
    with ZipFile(root / "data/raw/BUSBRA.zip") as z:
        pairs = [read_pair(z, i) for i in ids]
    samples = []
    for _ in range(args.repeats + 1):
        t0 = perf_counter()
        for im, mk in pairs:
            extract_features(*preprocess_pair(im, mk))
        samples.append((perf_counter() - t0) / len(pairs) * 1000)
    nums["extract_ms_per_image"] = round(float(np.median(samples[1:])), 2)

    # 8. Logistic regression coefficients of the frozen development-trained ensemble
    write_lr_coefficient_tables(root, out)

    (out / "report_numbers.json").write_text(json.dumps(nums, indent=2, default=str), encoding="utf-8")
    print("tables written to", out)


if __name__ == "__main__":
    main()
```

**`analysis/development_analysis.py`**

```python
"""Optional development analyses for the BUS-BRA submission."""

import argparse
import io
import platform
import statistics
from pathlib import Path
from time import perf_counter
from zipfile import ZipFile

import joblib
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

from src.features import extract_features, preprocess_pair, read_pair
from src.models import fit_model, load_development, patient_predictions
from src.svm_model import fit_svm, patient_svm_predictions


def timing(operation, repeats):
    operation()
    samples = []
    for _ in range(repeats):
        start = perf_counter()
        operation()
        samples.append((perf_counter() - start) * 1000)
    quartiles = statistics.quantiles(samples, n=4, method="inclusive")
    return statistics.median(samples), quartiles[0], quartiles[2]


def selected_settings(path, model, fold):
    row = pd.read_csv(path).set_index("outer_fold").loc[fold]
    settings = {"C": float(row.C), "balance": str(row.balance).lower() == "true"}
    if model == "svm":
        settings["gamma"] = row.gamma if row.gamma == "scale" else float(row.gamma)
    return settings


def benchmark(root, development, repeats, fold):
    training = development.loc[development.dev_fold.ne(fold)]
    validation = development.loc[development.dev_fold.eq(fold)]
    assert set(training.Case).isdisjoint(validation.Case)
    samples = [rows for _, rows in validation.groupby("Case")][:20]
    records = []
    for name, fit, predict in (("logreg", fit_model, patient_predictions),
                               ("svm", fit_svm, patient_svm_predictions)):
        source = root / f"results/modeling/{name}_nested_cv_folds.csv"
        settings = selected_settings(source, name, fold)
        model = fit(training, **settings)
        blob = io.BytesIO()
        joblib.dump(model, blob, compress=3)
        for stage, operation in (
            ("fit including scaling, excluding tuning", lambda: fit(training, **settings)),
            ("serial patient prediction including aggregation",
             lambda: [predict(model, patient) for patient in samples]),
        ):
            divisor = len(samples) if stage.startswith("serial") else 1
            median, q1, q3 = timing(operation, repeats)
            records.append({"model": name, "stage": stage, "median_ms": median / divisor,
                            "q1_ms": q1 / divisor, "q3_ms": q3 / divisor,
                            "fit_images": len(training), "fit_patients": training.Case.nunique(),
                            "timed_patients": len(samples), "repeats": repeats,
                            "compressed_model_bytes": len(blob.getvalue())})
        if name == "logreg":
            from src.features import FEATURE_COLUMNS
            coefficients = pd.Series(model.named_steps["classifier"].coef_.ravel(),
                                     index=FEATURE_COLUMNS, name="standardized_coefficient")
            print("\nLR coefficients for this development fold and its selected settings")
            print(coefficients.loc[coefficients.abs().nlargest(8).index].to_string())
    archive = root / "data/raw/BUSBRA.zip"
    if archive.is_file():
        ids = validation.sort_values("ID").ID.head(20).tolist()
        with ZipFile(archive) as dataset:
            pairs = [read_pair(dataset, image_id) for image_id in ids]
        def extract_batch():
            for image, mask in pairs:
                extract_features(*preprocess_pair(image, mask))
        median, q1, q3 = timing(extract_batch, repeats)
        print("\nPreprocess and extract, excluding ZIP reads and image decoding (ms/image)")
        print({"median": median / len(ids), "q1": q1 / len(ids), "q3": q3 / len(ids)})
    return pd.DataFrame(records)


def failure_breakdown(development, predictions):
    metadata = development.groupby("Case").agg(
        area=("shape_area_fraction", "mean"), views=("ID", "size"),
        pathology=("Pathology", "first"))
    metadata["area_tertile"] = pd.qcut(metadata.area, 3, labels=["small", "middle", "large"])
    frame = predictions.merge(metadata.reset_index(), on="Case", validate="many_to_one")
    if len(frame) != len(predictions) or not (frame.actual == frame.pathology.eq("malignant")).all():
        raise ValueError("OOF patient coverage or diagnoses disagree with development features")
    if frame.duplicated(["model", "Case"]).any():
        raise ValueError("Expected one OOF prediction per patient and model")
    frame["error"] = frame.actual.ne(frame.predicted)
    for group in ("area_tertile", "views"):
        print(f"\nDevelopment OOF errors by {group} and pathology")
        print(frame.groupby(["model", "pathology", group], observed=True).error
              .agg(patients="size", errors="sum", error_rate="mean").to_string())
    fold_metrics = [{"model": model, "fold": fold,
                     "BA": balanced_accuracy_score(rows.actual, rows.predicted)}
                    for (model, fold), rows in frame.groupby(["model", "outer_fold"])]
    print("\nClean OOF balanced accuracy across outer folds")
    print(pd.DataFrame(fold_metrics).groupby("model").BA.agg(["mean", "std", "count"]).to_string())


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", type=Path, default=Path.cwd())
    parser.add_argument("--repeats", type=int, default=10)
    parser.add_argument("--fold", type=int, choices=range(5), default=0)
    args = parser.parse_args()
    if args.repeats < 10:
        parser.error("--repeats must be at least 10")
    root = args.root.resolve()
    development = load_development(root / "data/processed/development_features.csv",
                                   root / "splits/image_splits.csv")
    print(f"{platform.platform()} | CPU: {platform.processor()}")
    print(benchmark(root, development, args.repeats, args.fold).to_string(index=False))
    path = root / "results/robustness/calibration_step20/calibrated_clean_oof.csv"
    if path.is_file():
        failure_breakdown(development, pd.read_csv(path))


if __name__ == "__main__":
    main()
```

**`analysis/threshold_analysis.py`**

```python
"""Exploratory, development-only operating-threshold analysis (mitigation for missed cancers).

    python analysis/threshold_analysis.py --oof reports/submission_v2/tables/calibrated_clean_oof.csv \
        --tables reports/submission_v2/tables --figures reports/submission_v2/figures

Uses ONLY the calibrated out-of-fold development predictions (851 patients per model).
The held-out test set is not read: its threshold (0.5) was frozen before it was scored,
and choosing a new one after seeing test errors would contaminate that evaluation.

Outputs
  table_threshold_fixed.{csv,md}   sensitivity/specificity at fixed thresholds
  table_threshold_policy.{csv,md}  target-sensitivity thresholds chosen on the other four outer folds and
                                   applied to the fifth. Descriptive only: the probabilities for those four
                                   folds came from ensembles trained partly on the fifth fold's patients, so
                                   this is NOT an unbiased estimate of a threshold-selection policy.

Optional --manifest splits/image_splits.csv validates the predictions against the locked split:
patients must be exactly the development patients, with matching labels and outer folds.
Thresholds are reported at 4 decimals; rounding them further can change the result.
  threshold_numbers.json           numbers quoted in the report
  08_threshold_curves.{png,svg}    sensitivity and specificity versus threshold
"""
import argparse
import json
import math
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NAMES = {"logreg": "Logistic Regression", "svm": "RBF SVM"}
SHORT = {"logreg": "LR", "svm": "SVM"}
FIXED = (0.5, 0.4, 0.3, 0.2)
TARGETS = (0.85, 0.90, 0.95)


def md(df):
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    out += ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(out)


def rates(y, p, t):
    pred = p >= t
    tp = int((pred & (y == 1)).sum()); fn = int((~pred & (y == 1)).sum())
    tn = int((~pred & (y == 0)).sum()); fp = int((pred & (y == 0)).sum())
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    return {"sensitivity": sens, "specificity": spec, "balanced_accuracy": (sens + spec) / 2,
            "tp": tp, "fn": fn, "tn": tn, "fp": fp}


def threshold_for_target(y, p, target):
    """Largest threshold whose sensitivity is at least `target` (maximises specificity under the constraint)."""
    pos = np.sort(p[y == 1])[::-1]
    k = math.ceil(target * len(pos))
    return float(pos[k - 1])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--oof", type=Path, required=True)
    ap.add_argument("--tables", type=Path, required=True)
    ap.add_argument("--figures", type=Path, required=True)
    ap.add_argument("--manifest", type=Path, default=None, help="locked splits/image_splits.csv to validate against")
    a = ap.parse_args()
    oof = pd.read_csv(a.oof)
    required = {"model", "Case", "actual", "prob_malignant", "outer_fold"}
    if not required.issubset(oof.columns) or oof.duplicated(["model", "Case"]).any():
        raise SystemExit("Expected one calibrated out-of-fold prediction per patient and model")
    if set(oof.model) != {"logreg", "svm"}:
        raise SystemExit(f"Expected exactly the models logreg and svm, found {sorted(set(oof.model))}")
    probs = pd.to_numeric(oof.prob_malignant, errors="coerce").to_numpy(float)
    if not np.isfinite(probs).all() or (probs < 0).any() or (probs > 1).any():
        raise SystemExit("Probabilities must be finite and within [0, 1]")
    if not oof.actual.isin([0, 1]).all():
        raise SystemExit("Labels must be 0 or 1")
    if oof.groupby("model").Case.apply(frozenset).nunique() != 1:
        raise SystemExit("Both models must cover the same patients")
    if a.manifest is not None:
        man = pd.read_csv(a.manifest)
        dev = man[man.split.eq("development")].groupby("Case").agg(pathology=("Pathology", "first"), fold=("dev_fold", "first"))
        test_cases = set(man.loc[man.split.eq("test"), "Case"])
        for m, g in oof.groupby("model"):
            if set(g.Case) != set(dev.index):
                raise SystemExit(f"{m}: predictions do not cover exactly the development patients")
            if set(g.Case) & test_cases:
                raise SystemExit(f"{m}: predictions include test patients")
            ref = dev.loc[g.Case]
            if not (g.actual.to_numpy() == ref.pathology.eq("malignant").astype(int).to_numpy()).all():
                raise SystemExit(f"{m}: labels differ from the locked manifest")
            if not (g.outer_fold.to_numpy() == ref.fold.to_numpy()).all():
                raise SystemExit(f"{m}: outer folds differ from the locked manifest")
        print("Validated against the locked manifest: development patients only, labels and folds match")
    nums = {"patients": int(oof.Case.nunique()), "malignant": int(oof.loc[oof.model.eq("svm"), "actual"].sum())}

    fixed_rows, policy_rows = [], []
    for m in ("logreg", "svm"):
        g = oof[oof.model.eq(m)]
        y, p = g.actual.to_numpy(int), g.prob_malignant.to_numpy(float)
        for t in FIXED:
            r = rates(y, p, t)
            fixed_rows.append({"Model": SHORT[m], "Threshold": f"{t:.1f}", "Sensitivity (%)": f"{100*r['sensitivity']:.1f}",
                               "Specificity (%)": f"{100*r['specificity']:.1f}", "BA (%)": f"{100*r['balanced_accuracy']:.1f}",
                               "Missed cancers (FN)": r["fn"], "False alarms (FP)": r["fp"]})
            nums[f"{m}|fixed|{t}"] = {k: (round(100 * v, 2) if isinstance(v, float) else v) for k, v in r.items()}
        for target in TARGETS:
            preds, chosen = [], []
            for k in sorted(g.outer_fold.unique()):
                rest, held = g[g.outer_fold.ne(k)], g[g.outer_fold.eq(k)]
                t = threshold_for_target(rest.actual.to_numpy(int), rest.prob_malignant.to_numpy(float), target)
                chosen.append(t)
                preds.append(pd.DataFrame({"y": held.actual.to_numpy(int), "pred": held.prob_malignant.to_numpy(float) >= t}))
            P = pd.concat(preds)
            r = rates(P.y.to_numpy(), P.pred.to_numpy().astype(float), 0.5)
            full_t = threshold_for_target(y, p, target)
            policy_rows.append({"Model": SHORT[m], "Target sensitivity": f"{int(target*100)}%",
                                "Thresholds chosen per fold": f"{min(chosen):.4f}-{max(chosen):.4f}",
                                "Sensitivity (%)": f"{100*r['sensitivity']:.1f}", "Specificity (%)": f"{100*r['specificity']:.1f}",
                                "BA (%)": f"{100*r['balanced_accuracy']:.1f}", "Missed cancers (FN)": r["fn"],
                                "False alarms (FP)": r["fp"], "Threshold from all development": f"{full_t:.4f}"})
            nums[f"{m}|policy|{target}"] = {**{k: (round(100 * v, 2) if isinstance(v, float) else v) for k, v in r.items()},
                                           "fold_thresholds": [round(c, 4) for c in chosen], "full_development_threshold": round(full_t, 4)}

    a.tables.mkdir(parents=True, exist_ok=True)
    tf, tp_ = pd.DataFrame(fixed_rows), pd.DataFrame(policy_rows)
    tf.to_csv(a.tables / "table_threshold_fixed.csv", index=False)
    (a.tables / "table_threshold_fixed.md").write_text(md(tf), encoding="utf-8")
    tp_.to_csv(a.tables / "table_threshold_policy.csv", index=False)
    (a.tables / "table_threshold_policy.md").write_text(md(tp_), encoding="utf-8")
    (a.tables / "threshold_numbers.json").write_text(json.dumps(nums, indent=2), encoding="utf-8")

    # Figure: sensitivity and specificity against threshold, per model
    grid = np.linspace(0.01, 0.99, 197)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
    for ax, m in zip(axes, ("logreg", "svm")):
        g = oof[oof.model.eq(m)]
        y, p = g.actual.to_numpy(int), g.prob_malignant.to_numpy(float)
        sens = [rates(y, p, t)["sensitivity"] for t in grid]
        spec = [rates(y, p, t)["specificity"] for t in grid]
        ax.plot(grid, sens, color="#E4572E", lw=2, label="Sensitivity")
        ax.plot(grid, spec, color="#1F6F78", lw=2, ls="--", label="Specificity")
        t90 = nums[f"{m}|policy|0.9"]["full_development_threshold"]
        ax.axvline(0.5, color="#5B6B70", lw=1, ls=":")
        ax.axvline(t90, color="#12343B", lw=1)
        ax.text(0.505, 0.03, "fixed 0.5", fontsize=8, color="#5B6B70")
        ax.text(t90 + 0.005, 0.10, f"90% target\n{t90:.4f}", fontsize=8, color="#12343B")
        ax.set_title(NAMES[m], fontsize=11)
        ax.set_xlabel("Probability threshold (patient is called malignant at or above it)")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=.25)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    axes[0].set_ylabel("Rate (development, calibrated out-of-fold)")
    axes[0].legend(loc="lower left", fontsize=9, frameon=False, bbox_to_anchor=(0.52, 0.12))
    fig.suptitle("Development-only operating-threshold analysis (not applied to the test set)", fontsize=11)
    fig.tight_layout()
    a.figures.mkdir(parents=True, exist_ok=True)
    fig.savefig(a.figures / "08_threshold_curves.png", dpi=150)
    fig.savefig(a.figures / "08_threshold_curves.svg")
    print(tf.to_string(index=False)); print(); print(tp_.to_string(index=False))


if __name__ == "__main__":
    main()
```

**`src/coherent_eval.py`**

```python
"""Step 16: separately labelled, post-hoc coherent-boundary sensitivity analysis.

No search, calibration, threshold selection, raw images or test feature access.
All preprocessing/model fits use clean outer-training data via Step 14 helpers.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd

from .coherent_features import COHERENT_CONDITIONS, verify_coherent_geometry
from .features import FEATURE_COLUMNS
from .models import ROOT, NUM_OUTER_FOLDS, fit_model, load_development
from .perturbed_features import INFO_COLUMNS
from .robustness_eval import (
    MODEL_NAMES, ID_COLUMNS, _predict, compare_one_condition, load_fold_settings,
    parse_boolean_column, summarize_pair, validate_clean_reference,
)
from .svm_model import fit_svm

PROTOCOL = "step16_posthoc_spatially_coherent"
ORIGINAL_PROTOCOL = "step14_original_boundary"
PREDICTION_FILE = "coherent_paired_patient_predictions.csv"


def load_coherent(path: Path, clean: pd.DataFrame) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype={"ID": str}, low_memory=False)
    if list(frame.columns) != list(INFO_COLUMNS) + list(FEATURE_COLUMNS):
        raise ValueError("Coherent table schema/order differs from Step 15")
    keys = list(ID_COLUMNS) + ["condition", "valid"]
    if frame[keys].isna().any().any() or frame.duplicated(["ID", "condition"]).any():
        raise ValueError("Missing metadata or duplicate coherent ID-condition")
    if (len(frame) != len(clean) * len(COHERENT_CONDITIONS)
            or set(frame.condition) != set(COHERENT_CONDITIONS)
            or set(frame.ID) != set(clean.ID)
            or not frame.groupby("ID").condition.nunique().eq(4).all()):
        raise ValueError("Coherent table must cover every development image and four conditions")
    metadata = clean.set_index("ID")
    for col in ID_COLUMNS[1:]:
        if not np.array_equal(frame[col], metadata.loc[frame.ID, col]):
            raise ValueError(f"Coherent metadata differs from locked clean manifest: {col}")
    frame["valid"] = parse_boolean_column(frame.valid)
    # Step 15 completed all four conditions; missing vectors are a failed checkpoint.
    if not frame.valid.all():
        raise ValueError("Step 15 coherent checkpoint contains unavailable variants")
    if not np.isfinite(frame[list(FEATURE_COLUMNS)].to_numpy(float)).all():
        raise ValueError("Nonfinite coherent features")
    if not frame.actual_iou.between(0, 1, inclusive="neither").all():
        raise ValueError("Invalid coherent IoU")
    return frame


def validate_manifest_isolation(manifest_path: Path, dev: pd.DataFrame) -> None:
    # Inspect split identifiers only; no held-out labels/images/features are loaded.
    manifest = pd.read_csv(manifest_path, usecols=["ID", "Case", "split", "dev_fold"])
    if manifest[["ID", "Case", "split"]].isna().any().any() or manifest.ID.duplicated().any():
        raise ValueError("Invalid locked manifest identifiers")
    if set(dev.Case) & set(manifest.loc[manifest.split.ne("development"), "Case"]):
        raise ValueError("Leakage: patient shared by development and held-out split")
    if not dev.dev_fold.isin(range(NUM_OUTER_FOLDS)).all():
        raise ValueError("Outer folds must be integers 0..4")


def load_original(path: Path, dev: pd.DataFrame) -> pd.DataFrame:
    """Validate Step 14's patient reference before any original/coherent comparison."""
    frame = pd.read_csv(path)
    required = {"model", "condition", "Case", "actual", "outer_fold", "n_images",
                "eligible", "mean_iou", "clean_score", "perturbed_score",
                "clean_predicted", "perturbed_predicted"}
    if not required.issubset(frame):
        raise ValueError("Step 14 patient reference schema missing columns")
    frame = frame.loc[frame.condition.isin(COHERENT_CONDITIONS)].copy()
    if set(frame.model) != set(MODEL_NAMES) or frame.duplicated(["model", "condition", "Case"]).any():
        raise ValueError("Invalid Step 14 model/patient coverage")
    frame["eligible"] = parse_boolean_column(frame.eligible)
    patients = dev.groupby("Case").agg(Pathology=("Pathology", "first"),
                                      outer_fold=("dev_fold", "first"), n_images=("ID", "size"))
    for model in MODEL_NAMES:
        for condition in COHERENT_CONDITIONS:
            part = frame.loc[frame.model.eq(model) & frame.condition.eq(condition)]
            if len(part) != len(patients) or set(part.Case) != set(patients.index):
                raise ValueError("Step 14 patient coverage differs from development")
            ref = patients.loc[part.Case]
            for col in ("outer_fold", "n_images"):
                if not np.array_equal(part[col], ref[col]):
                    raise ValueError(f"Step 14 metadata mismatch: {col}")
            if not np.array_equal(part.actual, ref.Pathology.eq("malignant").astype(int)):
                raise ValueError("Step 14 patient label mismatch")
            if not np.isfinite(part.clean_score).all() or not part.clean_predicted.isin([0, 1]).all():
                raise ValueError("Invalid Step 14 clean predictions")
            eligible = part.loc[part.eligible]
            if (not np.isfinite(eligible[["perturbed_score", "mean_iou"]]).all().all()
                    or not eligible.perturbed_predicted.isin([0, 1]).all()
                    or not eligible.mean_iou.between(0, 1).all()):
                raise ValueError("Invalid Step 14 eligible predictions")
    return frame


def require_same_clean(left: pd.DataFrame, right: pd.DataFrame) -> None:
    a = left.sort_values("Case").reset_index(drop=True)
    b = right.sort_values("Case").reset_index(drop=True)
    for col in ("Case", "actual", "outer_fold", "n_images", "clean_predicted"):
        if not np.array_equal(a[col], b[col]):
            raise ValueError(f"Step 14 paired clean mismatch: {col}")
    if not np.allclose(a.clean_score, b.clean_score, rtol=1e-6, atol=1e-7):
        raise ValueError("Step 14 paired clean score mismatch")


def protocol_comparison(original: pd.DataFrame, coherent: pd.DataFrame) -> pd.DataFrame:
    """Both methods use the intersection of eligible patients, within each model/condition."""
    rows = []
    for model in MODEL_NAMES:
        for condition in COHERENT_CONDITIONS:
            old = original.loc[original.model.eq(model) & original.condition.eq(condition)]
            new = coherent.loc[coherent.model.eq(model) & coherent.condition.eq(condition)]
            require_same_clean(old, new)
            common = set(old.loc[old.eligible, "Case"]) & set(new.loc[new.eligible, "Case"])
            for label, part in ((ORIGINAL_PROTOCOL, old), (PROTOCOL, new)):
                row = summarize_pair(part.loc[part.Case.isin(common)], model=model, condition=condition)
                row.update(protocol=label, common_eligible_patients=len(common),
                           patients_outside_common=len(old) - len(common))
                rows.append(row)
    return pd.DataFrame(rows)


def model_comparison(predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for condition in COHERENT_CONDITIONS:
        lr = predictions.loc[predictions.model.eq("logreg") & predictions.condition.eq(condition)]
        svm = predictions.loc[predictions.model.eq("svm") & predictions.condition.eq(condition)]
        common = set(lr.loc[lr.eligible, "Case"]) & set(svm.loc[svm.eligible, "Case"])
        a = summarize_pair(lr.loc[lr.Case.isin(common)], model="logreg", condition=condition)
        b = summarize_pair(svm.loc[svm.Case.isin(common)], model="svm", condition=condition)
        rows.append({"protocol": PROTOCOL, "condition": condition,
                     "common_eligible_patients": len(common),
                     "logreg_delta_balanced_accuracy": a["delta_balanced_accuracy"],
                     "svm_delta_balanced_accuracy": b["delta_balanced_accuracy"],
                     "svm_minus_logreg_delta_balanced_accuracy": b["delta_balanced_accuracy"] - a["delta_balanced_accuracy"],
                     "logreg_flip_rate": a["flip_rate"], "svm_flip_rate": b["flip_rate"],
                     "logreg_new_false_negatives": a["new_false_negatives"],
                     "svm_new_false_negatives": b["new_false_negatives"]})
    return pd.DataFrame(rows)


def sha256(path: Path) -> str:
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def run(clean_path: Path, manifest_path: Path, coherent_path: Path,
        geometry_path: Path, modeling_dir: Path, original_dir: Path, output_dir: Path) -> dict:
    paths = {"clean": clean_path, "manifest": manifest_path, "coherent_features": coherent_path,
             "coherent_geometry": geometry_path,
             "step14_predictions": original_dir / "robustness_paired_patient_predictions.csv"}
    for name in MODEL_NAMES:
        for kind in ("folds", "predictions"):
            paths[f"{name}_{kind}"] = modeling_dir / f"{name}_nested_cv_{kind}.csv"
    filenames = [PREDICTION_FILE, "coherent_condition_summary.csv", "coherent_outer_fold_metrics.csv",
                 "step14_vs_step16_common_patient_summary.csv", "coherent_model_comparison.csv",
                 "coherent_summary.json"]
    if output_dir.resolve() in (original_dir.resolve(), modeling_dir.resolve()):
        raise ValueError("Step 16 requires a separate output directory")
    targets = [output_dir / name for name in filenames]
    if any(p.exists() for p in targets):
        raise FileExistsError("Step 16 output exists; choose a new --output directory")
    if {p.resolve() for p in paths.values()} & {p.resolve() for p in targets}:
        raise ValueError("Output would overwrite a locked input")
    fingerprints = {key: {"path": str(path.resolve()), "sha256": sha256(path)} for key, path in paths.items()}
    dev = load_development(clean_path, manifest_path)
    validate_manifest_isolation(manifest_path, dev)
    coherent = load_coherent(coherent_path, dev)
    verify_coherent_geometry(coherent, pd.read_csv(geometry_path, dtype={"ID": str}), dev)
    original = load_original(paths["step14_predictions"], dev)
    settings = {name: load_fold_settings(paths[f"{name}_folds"], name) for name in MODEL_NAMES}
    print(f"STEP 16 — POST-HOC COHERENT BOUNDARY SENSITIVITY\n"
          f"Development: {len(dev)} images | {dev.Case.nunique()} patients | {len(coherent)} vectors", flush=True)
    predictions, folds = [], []
    for name in MODEL_NAMES:
        baselines, model_parts = [], []
        for fold in range(NUM_OUTER_FOLDS):
            train, val = dev.loc[dev.dev_fold.ne(fold)], dev.loc[dev.dev_fold.eq(fold)]
            if set(train.Case) & set(val.Case):
                raise ValueError("Leakage: outer training/validation patients overlap")
            model = (fit_model(train, **settings[name][fold]) if name == "logreg"
                     else fit_svm(train, **settings[name][fold]))
            baselines.append(_predict(model, val, name, fold))
            for condition in COHERENT_CONDITIONS:
                variant = coherent.loc[coherent.dev_fold.eq(fold) & coherent.condition.eq(condition)]
                rec = compare_one_condition(model, name, val, variant, fold)
                model_parts.append(rec)
                folds.append(summarize_pair(rec, model=name, condition=condition, fold=fold))
            print(f"{name.upper()} locked outer fold {fold}: clean training only", flush=True)
        validate_clean_reference(pd.concat(baselines, ignore_index=True), paths[f"{name}_predictions"], name)
        predictions.extend(model_parts)
        print(f"{name.upper()} saved clean OOF: MATCHED", flush=True)
    predictions = pd.concat(predictions, ignore_index=True)
    if (len(predictions) != dev.Case.nunique() * 4 * 2
            or predictions.duplicated(["model", "condition", "Case"]).any()):
        raise ValueError("Missing/duplicate coherent patient predictions")
    comparison = protocol_comparison(original, predictions)
    summary = pd.DataFrame([summarize_pair(part, model=name, condition=condition)
                            for (name, condition), part in predictions.groupby(["model", "condition"])])
    predictions.insert(0, "protocol", PROTOCOL)
    summary.insert(0, "protocol", PROTOCOL)
    fold_table = pd.DataFrame(folds)
    fold_table.insert(0, "protocol", PROTOCOL)
    models = model_comparison(predictions)
    # Refuse publication if inputs changed during fitting.
    if any(sha256(path) != fingerprints[key]["sha256"] for key, path in paths.items()):
        raise RuntimeError("Locked inputs changed during evaluation")
    result = {"protocol": PROTOCOL, "analysis_status": "descriptive post-hoc sensitivity analysis",
              "development_images": len(dev), "development_patients": int(dev.Case.nunique()),
              "coherent_feature_vectors": len(coherent), "patient_rows_saved": len(predictions),
              "conditions": list(COHERENT_CONDITIONS), "model_names": list(MODEL_NAMES),
              "clean_reference_oof": {name: "MATCHED" for name in MODEL_NAMES},
              "step14_clean_predictions": "MATCHED", "coherent_geometry": "MATCHED",
              "held_out_features_images_predictions_used": False,
              "held_out_manifest_identifiers_checked_for_isolation": True,
              "training": "clean outer-training only; recorded inner settings; no retuning",
              "thresholds": {"logreg_mean_probability": 0.5, "svm_mean_decision_margin": 0.0},
              "protocol_comparison_population": "condition-specific intersection of eligible patients",
              "original_step14_outputs": "read-only; separately labelled step14_original_boundary",
              "translations": "unchanged Step 14 definitions; not reevaluated here",
              "inputs": fingerprints, "locked_settings": settings,
              "environment": {"python": platform.python_version(), **{pkg: version(pkg) for pkg in
                              ("numpy", "pandas", "scikit-learn", "scipy")}},
              "results": summary.to_dict("records")}
    output_dir.mkdir(parents=True, exist_ok=True)
    for frame, filename in zip((predictions, summary, fold_table, comparison, models), filenames[:-1]):
        frame.to_csv(output_dir / filename, index=False, float_format="%.12g")
    (output_dir / filenames[-1]).write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print(summary[["model", "condition", "eligible_patients", "delta_balanced_accuracy", "flip_rate", "new_false_negatives"]].to_string(index=False))
    print(f"Step 14 clean predictions: MATCHED | Held-out evaluation: NOT USED\nSaved: {output_dir}")
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--clean", type=Path, default=ROOT / "data/processed/development_features.csv")
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--coherent", type=Path, default=ROOT / "data/processed/development_coherent_features.csv")
    parser.add_argument("--geometry", type=Path, default=ROOT / "results/robustness/coherent_geometry_development.csv")
    parser.add_argument("--modeling", type=Path, default=ROOT / "results/modeling")
    parser.add_argument("--original", type=Path, default=ROOT / "results/robustness")
    parser.add_argument("--output", type=Path, default=ROOT / "results/robustness/coherent_step16")
    args = parser.parse_args()
    run(args.clean, args.manifest, args.coherent, args.geometry, args.modeling, args.original, args.output)


if __name__ == "__main__":
    main()
```

**`src/coherent_features.py`**

```python
"""Step 15: post-hoc coherent-boundary feature extraction, development-only.

Run from project root:
    python -m src.coherent_features --limit 12
    python -m src.coherent_features --all-development

Alternative protocol: four coherent erosion/dilation conditions only.
Original Step 13 feature tables, original Step 14 results and translation
conditions are neither read as modified data nor overwritten.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd

from .coherent_perturbations import coherent_boundary_change
from .features import FEATURE_COLUMNS, PROJECT_ROOT, extract_features, preprocess_pair, read_pair
from .perturbations import TARGET_IOUS, mask_iou
from .perturbed_features import INFO_COLUMNS

COHERENT_CONDITIONS = (
    "erosion_mild", "dilation_mild", "erosion_moderate", "dilation_moderate"
)


def coherent_image_rows(image_id: str, case: int, pathology: str, dev_fold: int,
                        image: np.ndarray, original_mask: np.ndarray) -> list[dict]:
    """Generate four alternative records from one UNCHANGED image/mask pair."""
    if image.shape != original_mask.shape or image.ndim != 2:
        raise ValueError("Image and original mask must have matching 2D shape")
    original_image_copy = image.copy()
    original_mask_copy = original_mask.copy()
    rows = []
    for condition in COHERENT_CONDITIONS:
        kind, severity = condition.split("_")
        changed = coherent_boundary_change(original_mask, TARGET_IOUS[severity], kind)
        if np.array_equal(changed, original_mask):
            raise AssertionError("A coherent perturbation left the mask unchanged")
        changed_pixels = int(np.logical_xor(original_mask, changed).sum())
        row = {
            "ID": str(image_id), "Case": int(case), "Pathology": str(pathology),
            "split": "development", "dev_fold": int(dev_fold), "condition": condition,
            "valid": True, "failure_reason": "",
            "actual_iou": mask_iou(original_mask, changed),
            "changed_pixels": changed_pixels,
            "area_ratio": float(changed.sum() / original_mask.sum()),
            "amount_pixels": changed_pixels,
        }
        row.update(extract_features(image, changed))
        rows.append(row)
    if not np.array_equal(image, original_image_copy):
        raise AssertionError("Original ultrasound image was mutated")
    if not np.array_equal(original_mask, original_mask_copy):
        raise AssertionError("Original lesion mask was mutated")
    return rows


def verify_coherent_geometry(generated: pd.DataFrame, geometry: pd.DataFrame,
                             clean: pd.DataFrame) -> None:
    """Reject drift between Step 14.3 geometry and newly extracted features."""
    keys = ["ID", "condition"]
    required = set(keys + ["valid", "actual_iou", "changed_pixels",
                           "circularity_clean", "circularity_coherent_protocol"])
    if not required.issubset(geometry):
        raise ValueError(f"Coherent geometry report lacks: {required - set(geometry)}")
    if generated.duplicated(keys).any() or geometry.duplicated(keys).any():
        raise ValueError("Duplicate ID/condition in alternative geometry audit")
    if geometry["valid"].astype(str).str.lower().ne("true").any():
        raise ValueError("Step 14.3 reported an invalid coherent mask")
    merged = generated.merge(geometry[list(required)], on=keys, how="outer",
                             suffixes=("_new", "_audit"), indicator=True,
                             validate="one_to_one")
    if len(merged) != len(generated) or not merged["_merge"].eq("both").all():
        raise ValueError("Geometry audit and extracted features have different coverage")
    for a, b in (("actual_iou_new", "actual_iou_audit"),
                 ("changed_pixels_new", "changed_pixels_audit"),
                 ("shape_circularity", "circularity_coherent_protocol")):
        if not np.allclose(merged[a].to_numpy(float), merged[b].to_numpy(float),
                           rtol=1e-8, atol=1e-8):
            raise ValueError(f"Coherent geometry mismatch: {a} versus {b}")
    reference = clean.set_index("ID")["shape_circularity"]
    old_circ = reference.loc[merged["ID"]].to_numpy(float)
    if not np.allclose(old_circ, merged["circularity_clean"].to_numpy(float),
                       rtol=1e-8, atol=1e-8):
        raise ValueError("Coherent audit's original circularity differs from clean features")


def extract_coherent_development(zip_path: Path, manifest_path: Path, clean_path: Path,
                                 geometry_path: Path, output_path: Path,
                                 summary_path: Path, limit: int | None = 12) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_path, dtype={"ID": str})
    needed = {"ID", "Case", "Pathology", "split", "dev_fold"}
    if not needed.issubset(manifest) or manifest.ID.duplicated().any():
        raise ValueError("Invalid frozen image split manifest")
    dev = manifest.loc[manifest.split.eq("development")].sort_values("ID")
    if limit is not None:
        if limit < 1:
            raise ValueError("--limit must be positive")
        dev = dev.head(limit)
    if dev.empty or dev.dev_fold.isna().any() or not dev.dev_fold.between(0, 4).all():
        raise ValueError("Invalid development subset")

    clean = pd.read_csv(clean_path, dtype={"ID": str})
    if clean.ID.duplicated().any() or not set(dev.ID).issubset(set(clean.ID)):
        raise ValueError("Missing or duplicate original clean features")
    indexed_clean = clean.set_index("ID")
    audit = pd.read_csv(geometry_path, dtype={"ID": str})
    audit = audit.loc[audit.ID.isin(set(dev.ID))].copy()
    if len(audit) != len(dev) * len(COHERENT_CONDITIONS):
        raise ValueError("Geometry audit lacks four conditions per selected image")

    rows = []
    with ZipFile(zip_path) as archive:
        for idx, item in enumerate(dev.itertuples(index=False), 1):
            cached_row = indexed_clean.loc[item.ID]
            if (int(cached_row.Case) != int(item.Case)
                    or str(cached_row.Pathology) != str(item.Pathology)
                    or str(cached_row.split) != "development"
                    or int(cached_row.dev_fold) != int(item.dev_fold)):
                raise ValueError(f"Clean features/manifest disagree for {item.ID}")
            image, mask = preprocess_pair(*read_pair(archive, item.ID))
            recomputed_clean = extract_features(image, mask)
            cached_features = cached_row[list(FEATURE_COLUMNS)].to_numpy(float)
            if not np.allclose(cached_features, list(recomputed_clean.values()),
                               rtol=1e-8, atol=1e-8):
                raise ValueError(f"Original features differ for {item.ID}")
            rows.extend(coherent_image_rows(item.ID, item.Case, item.Pathology,
                                            item.dev_fold, image, mask))
            if idx % 100 == 0:
                print(f"Extracted coherent features for {idx}/{len(dev)} images...", flush=True)
    result = pd.DataFrame(rows, columns=list(INFO_COLUMNS) + list(FEATURE_COLUMNS))
    if len(result) != len(dev) * 4 or result.duplicated(["ID", "condition"]).any():
        raise AssertionError("Missing/duplicate alternative perturbation rows")
    if not result.split.eq("development").all() or not result.valid.all():
        raise AssertionError("Unexpected split or invalid alternative perturbation")
    if not np.isfinite(result[list(FEATURE_COLUMNS)].to_numpy(float)).all():
        raise AssertionError("Nonfinite alternative feature")
    verify_coherent_geometry(result, audit, clean)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, float_format="%.12g")
    summary = {
        "status": "post-hoc coherent boundary sensitivity features; original protocol preserved",
        "development_images": int(len(dev)), "development_patients": int(dev.Case.nunique()),
        "coherent_conditions": list(COHERENT_CONDITIONS),
        "feature_vectors": int(len(result)), "valid_feature_vectors": int(result.valid.sum()),
        "numeric_features": len(FEATURE_COLUMNS), "test_images_seen": 0,
        "clean_features_reproduced": True, "coherent_geometry_audit_matched": True,
        "original_step14_outputs_untouched": True,
    }
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print("BUS-BRA — POST-HOC COHERENT FEATURE EXTRACTION")
    print("=" * 51)
    print(f"Development images: {len(dev)} | Patients: {dev.Case.nunique()} | Features: {len(FEATURE_COLUMNS)}")
    print(f"Valid alternative feature vectors: {len(result)}/{len(dev) * 4}")
    print("Cached clean features and Step 14.3 geometry: MATCHED")
    print("Original Step 14 robustness results: UNCHANGED")
    print("Held-out test: NOT USED")
    print(f"Saved: {output_path}")
    return result


def main() -> None:
    p = argparse.ArgumentParser(description="Generate alternative coherent-boundary feature table")
    p.add_argument("--zip", type=Path, default=PROJECT_ROOT / "data/raw/BUSBRA.zip")
    p.add_argument("--manifest", type=Path, default=PROJECT_ROOT / "splits/image_splits.csv")
    p.add_argument("--clean", type=Path, default=PROJECT_ROOT / "data/processed/development_features.csv")
    p.add_argument("--geometry", type=Path, default=None)
    p.add_argument("--output", type=Path, default=None)
    p.add_argument("--summary", type=Path, default=None)
    p.add_argument("--limit", type=int, default=12)
    p.add_argument("--all-development", action="store_true")
    args = p.parse_args()
    if args.all_development and args.limit != 12:
        p.error("Do not combine --all-development with a custom --limit")
    limit = None if args.all_development else args.limit
    root = PROJECT_ROOT / "results/robustness"
    geom = args.geometry or root / ("coherent_geometry_development.csv" if limit is None
                                   else f"coherent_geometry_smoke_{limit}.csv")
    out = args.output or PROJECT_ROOT / "data/processed" / (
        "development_coherent_features.csv" if limit is None else f"coherent_features_smoke_{limit}.csv")
    summary = args.summary or root / ("coherent_feature_summary.json" if limit is None
                                    else f"coherent_feature_smoke_{limit}_summary.json")
    for path in (args.zip, args.manifest, args.clean, geom):
        if not path.is_file():
            raise FileNotFoundError(path)
    extract_coherent_development(args.zip, args.manifest, args.clean, geom, out, summary, limit)


if __name__ == "__main__":
    main()
```

**`src/coherent_perturbations.py`**

```python
"""Exploratory Step 14.2: spatially localized boundary-perturbation sensitivity audit.

IMPORTANT: This does not replace the frozen Step 11--14 protocol or its saved
results. It creates an alternative, post-hoc perturbation mechanism solely to
assess dependence on the original pseudo-random boundary-pixel tie-breaking.

Algorithm: At a label-blind, deterministic boundary anchor, grow one connected
4-neighbour patch into (dilation) or from (erosion) the lesion. Its priority
favours the original contour and proximity to the anchor; the 10%-area change
includes the 5%-area change exactly. Ultrasound pixels never change.

Commands:
  python -m src.coherent_perturbations --limit 12
  python -m src.coherent_perturbations --all-development
"""
from __future__ import annotations

import argparse
import json
import heapq
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.measure import perimeter_crofton

from .features import PROJECT_ROOT, preprocess_pair, read_pair
from .perturbations import (MIN_PIXELS, TARGET_IOUS, Variant,
                            make_variants, mask_iou, partial_boundary_change)

NEIGHBORS = ((-1, 0), (0, -1), (0, 1), (1, 0))


def choose_anchor(mask: np.ndarray) -> tuple[int, int]:
    """Choose an interior contour pixel with maximum image-frame clearance."""
    boundary = mask & ~ndi.binary_erosion(mask, structure=ndi.generate_binary_structure(2, 1))
    yy, xx = np.nonzero(boundary)
    if len(yy) == 0:
        raise ValueError("Lesion has no usable boundary")
    h, w = mask.shape
    clearance = np.minimum.reduce([yy, xx, h - yy - 1, w - xx - 1])
    # np.nonzero is row-major; the first maximum is a deterministic tie-break.
    i = int(np.argmax(clearance))
    return int(yy[i]), int(xx[i])


def _grow_patch(mask: np.ndarray, kind: str, n: int, anchor: tuple[int, int]) -> np.ndarray:
    """Grow n changed pixels in a single 4-connected region, near one anchor."""
    h, w = mask.shape
    if kind == "erosion":
        allowed = mask
        origin = anchor
        depth = ndi.distance_transform_edt(mask)
    elif kind == "dilation":
        allowed = ~mask
        adjacent = [(anchor[0] + dy, anchor[1] + dx) for dy, dx in NEIGHBORS]
        adjacent = [(y, x) for y, x in adjacent
                    if 0 <= y < h and 0 <= x < w and allowed[y, x]]
        if not adjacent:
            raise ValueError("No exterior pixel adjacent to boundary anchor")
        # Pick the nearest admissible exterior pixel, using a fixed neighbor order.
        origin = adjacent[0]
        depth = ndi.distance_transform_edt(~mask)
    else:
        raise ValueError("kind must be erosion or dilation")
    if n <= 0 or n > int(allowed.sum()):
        raise ValueError("Number of changed pixels is invalid")

    sigma = max(3.0, 0.60 * np.sqrt(float(mask.sum()) / np.pi))
    seen = np.zeros((h, w), dtype=bool)
    chosen = np.zeros((h, w), dtype=bool)
    q: list[tuple[float, int, int]] = []

    def push(y: int, x: int) -> None:
        if 0 <= y < h and 0 <= x < w and allowed[y, x] and not seen[y, x]:
            seen[y, x] = True
            radial = ((y - origin[0]) ** 2 + (x - origin[1]) ** 2) / (2.0 * sigma ** 2)
            # Penalize depth; remove/add along existing contour first.
            heapq.heappush(q, (float(radial + 0.45 * depth[y, x]), y, x))

    push(*origin)
    changed = 0
    while changed < n:
        if not q:
            raise ValueError("Could not grow coherent patch to desired size")
        _, y, x = heapq.heappop(q)
        chosen[y, x] = True
        changed += 1
        for dy, dx in NEIGHBORS:
            push(y + dy, x + dx)
    return chosen


def coherent_boundary_change(mask: np.ndarray, target_iou: float, kind: str) -> np.ndarray:
    original = np.asarray(mask, dtype=bool)
    if original.ndim != 2 or int(original.sum()) < MIN_PIXELS:
        raise ValueError("Invalid or too-small lesion mask")
    if kind not in ("erosion", "dilation") or not 0.0 < target_iou < 1.0:
        raise ValueError("Invalid perturbation type or target")
    area = int(original.sum())
    n = max(1, round(area * (1 - target_iou)) if kind == "erosion"
            else round(area * (1 / target_iou - 1)))
    if kind == "erosion" and area - n < MIN_PIXELS:
        raise ValueError("Erosion would leave too few lesion pixels")
    patch = _grow_patch(original, kind, n, choose_anchor(original))
    changed = original.copy()
    changed[patch] = (kind == "dilation")
    return changed


def make_coherent_variants(mask: np.ndarray) -> dict[str, Variant | None]:
    """Same eight condition names; only erosion/dilation algorithm differs.

    Translation variants use the unmodified original implementation, so the
    comparison changes ONE aspect of the existing experimental protocol.
    """
    original = np.asarray(mask, dtype=bool)
    # Reuse *exactly* the Step 11 translation variants and clipping policy.
    original_variants = make_variants(original)
    out: dict[str, Variant | None] = {}
    for severity, target in TARGET_IOUS.items():
        for kind in ("erosion", "dilation"):
            name = f"{kind}_{severity}"
            changed = coherent_boundary_change(original, target, kind)
            count = int(np.logical_xor(original, changed).sum())
            out[name] = Variant(name, changed, mask_iou(original, changed), count,
                                float(changed.sum() / original.sum()), count)
    for key in ("shift_left_moderate", "shift_right_moderate",
                "shift_up_moderate", "shift_down_moderate"):
        out[key] = original_variants[key]
    return out


def _circ(mask: np.ndarray) -> float:
    perimeter = float(perimeter_crofton(mask, directions=4))
    return float(4 * np.pi * mask.sum() / (perimeter ** 2))


def audit(zip_path: Path, manifest_path: Path, out_path: Path, *, limit: int | None = 12) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_path, dtype={"ID": str})
    necessary = {"ID", "Case", "Pathology", "split", "dev_fold"}
    if not necessary.issubset(manifest.columns) or manifest.ID.duplicated().any():
        raise ValueError("Invalid locked image manifest")
    dev = manifest.loc[manifest.split.eq("development")].sort_values("ID")
    if limit is not None:
        if limit < 1:
            raise ValueError("Limit must be positive")
        dev = dev.head(limit)
    if dev.empty:
        raise ValueError("No development records")
    rows = []
    with ZipFile(zip_path) as archive:
        for idx, item in enumerate(dev.itertuples(index=False), 1):
            img, original = preprocess_pair(*read_pair(archive, str(item.ID)))
            frozen = img.copy()
            old_circ = _circ(original)
            # Four *changed* conditions only. The four translations are exactly
            # the original algorithm and therefore need no repeated audit.
            # Avoid expensive duplicate construction of 96 translation masks
            # per image (24 shifts x four directions) for this geometry audit.
            for severity, target in TARGET_IOUS.items():
                for kind in ("erosion", "dilation"):
                    condition = f"{kind}_{severity}"
                    old_mask = partial_boundary_change(original, target, kind)
                    new_mask = coherent_boundary_change(original, target, kind)
                    rows.append({
                        "ID": str(item.ID), "Case": int(item.Case),
                        "condition": condition, "valid": True,
                        "actual_iou": mask_iou(original, new_mask),
                        "changed_pixels": int(np.logical_xor(original, new_mask).sum()),
                        "circularity_clean": old_circ,
                        "circularity_original_protocol": _circ(old_mask),
                        "circularity_coherent_protocol": _circ(new_mask),
                    })
            if not np.array_equal(img, frozen):
                raise AssertionError("Ultrasound image mutated")
            if idx % 100 == 0:
                print(f"Audited {idx}/{len(dev)} development images...", flush=True)
    result = pd.DataFrame(rows)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(out_path, index=False, float_format="%.12g")
    old_d = (result.circularity_original_protocol - result.circularity_clean).abs()
    new_d = (result.circularity_coherent_protocol - result.circularity_clean).abs()
    result = result.assign(abs_circ_change_original=old_d, abs_circ_change_coherent=new_d)
    s = result.groupby("condition", sort=False).agg(
        valid=("valid", "sum"), median_iou=("actual_iou", "median"),
        median_abs_circ_original=("abs_circ_change_original", "median"),
        median_abs_circ_coherent=("abs_circ_change_coherent", "median")
    ).round(4)
    summary = {
        "purpose": "post-hoc geometry-only algorithm sensitivity check; NOT replacement of Step 14",
        "development_images": int(len(dev)), "changed_conditions": 4,
        "unchanged_translation_conditions": 4, "test_patients_used": 0,
        "model_predictions_used": 0, "original_step_14_results_untouched": True,
        "valid_variants": int(result.valid.sum()), "intended_variants": int(len(result))
    }
    (out_path.parent / (out_path.stem + "_summary.json")).write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print("BUS-BRA STEP 14.2 — COHERENT-BOUNDARY SENSITIVITY AUDIT")
    print("=" * 65)
    print(f"Development images: {len(dev)} | Changed-condition rows: {len(result)} | Valid: {result.valid.sum()}")
    print("Four original translation conditions are unchanged and are not regenerated here.")
    print(s.to_string())
    print("Geometry-only; NO model training/predictions; original Step 14 results preserved.")
    print(f"Saved: {out_path}")
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description="Post-hoc alternative coherent boundary masks")
    parser.add_argument("--zip", type=Path, default=PROJECT_ROOT / "data/raw/BUSBRA.zip")
    parser.add_argument("--manifest", type=Path, default=PROJECT_ROOT / "splits/image_splits.csv")
    parser.add_argument("--limit", type=int, default=12)
    parser.add_argument("--all-development", action="store_true")
    args = parser.parse_args()
    if args.all_development and args.limit != 12:
        parser.error("Do not combine --all-development with --limit")
    limit = None if args.all_development else args.limit
    dest = PROJECT_ROOT / "results/robustness" / (
        "coherent_geometry_development.csv" if limit is None else f"coherent_geometry_smoke_{limit}.csv"
    )
    audit(args.zip, args.manifest, dest, limit=limit)


if __name__ == "__main__":
    main()
```

**`src/data.py`**

```python
"""BUS-BRA v1.0 dataset audit. Run from the busbra-robustness project root.

Reads data/raw/BUSBRA.zip directly (no extraction needed).
Checks image/mask pairing, patient label consistency, image dimensions,
mask validity, and official cross-validation fold grouping.
Outputs are written to results/audit/ (excluded from Git by default).
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
from collections import Counter
from pathlib import Path
from zipfile import BadZipFile, ZipFile

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError


PROJECT_ROOT = Path(__file__).resolve().parents[1]


def scan_dataset(archive_path: Path, output_dir: Path) -> dict:
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Dataset archive not found: {archive_path}\n"
            "Download BUSBRA.zip from https://zenodo.org/records/8231412 "
            "and place it at data/raw/BUSBRA.zip."
        )

    output_dir.mkdir(parents=True, exist_ok=True)
    issues: list[dict] = []
    dimension_mismatches: list[dict] = []
    mask_area_fractions: list[float] = []

    with archive_path.open("rb") as archive_file:
        sha256 = hashlib.file_digest(archive_file, "sha256").hexdigest()

    try:
        with ZipFile(archive_path, "r") as zf:
            paths = set(zf.namelist())
            csv_path = "BUSBRA/bus_data.csv"
            if csv_path not in paths:
                raise ValueError(f"Missing {csv_path} inside archive")
            metadata = pd.read_csv(io.BytesIO(zf.read(csv_path)))
            required = {"ID", "Case", "Pathology", "Width", "Height"}
            missing_columns = required - set(metadata.columns)
            if missing_columns:
                raise ValueError(f"Missing metadata columns: {sorted(missing_columns)}")

            image_paths = {
                p for p in paths if p.startswith("BUSBRA/Images/") and p.endswith(".png")
            }
            mask_paths = {
                p for p in paths if p.startswith("BUSBRA/Masks/") and p.endswith(".png")
            }

            duplicate_ids = int(metadata["ID"].duplicated().sum())
            label_values = metadata["Pathology"].astype(str).str.strip().str.lower()
            if not label_values.isin(["benign", "malignant"]).all():
                issues.append({"type": "unexpected_pathology_value", "detail": str(sorted(label_values.unique()))})
            case_label_count = (
                metadata.assign(_label=label_values).groupby("Case") ["_label"].nunique()
            )
            inconsistent_cases = case_label_count[case_label_count != 1].index.tolist()
            if inconsistent_cases:
                issues.append({"type": "inconsistent_patient_labels", "detail": str(inconsistent_cases[:20])})
            if duplicate_ids:
                issues.append({"type": "duplicate_image_ID", "detail": str(duplicate_ids)})

            expected_images = {f"BUSBRA/Images/{x}.png" for x in metadata["ID"].astype(str)}
            expected_masks = {
                f"BUSBRA/Masks/mask_{x.removeprefix('bus_')}.png"
                for x in metadata["ID"].astype(str)
            }
            for category, actual, expected in (
                ("images", image_paths, expected_images),
                ("masks", mask_paths, expected_masks),
            ):
                for p in sorted(expected - actual):
                    issues.append({"type": f"missing_{category}", "detail": p})
                for p in sorted(actual - expected):
                    issues.append({"type": f"unlisted_{category}", "detail": p})

            bad_image_mask_dimensions = 0
            empty_masks = 0
            nonbinary_masks = 0
            unreadable_files = 0
            verified_pairs = 0
            for row in metadata.itertuples(index=False):
                identifier = str(row.ID)
                img_path = f"BUSBRA/Images/{identifier}.png"
                mask_path = f"BUSBRA/Masks/mask_{identifier.removeprefix('bus_')}.png"
                if img_path not in paths or mask_path not in paths:
                    continue
                try:
                    with Image.open(io.BytesIO(zf.read(img_path))) as image:
                        image.load()
                        image_size = image.size
                    with Image.open(io.BytesIO(zf.read(mask_path))) as mask:
                        mask.load()
                        mask_size = mask.size
                        mask_array = np.asarray(mask.convert("L"))
                except (OSError, ValueError, UnidentifiedImageError) as exc:
                    unreadable_files += 1
                    issues.append({"type": "unreadable_image_or_mask", "detail": f"{identifier}: {exc}"})
                    continue

                if image_size != mask_size:
                    bad_image_mask_dimensions += 1
                    issues.append({"type": "image_mask_size_mismatch", "detail": identifier})
                if image_size != (int(row.Width), int(row.Height)):
                    dimension_mismatches.append({
                        "ID": identifier,
                        "csv_width": int(row.Width),
                        "csv_height": int(row.Height),
                        "actual_width": image_size[0],
                        "actual_height": image_size[1],
                    })
                if not np.any(mask_array):
                    empty_masks += 1
                    issues.append({"type": "empty_mask", "detail": identifier})
                unique_vals = np.unique(mask_array)
                if not set(unique_vals.tolist()).issubset({0, 1, 255}):
                    nonbinary_masks += 1
                    issues.append({"type": "nonbinary_mask", "detail": f"{identifier}: values={unique_vals[:8]}"})
                mask_area_fractions.append(float(np.count_nonzero(mask_array) / mask_array.size))
                verified_pairs += 1

            official_folds: dict[str, dict] = {}
            for fold_filename in ("5-fold-cv.csv", "10-fold-cv.csv"):
                fold_path = f"BUSBRA/{fold_filename}"
                if fold_path not in paths:
                    issues.append({"type": "missing_official_fold_file", "detail": fold_filename})
                    continue
                folds = pd.read_csv(io.BytesIO(zf.read(fold_path)))
                if not {"ID", "kFold"}.issubset(folds.columns):
                    issues.append({"type": "invalid_fold_columns", "detail": fold_filename})
                    continue
                combined = metadata[["ID", "Case"]].merge(
                    folds[["ID", "kFold"]], on="ID", how="left", validate="one_to_one"
                )
                missing_folds = int(combined["kFold"].isna().sum())
                split_cases = int((combined.groupby("Case")["kFold"].nunique() > 1).sum())
                official_folds[fold_filename] = {
                    "fold_counts_by_image": {
                        str(k): int(v) for k, v in folds["kFold"].value_counts().sort_index().items()
                    },
                    "images_missing_fold": missing_folds,
                    "patients_assigned_to_multiple_folds": split_cases,
                }
                if missing_folds or split_cases:
                    issues.append({
                        "type": "fold_integrity_problem", "detail": f"{fold_filename}: missing={missing_folds}, split_cases={split_cases}"
                    })

            patient_labels = (
                metadata.assign(_label=label_values)
                .drop_duplicates(subset=["Case"])["_label"]
                .value_counts().to_dict()
            )
            image_labels = label_values.value_counts().to_dict()
            summary = {
                "archive_name": archive_path.name,
                "archive_sha256": sha256,
                "image_png_files": len(image_paths),
                "mask_png_files": len(mask_paths),
                "metadata_rows": len(metadata),
                "patients": int(metadata["Case"].nunique()),
                "patient_labels": {k: int(v) for k, v in patient_labels.items()},
                "image_labels": {k: int(v) for k, v in image_labels.items()},
                "images_per_patient": {
                    str(k): int(v) for k, v in metadata.groupby("Case").size().value_counts().sort_index().items()
                },
                "duplicate_image_ids": duplicate_ids,
                "inconsistent_patient_labels": len(inconsistent_cases),
                "verified_image_mask_pairs": verified_pairs,
                "image_mask_dimension_mismatches": bad_image_mask_dimensions,
                "metadata_dimension_mismatches": len(dimension_mismatches),
                "empty_masks": empty_masks,
                "nonbinary_masks": nonbinary_masks,
                "unreadable_pairs": unreadable_files,
                "mask_area_fraction_min": min(mask_area_fractions) if mask_area_fractions else None,
                "mask_area_fraction_max": max(mask_area_fractions) if mask_area_fractions else None,
                "official_folds": official_folds,
                "blocking_issue_count": len(issues),
            }
    except BadZipFile as exc:
        raise ValueError("The BUSBRA.zip archive is corrupted or not a ZIP file") from exc

    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    pd.DataFrame(dimension_mismatches, columns=[
        "ID", "csv_width", "csv_height", "actual_width", "actual_height"
    ]).to_csv(output_dir / "metadata_dimension_mismatches.csv", index=False)
    pd.DataFrame(issues, columns=["type", "detail"]).to_csv(output_dir / "issues.csv", index=False)

    print("\nBUS-BRA DATA AUDIT")
    print("=" * 46)
    print(f"Image files / mask files: {summary['image_png_files']} / {summary['mask_png_files']}")
    print(f"Metadata rows: {summary['metadata_rows']}; unique patients: {summary['patients']}")
    print(f"Patient labels: {summary['patient_labels']}")
    print(f"Verified image-mask pairs: {verified_pairs}")
    print(f"Image-mask size mismatches: {bad_image_mask_dimensions}")
    print(f"CSV-vs-image dimension discrepancies: {len(dimension_mismatches)} (nonblocking; investigate)")
    print(f"Empty masks: {empty_masks}; nonbinary masks: {nonbinary_masks}")
    for name, checks in official_folds.items():
        print(f"{name}: split patients={checks['patients_assigned_to_multiple_folds']}, missing IDs={checks['images_missing_fold']}")
    print(f"Blocking issues: {len(issues)}")
    print(f"Report: {output_dir / 'summary.json'}")
    return summary


def main() -> None:
    parser = argparse.ArgumentParser(description="Audit BUS-BRA image/mask/label integrity")
    parser.add_argument("--zip", type=Path, default=PROJECT_ROOT / "data" / "raw" / "BUSBRA.zip")
    parser.add_argument("--output", type=Path, default=PROJECT_ROOT / "results" / "audit")
    args = parser.parse_args()
    summary = scan_dataset(args.zip, args.output)
    if summary["blocking_issue_count"]:
        raise SystemExit("Audit found blocking issues; read results/audit/issues.csv")


if __name__ == "__main__":
    main()
```

**`src/evaluation.py`**

```python

```

**`src/feature_audit.py`**

```python
"""Audit BUS-BRA development features without fitting a model.

Place in src/feature_audit.py and run from the repository root:
    .\\.venv\\Scripts\\python.exe -m src.feature_audit

Read-only for source data. All outputs go to results/feature_audit/.
"""
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
FEATURES = ROOT / "data" / "processed" / "development_features.csv"
MANIFEST = ROOT / "splits" / "image_splits.csv"
OUT = ROOT / "results" / "feature_audit"
ID_COLS = ["ID", "Case", "Pathology", "split", "dev_fold"]


def fail(message: str) -> None:
    raise AssertionError(message)


def validate_bounds(f: pd.DataFrame) -> list[str]:
    errors = []
    checks = {
        "shape_area_fraction": (0, 1),
        "shape_eccentricity": (0, 1),
        "shape_solidity": (0, 1),
        "shape_extent": (0, 1),
        "int_mean": (0, 1),
        "int_std": (0, 1),
        "int_median": (0, 1),
        "int_p10": (0, 1),
        "int_p90": (0, 1),
        "glcm_homogeneity_d1": (0, 1),
        "glcm_homogeneity_d3": (0, 1),
        "glcm_asm_d1": (0, 1),
        "glcm_asm_d3": (0, 1),
        "glcm_correlation_d1": (-1, 1),
        "glcm_correlation_d3": (-1, 1),
    }
    for name, (lo, hi) in checks.items():
        if name not in f.columns:
            errors.append(f"Missing required bounded feature: {name}")
        elif not f[name].between(lo - 1e-8, hi + 1e-8).all():
            errors.append(f"Invalid range: {name} outside [{lo}, {hi}]")
    if "shape_axis_ratio" in f and (f["shape_axis_ratio"] < 1 - 1e-8).any():
        errors.append("shape_axis_ratio < 1")
    for name in f.columns:
        if name.startswith(("glcm_contrast_", "glcm_dissimilarity_")) or name == "int_entropy":
            if (f[name] < -1e-8).any():
                errors.append(f"Negative value in {name}")
    if {"int_p10", "int_median", "int_p90"}.issubset(f.columns):
        if (f["int_p10"] > f["int_median"] + 1e-8).any() or (f["int_median"] > f["int_p90"] + 1e-8).any():
            errors.append("Intensity percentile order is inconsistent")
    return errors


def main() -> None:
    if not FEATURES.exists():
        raise FileNotFoundError(f"Missing features: {FEATURES}")
    if not MANIFEST.exists():
        raise FileNotFoundError(f"Missing manifest: {MANIFEST}")
    d = pd.read_csv(FEATURES, dtype={"ID": str})
    m = pd.read_csv(MANIFEST, dtype={"ID": str})
    if d.columns.duplicated().any():
        fail("Feature table has duplicated column names")
    if not set(ID_COLS).issubset(d.columns):
        fail(f"Missing ID/split columns: {set(ID_COLS) - set(d.columns)}")
    feature_names = [c for c in d.columns if c not in ID_COLS]
    if len(feature_names) != 23:
        fail(f"Expected exactly 23 features, found {len(feature_names)}")
    if not all(pd.api.types.is_numeric_dtype(d[c]) for c in feature_names):
        fail("One or more features have nonnumeric dtype")
    if d.ID.duplicated().any() or m.ID.duplicated().any():
        fail("Duplicate IDs in features or split manifest")
    dev = m.loc[m["split"].eq("development"), ID_COLS]
    if set(d.ID) != set(dev.ID):
        missing = sorted(set(dev.ID) - set(d.ID))
        extra = sorted(set(d.ID) - set(dev.ID))
        fail(f"Image coverage differs from locked development split; missing={missing[:5]}, extra={extra[:5]}")
    merged = d[ID_COLS].merge(dev, on="ID", how="inner", suffixes=("_features", "_manifest"), validate="one_to_one")
    for field in ("Case", "Pathology", "split", "dev_fold"):
        if not (merged[f"{field}_features"].astype(str) == merged[f"{field}_manifest"].astype(str)).all():
            fail(f"Feature table differs from locked split manifest in {field}")
    if not d["split"].eq("development").all() or d.Case.nunique() != 851 or len(d) != 1514:
        fail("Unexpected development partition counts")
    for col in ("Pathology", "dev_fold"):
        if d.groupby("Case")[col].nunique().max() != 1:
            fail(f"Inconsistent {col} across images of a patient")
    if set(d["dev_fold"].unique()) != set(range(5)):
        fail("Expected folds 0–4")
    x = d[feature_names]
    if x.isna().any().any() or not np.isfinite(x.to_numpy(dtype=float)).all():
        fail("Missing or nonfinite features")
    errors = validate_bounds(x)
    if errors:
        fail("Feature range checks failed:\n" + "\n".join(errors))
    zero_var = x.columns[x.nunique(dropna=False) <= 1].tolist()
    if zero_var:
        fail(f"Constant features found: {zero_var}")

    OUT.mkdir(parents=True, exist_ok=True)
    summary = x.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
    summary.to_csv(OUT / "feature_distribution_summary.csv", float_format="%.8g")
    corr = x.corr(method="pearson").abs()
    pairs = []
    for i, a in enumerate(feature_names):
        for b in feature_names[i+1:]:
            val = float(corr.loc[a, b])
            if np.isfinite(val) and val >= 0.95:
                pairs.append({"feature_a": a, "feature_b": b, "abs_pearson_r": val})
    cp = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "abs_pearson_r"])
    cp = cp.sort_values("abs_pearson_r", ascending=False)
    cp.to_csv(OUT / "high_correlations.csv", index=False, float_format="%.6g")

    # Flag potential extremes for human review, but do NOT drop them.
    med = x.median()
    mad = (x - med).abs().median()
    flagged = []
    for name in feature_names:
        if mad[name] <= 1e-12:
            continue
        score = 0.6745 * (x[name] - med[name]).abs() / mad[name]
        for i in np.where(score.to_numpy() > 6)[0]:
            flagged.append({"ID": d.iloc[i]["ID"], "Case": d.iloc[i]["Case"],
                            "feature": name, "value": float(x.iloc[i][name]),
                            "robust_z": float(score.iloc[i])})
    flags = pd.DataFrame(flagged, columns=["ID", "Case", "feature", "value", "robust_z"])
    flags = flags.sort_values("robust_z", ascending=False)
    flags.to_csv(OUT / "potential_outliers.csv", index=False, float_format="%.6g")

    report = {
        "images": int(len(d)), "patients": int(d.Case.nunique()),
        "features": len(feature_names), "test_images_seen": 0,
        "missing_values": int(x.isna().sum().sum()),
        "duplicate_ids": int(d.ID.duplicated().sum()),
        "duplicate_feature_vectors": int(x.duplicated().sum()),
        "constant_features": zero_var,
        "high_correlation_pairs_abs_r_at_least_0_95": len(cp),
        "potential_outlier_values_robust_z_over_6": len(flags),
        "class_patients": {str(k): int(v) for k, v in d.groupby("Pathology").Case.nunique().items()},
        "fold_validation_patients": {str(k): int(v) for k, v in d.groupby("dev_fold").Case.nunique().items()},
    }
    (OUT / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("\nBUS-BRA FEATURE QUALITY AUDIT")
    print("=" * 45)
    print(f"Images: {report['images']} | Patients: {report['patients']} | Features: {report['features']}")
    print("Locked patient/image split: MATCHED")
    print("Feature ranges and quantile ordering: PASSED")
    print(f"Missing: {report['missing_values']} | Constant features: {len(zero_var)} | Duplicate feature vectors: {report['duplicate_feature_vectors']}")
    print(f"Highly correlated feature pairs (|r| >= 0.95): {len(cp)}")
    if len(cp):
        print(cp.head(8).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"Potential extreme feature values for review: {len(flags)} (NOT removed)")
    print(f"Reports saved in: {OUT}")
    print("SUCCESS: Structural feature validation passed; review correlations and outliers next.")


if __name__ == "__main__":
    main()
```

**`src/features.py`**

```python
"""Reproducible image + lesion-mask features for BUS-BRA.

Install as src/features.py. From the project root, use:
    .\\.venv\\Scripts\\python.exe -m src.features --limit 12

This smoke test extracts 23 handcrafted features from DEVELOPMENT images only.
The test split is deliberately excluded; no scaling or model fitting happens here.

Preprocessing: grayscale image, mask nearest-neighbor; both resized with
aspect ratio preserved to longest side = 256 pixels. Masked GLCMs count ONLY
pixel pairs with BOTH pixels inside the lesion. No blacked-out background.
"""
from __future__ import annotations

import argparse
import io
import json
import math
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from PIL import Image
from skimage.measure import perimeter_crofton, regionprops

PROJECT_ROOT = Path(__file__).resolve().parents[1]
MAX_SIDE = 256
GLCM_LEVELS = 32
GLCM_DISTANCES = (1, 3)
GLCM_DIRECTIONS = ((0, 1), (1, 1), (1, 0), (-1, 1))
SHAPE = (
    "shape_area_fraction", "shape_axis_ratio", "shape_eccentricity",
    "shape_solidity", "shape_extent", "shape_circularity",
)
INTENSITY = (
    "int_mean", "int_std", "int_median", "int_p10", "int_p90",
    "int_skewness", "int_entropy",
)
TEXTURE_PROPERTIES = ("contrast", "dissimilarity", "homogeneity", "asm", "correlation")
TEXTURE = tuple(f"glcm_{p}_d{d}" for d in GLCM_DISTANCES for p in TEXTURE_PROPERTIES)
FEATURE_COLUMNS = SHAPE + INTENSITY + TEXTURE
assert len(FEATURE_COLUMNS) == 23


def preprocess_pair(image: np.ndarray, mask: np.ndarray, max_side: int = MAX_SIDE):
    """Convert arrays to uint8 grayscale and bool mask; resize together.

    Scaling factor is computed from the ACTUAL image dimensions (not CSV).
    """
    if image.ndim != 2 or mask.ndim != 2 or image.shape != mask.shape:
        raise ValueError("Expected same-shaped two-dimensional image and mask")
    if max_side < 32:
        raise ValueError("max_side must be >= 32")
    if not np.any(mask):
        raise ValueError("Empty lesion mask")
    if not np.issubdtype(image.dtype, np.number):
        raise ValueError("Grayscale image must be numeric")
    if not np.isfinite(image).all():
        raise ValueError("Image contains NaN or Inf")
    arr = np.clip(image, 0, 255).astype(np.uint8)
    binary = (mask != 0).astype(np.uint8) * 255
    height, width = arr.shape
    scale = max_side / max(height, width)
    output_size = (max(1, round(width * scale)), max(1, round(height * scale)))
    img_out = np.asarray(Image.fromarray(arr).resize(output_size, Image.Resampling.BILINEAR))
    mask_out = np.asarray(Image.fromarray(binary).resize(output_size, Image.Resampling.NEAREST)) > 0
    if not mask_out.any():
        raise ValueError("Mask became empty after resize")
    return img_out, mask_out


def _glcm_properties(gray: np.ndarray, lesion: np.ndarray) -> dict[str, float]:
    """Calculate masked, symmetric, normalized GLCMs at two distances.

    For each direction/distance, only lesion-interior pixel pairs count.
    Each direction contributes equally to the final per-distance means.
    """
    quantized = np.minimum(gray.astype(np.uint16) * GLCM_LEVELS // 256, GLCM_LEVELS - 1)
    h, w = lesion.shape
    grid = np.arange(GLCM_LEVELS, dtype=np.float64)
    ii = grid[:, None]
    jj = grid[None, :]
    dif = ii - jj
    results: dict[str, float] = {}

    for distance in GLCM_DISTANCES:
        directional: list[dict[str, float]] = []
        for step_y, step_x in GLCM_DIRECTIONS:
            dy, dx = distance * step_y, distance * step_x
            sy0, sy1 = max(0, -dy), min(h, h - dy)
            sx0, sx1 = max(0, -dx), min(w, w - dx)
            if sy0 >= sy1 or sx0 >= sx1:
                continue
            target_y0, target_y1 = sy0 + dy, sy1 + dy
            target_x0, target_x1 = sx0 + dx, sx1 + dx
            inside = lesion[sy0:sy1, sx0:sx1] & lesion[target_y0:target_y1, target_x0:target_x1]
            if not inside.any():
                continue
            a = quantized[sy0:sy1, sx0:sx1][inside]
            b = quantized[target_y0:target_y1, target_x0:target_x1][inside]
            count = np.bincount(a * GLCM_LEVELS + b, minlength=GLCM_LEVELS**2)
            cm = count.reshape((GLCM_LEVELS, GLCM_LEVELS)).astype(np.float64)
            cm += cm.T
            cm /= cm.sum()
            p_i, p_j = cm.sum(axis=1), cm.sum(axis=0)
            mu_i, mu_j = np.dot(grid, p_i), np.dot(grid, p_j)
            var_i = np.dot((grid - mu_i)**2, p_i)
            var_j = np.dot((grid - mu_j)**2, p_j)
            if var_i < 1e-12 or var_j < 1e-12:
                corr = 1.0
            else:
                corr = float(np.sum(cm * (ii - mu_i) * (jj - mu_j)) / math.sqrt(var_i * var_j))
            directional.append({
                "contrast": float(np.sum((dif**2) * cm)),
                "dissimilarity": float(np.sum(np.abs(dif) * cm)),
                "homogeneity": float(np.sum(cm / (1.0 + dif**2))),
                "asm": float(np.sum(cm**2)),
                "correlation": corr,
            })
        if not directional:
            raise ValueError(f"Mask too small for masked GLCM at distance {distance}")
        for prop in TEXTURE_PROPERTIES:
            results[f"glcm_{prop}_d{distance}"] = float(np.mean([d[prop] for d in directional]))
    return results


def extract_features(image: np.ndarray, mask: np.ndarray) -> dict[str, float]:
    """Extract 23 features from a PREPROCESSED grayscale image and binary mask.

    For perturbation experiments, change *mask* only and call this function
    again on the SAME preprocessed image. Do NOT resize perturbed masks anew.
    """
    if image.ndim != 2 or mask.ndim != 2 or image.shape != mask.shape:
        raise ValueError("image and mask must be 2D arrays of matching size")
    binary = mask.astype(bool)
    if binary.sum() < 15:
        raise ValueError("Lesion mask contains fewer than 15 pixels")
    if not np.isfinite(image).all():
        raise ValueError("Image contains nonfinite values")
    if np.min(image) < 0 or np.max(image) > 255:
        raise ValueError("Expected grayscale intensities from 0 to 255")

    props = regionprops(binary.astype(np.uint8))[0]
    area = float(binary.sum())
    perim = float(perimeter_crofton(binary, directions=4))
    minor = float(props.axis_minor_length)
    major = float(props.axis_major_length)
    if perim <= 0 or minor <= 0:
        raise ValueError("Degenerate lesion shape")

    pixels = np.asarray(image[binary], dtype=np.float64)
    center = float(pixels.mean())
    sd = float(pixels.std(ddof=0))
    hist = np.histogram(pixels, bins=32, range=(0, 256))[0].astype(np.float64)
    probabilities = hist[hist > 0] / len(pixels)
    entropy = float(-(probabilities * np.log2(probabilities)).sum())
    skewness = 0.0 if sd < 1e-12 else float(np.mean(((pixels - center) / sd) ** 3))
    feats = {
        "shape_area_fraction": float(area / binary.size),
        "shape_axis_ratio": float(major / minor),
        "shape_eccentricity": float(props.eccentricity),
        "shape_solidity": float(props.solidity),
        "shape_extent": float(props.extent),
        "shape_circularity": float(4.0 * math.pi * area / (perim**2)),
        "int_mean": center / 255.0,
        "int_std": sd / 255.0,
        "int_median": float(np.median(pixels)) / 255.0,
        "int_p10": float(np.percentile(pixels, 10)) / 255.0,
        "int_p90": float(np.percentile(pixels, 90)) / 255.0,
        "int_skewness": skewness,
        "int_entropy": entropy,
    }
    feats.update(_glcm_properties(np.asarray(image, dtype=np.uint8), binary))
    assert tuple(feats) == FEATURE_COLUMNS, "Internal feature schema changed unexpectedly"
    if not np.isfinite(list(feats.values())).all():
        raise ValueError("Extracted nonfinite feature value")
    return feats


def read_pair(zf: ZipFile, image_id: str):
    """Read one BUS-BRA image and matching binary lesion mask from ZIP."""
    image_path = f"BUSBRA/Images/{image_id}.png"
    mask_path = f"BUSBRA/Masks/mask_{image_id.removeprefix('bus_')}.png"
    with Image.open(io.BytesIO(zf.read(image_path))) as im:
        image = np.asarray(im.convert("L"))
    with Image.open(io.BytesIO(zf.read(mask_path))) as m:
        mask = np.asarray(m.convert("L")) > 0
    return image, mask


def extract_development(zip_path: Path, manifest_path: Path, output_path: Path,
                        limit: int | None = 12) -> pd.DataFrame:
    """Extract features only from images preassigned to development.

    `limit=None` processes all 1,514 development images. A small fixed limit
    is useful for initial correctness checks without a long run.
    """
    manifest = pd.read_csv(manifest_path, dtype={"ID": str})
    expected = {"ID", "Case", "Pathology", "split", "dev_fold"}
    if not expected.issubset(manifest.columns):
        raise ValueError(f"Missing manifest fields: {expected - set(manifest.columns)}")
    if manifest.ID.duplicated().any():
        raise ValueError("Duplicated image ID in split manifest")
    subset = manifest.loc[manifest.split.eq("development")].sort_values("ID")
    if subset.empty:
        raise ValueError("No development rows in the split manifest")
    if limit is not None:
        if limit <= 0:
            raise ValueError("--limit must be positive")
        subset = subset.head(limit)
    all_rows: list[dict] = []
    with ZipFile(zip_path) as zf:
        for n, rec in enumerate(subset.itertuples(index=False), start=1):
            image, mask = read_pair(zf, rec.ID)
            image, mask = preprocess_pair(image, mask)
            feat = extract_features(image, mask)
            all_rows.append({
                "ID": str(rec.ID), "Case": rec.Case, "Pathology": rec.Pathology,
                "split": rec.split, "dev_fold": int(rec.dev_fold), **feat,
            })
            if n % 100 == 0:
                print(f"Extracted {n}/{len(subset)} development images...")
    result = pd.DataFrame(all_rows)
    if len(result) != len(subset) or not result["split"].eq("development").all():
        raise RuntimeError("Output violates development-only constraint")
    if result[list(FEATURE_COLUMNS)].isna().any().any():
        raise RuntimeError("Feature output contains missing values")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, float_format="%.12g")
    print(f"Extracted {len(result)} development images")
    print(f"Computed {len(FEATURE_COLUMNS)} features per image")
    print(f"Saved: {output_path}")
    return result


def main():
    parser = argparse.ArgumentParser(description="Extract BUS-BRA features (development patients only)")
    parser.add_argument("--zip", type=Path, default=PROJECT_ROOT / "data/raw/BUSBRA.zip")
    parser.add_argument("--manifest", type=Path, default=PROJECT_ROOT / "splits/image_splits.csv")
    parser.add_argument("--output", type=Path, default=None)
    parser.add_argument("--limit", type=int, default=12, help="Sample size for smoke test; default=12")
    parser.add_argument("--all-development", action="store_true", help="Process all development images")
    args = parser.parse_args()
    if args.all_development and args.limit != 12:
        parser.error("Do not combine --all-development with --limit")
    limit = None if args.all_development else args.limit
    filename = "development_features.csv" if limit is None else f"development_features_smoke_{limit}.csv"
    output = args.output or PROJECT_ROOT / "data/processed" / filename
    if not args.zip.is_file():
        raise FileNotFoundError(args.zip)
    if not args.manifest.is_file():
        raise FileNotFoundError(args.manifest)
    extract_development(args.zip, args.manifest, output, limit=limit)


if __name__ == "__main__":
    main()
```

**`src/final_report.py`**

```python
"""Step 22: read-only synthesis of saved experiments; no fitting or inference.

Writes an editable Markdown report, self-contained HTML, publication figures,
patient-level error audit, tables and checksums. The Step 21 marker must be complete.
"""
from __future__ import annotations

import argparse
import base64
import html
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

from . import patient_calibration as pc
from . import stable_selection as ss
from .coherent_features import COHERENT_CONDITIONS
from .models import ROOT
from .robustness_eval import MODEL_NAMES, parse_boolean_column, summarize_pair

COLORS = {"logreg": "#276B8E", "svm": "#B56A25"}
LABELS = {"logreg": "Logistic Regression", "svm": "RBF SVM"}
COND_LABELS = {"erosion_mild": "Mild erosion", "dilation_mild": "Mild dilation",
               "erosion_moderate": "Moderate erosion", "dilation_moderate": "Moderate dilation", "clean": "Clean"}
FIGURES = ("01_clean_metrics", "02_clean_roc", "03_reliability", "04_confusion",
           "05_heldout_boundary_changes", "06_original_and_coherent", "07_feature_selection")


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def source_paths(root):
    robust = root / "results/robustness"
    return {"manifest": root / "splits/image_splits.csv", "split_summary": root / "splits/split_summary.json",
            "heldout": robust / "heldout_step21", "calibration": robust / "calibration_step20",
            "step19": robust / "selection_step19", "step16": robust / "coherent_step16_run2",
            "step17": robust / "uncertainty_step17", "original": robust,
            "modeling": root / "results/modeling"}


def validate_saved_evaluation(paths):
    heldout, calibration = paths["heldout"], paths["calibration"]
    marker = read_json(calibration / "held_out_evaluation_started.json")
    summary = read_json(heldout / "held_out_summary.json")
    protocol = read_json(calibration / "final_evaluation_protocol.json")
    if marker.get("status") != "complete" or summary.get("status") != "locked_held_out_evaluation_complete":
        raise ValueError("Held-out evaluation is not complete; reporting cannot run it")
    for filename, fingerprint in marker["published_files_sha256"].items():
        if Path(filename).name != filename or pc.sha256(heldout / filename) != fingerprint:
            raise ValueError(f"Published held-out result hash mismatch: {filename}")
    if (marker["summary_sha256"] != pc.sha256(heldout / "held_out_summary.json")
            or marker["protocol_sha256"] != pc.sha256(calibration / "final_evaluation_protocol.json")
            or marker["model_sha256"] != pc.sha256(calibration / "frozen_calibrated_models.joblib")
            or summary["provenance"]["protocol_sha256"] != marker["protocol_sha256"]
            or pc.sha256(heldout / "frozen_protocol_copy.json") != marker["protocol_sha256"]
            or pc.sha256(paths["manifest"]) != marker["manifest_sha256"]
            or summary["model_fits"] != 0 or summary["calibrator_fits"] != 0 or summary["selection_or_retuning"] is not False):
        raise ValueError("Frozen held-out provenance or zero-fit checks failed")
    predictions = pd.read_csv(heldout / "held_out_patient_predictions.csv")
    pairs = pd.read_csv(heldout / "held_out_paired_predictions.csv")
    metrics = pd.read_csv(heldout / "held_out_condition_metrics.csv")
    pair_metrics = pd.read_csv(heldout / "held_out_paired_summary.csv")
    intervals = pd.read_csv(heldout / "held_out_intervals.csv")
    intervals["primary"] = parse_boolean_column(intervals.primary)
    manifest = pd.read_csv(paths["manifest"], dtype={"ID": str})
    test = manifest.loc[manifest.split.eq("test")]
    metadata = test.groupby("Case").agg(actual=("Pathology", lambda y: int(y.iloc[0] == "malignant")), n_images=("ID", "size"))
    n = len(metadata)
    if (n != summary["held_out_patients"] or len(test) != summary["held_out_images"]
            or len(predictions) != n * 10 or predictions.duplicated(["model", "condition", "Case"]).any()
            or len(pairs) != n * 8 or pairs.duplicated(["model", "condition", "Case"]).any()
            or len(metrics) != 10 or metrics.duplicated(["model", "condition"]).any()
            or len(pair_metrics) != 8 or pair_metrics.duplicated(["model", "condition"]).any()
            or len(intervals) != 181 or intervals.primary.sum() != 1):
        raise ValueError("Saved held-out coverage differs")
    for name in MODEL_NAMES:
        for condition in ("clean", *COHERENT_CONDITIONS):
            p = predictions.loc[predictions.model.eq(name) & predictions.condition.eq(condition)].sort_values("Case")
            if set(p.Case) != set(metadata.index) or not np.array_equal(p[["actual", "n_images"]], metadata.loc[p.Case, ["actual", "n_images"]]):
                raise ValueError("Saved patient metadata differs from the locked manifest")
            if not np.array_equal(p.predicted, p.prob_malignant.ge(.5).astype(int)):
                raise ValueError("Saved predictions disagree with the frozen threshold")
            recalculated = pc.probability_metrics(p)
            row = metrics.loc[metrics.model.eq(name) & metrics.condition.eq(condition)]
            if len(row) != 1:
                raise ValueError("Missing saved metric row")
            for key, value in recalculated.items():
                if not np.isclose(row.iloc[0][key], value, rtol=1e-8, atol=1e-9):
                    raise ValueError(f"Saved metric mismatch: {key}")
            if condition != "clean":
                pair = pairs.loc[pairs.model.eq(name) & pairs.condition.eq(condition)].sort_values("Case")
                baseline = predictions.loc[predictions.model.eq(name) & predictions.condition.eq("clean")].sort_values("Case")
                if (not np.array_equal(pair.Case, p.Case)
                        or not np.allclose(pair.clean_score, baseline.prob_malignant, rtol=1e-8, atol=1e-9)
                        or not np.allclose(pair.perturbed_score, p.prob_malignant, rtol=1e-8, atol=1e-9)
                        or not np.array_equal(pair.clean_predicted, baseline.predicted)
                        or not np.array_equal(pair.perturbed_predicted, p.predicted)):
                    raise ValueError("Saved clean/perturbed patient pairing differs")
                recalculated_pair = summarize_pair(pair.assign(eligible=True), model=name, condition=condition)
                saved_pair = pair_metrics.loc[pair_metrics.model.eq(name) & pair_metrics.condition.eq(condition)]
                if len(saved_pair) != 1:
                    raise ValueError("Missing saved pair summary")
                for key, value in recalculated_pair.items():
                    if isinstance(value, (int, float)) and not np.isclose(float(saved_pair.iloc[0][key]), value, rtol=1e-8, atol=1e-9):
                        raise ValueError(f"Saved pair metric mismatch: {key}")
    primary = intervals.loc[intervals.primary].iloc[0]
    if (primary["model"] != "svm" or primary["condition"] != "clean" or primary["metric"] != "balanced_accuracy"
            or not np.isclose(primary.estimate, summary["primary_result"]["estimate"])
            or not np.isclose(primary.ci_low, summary["primary_result"]["ci_low"])
            or not np.isclose(primary.ci_high, summary["primary_result"]["ci_high"])):
        raise ValueError("Primary endpoint differs from the saved analysis")
    return {"summary": summary, "protocol": protocol, "marker": marker, "predictions": predictions,
            "pairs": pairs, "metrics": metrics, "pair_metrics": pair_metrics,
            "intervals": intervals, "manifest": manifest, "primary": primary}


def failure_analysis(predictions, manifest):
    clean = predictions.loc[predictions.condition.eq("clean")]
    lr, svm = [clean.loc[clean.model.eq(name)].set_index("Case").sort_index() for name in MODEL_NAMES]
    if not np.array_equal(lr.index, svm.index):
        raise ValueError("Clean model patients are not paired")
    cases = pd.DataFrame({"Case": lr.index, "actual": lr.actual, "n_images": lr.n_images,
                          "lr_probability": lr.prob_malignant, "svm_probability": svm.prob_malignant,
                          "lr_predicted": lr.predicted, "svm_predicted": svm.predicted}).reset_index(drop=True)
    for short in ("lr", "svm"):
        cases[f"{short}_error"] = cases[f"{short}_predicted"].ne(cases.actual)
        cases[f"{short}_false_negative"] = cases.actual.eq(1) & cases[f"{short}_predicted"].eq(0)
        cases[f"{short}_false_positive"] = cases.actual.eq(0) & cases[f"{short}_predicted"].eq(1)
    image_ids = manifest.loc[manifest.split.eq("test")].groupby("Case").ID.agg(lambda ids: ";".join(sorted(ids)))
    cases["image_ids"] = image_ids.loc[cases.Case].to_numpy()
    overlap = {"both_wrong": int((cases.lr_error & cases.svm_error).sum()),
               "lr_only_wrong": int((cases.lr_error & ~cases.svm_error).sum()),
               "svm_only_wrong": int((~cases.lr_error & cases.svm_error).sum()),
               "both_correct": int((~cases.lr_error & ~cases.svm_error).sum()),
               "both_false_negative": int((cases.lr_false_negative & cases.svm_false_negative).sum()),
               "malignant_patients": int(cases.actual.sum()), "benign_patients": int(cases.actual.eq(0).sum())}
    examples = []
    for label, subset, column, ascending in (
            ("SVM false negative", cases.loc[cases.svm_false_negative], "svm_probability", True),
            ("SVM false positive", cases.loc[cases.svm_false_positive], "svm_probability", False)):
        for row in subset.sort_values([column, "Case"], ascending=[ascending, True]).head(3).itertuples():
            examples.append({"category": label, "Case": row.Case, "actual": row.actual,
                             "lr_probability": row.lr_probability, "svm_probability": row.svm_probability,
                             "image_ids": row.image_ids})
    return cases, overlap, pd.DataFrame(examples)


def load_history(paths):
    original = pd.read_csv(paths["original"] / "robustness_paired_condition_summary.csv")
    coherent = pd.read_csv(paths["step16"] / "coherent_condition_summary.csv")
    selection = pd.read_csv(paths["step19"] / "clean_summary.csv")
    selections = read_json(paths["step19"] / "outer_selections.json")
    cal = pd.read_csv(paths["calibration"] / "calibrated_clean_summary.csv")
    selection_summary = read_json(paths["step19"] / "evaluation_summary.json")
    cal_summary = read_json(paths["calibration"] / "calibration_summary.json")
    uncertainty = pd.read_csv(paths["step17"] / "paired_bootstrap_intervals.csv")
    costs = [{"stage": "19 nested feature selection", "base_inner_fits": selection_summary["fit_counts"]["inner"],
              "base_refits": selection_summary["fit_counts"]["outer_refits"], "calibrator_fits": 0,
              "baseline_check_fits": selection_summary["fit_counts"]["baseline_reproducibility"],
              "recorded_seconds": selection_summary["elapsed_nested_seconds"]},
             {"stage": "20 nested calibration and final fit", "base_inner_fits": cal_summary["base_inner_fits"],
              "base_refits": cal_summary["base_refits"], "calibrator_fits": cal_summary["sigmoid_fits"],
              "baseline_check_fits": 0, "recorded_seconds": None},
             {"stage": "21 held-out evaluation", "base_inner_fits": 0, "base_refits": 0, "calibrator_fits": 0,
              "baseline_check_fits": 0, "recorded_seconds": None}]
    return {"original": original, "coherent": coherent, "selection": selection, "selections": selections,
            "calibration": cal, "uncertainty": uncertainty, "costs": pd.DataFrame(costs),
            "baseline": {name: read_json(paths["modeling"] / f"{name}_nested_cv_summary.json") for name in MODEL_NAMES}}


def model_sizes(calibration):
    # Inspect fitted state only; never call predict/fit while reporting.
    bundle = joblib.load(calibration / "frozen_calibrated_models.joblib")
    rows = []
    for name, ensemble in bundle["models"].items():
        for index, member in enumerate(ensemble["members"]):
            classifier = member["base_model"].pipeline.named_steps["classifier"]
            rows.append({"model": name, "member": index, "features": len(member["base_model"].columns),
                         "support_vectors": int(classifier.support_vectors_.shape[0]) if name == "svm" else None,
                         "linear_coefficients": int(classifier.coef_.size) if name == "logreg" else None})
    return pd.DataFrame(rows)


def save_figure(fig, directory, name):
    fig.savefig(directory / f"{name}.png", dpi=210, bbox_inches="tight", facecolor="white")
    fig.savefig(directory / f"{name}.svg", bbox_inches="tight", facecolor="white")
    plt.close(fig)


def make_figures(context, history, directory):
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "axes.unicode_minus": False, "svg.fonttype": "none"})
    ci = context["intervals"]
    fig, axes = plt.subplots(1, 4, figsize=(8.4, 3.3))
    for ax, metric, title in zip(axes, ["balanced_accuracy", "sensitivity", "specificity", "roc_auc"], ["Balanced accuracy", "Sensitivity", "Specificity", "ROC AUC"]):
        for index, name in enumerate(MODEL_NAMES):
            row = ci.loc[ci.comparison.eq("absolute") & ci.model.eq(name) & ci.condition.eq("clean") & ci.metric.eq(metric)].iloc[0]
            ax.plot([index, index], [row.ci_low, row.ci_high], color=COLORS[name], lw=2)
            ax.scatter(index, row.estimate, color=COLORS[name], s=45, zorder=3)
        ax.set(xticks=[0, 1], xticklabels=["LR", "SVM"], ylim=(.45, 1.01), title=title)
        ax.grid(axis="y", alpha=.18)
    fig.suptitle("Held-out clean performance with pointwise 95% intervals", fontsize=13)
    fig.tight_layout()
    save_figure(fig, directory, FIGURES[0])
    fig, ax = plt.subplots(figsize=(7, 4))
    for name in MODEL_NAMES:
        p = context["predictions"].loc[context["predictions"].model.eq(name) & context["predictions"].condition.eq("clean")]
        fpr, tpr, _ = roc_curve(p.actual, p.prob_malignant)
        auc = context["metrics"].loc[context["metrics"].model.eq(name) & context["metrics"].condition.eq("clean"), "roc_auc"].iloc[0]
        ax.plot(fpr, tpr, label=f"{LABELS[name]}  AUC {auc:.4f}", color=COLORS[name], lw=2)
    ax.plot([0, 1], [0, 1], "--", color=".65", lw=1)
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Held-out clean ROC curves", xlim=(0, 1), ylim=(0, 1.02))
    ax.legend(loc="lower right", frameon=False)
    ax.grid(alpha=.15)
    save_figure(fig, directory, FIGURES[1])
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, name in zip(axes, MODEL_NAMES):
        p = context["predictions"].loc[context["predictions"].model.eq(name) & context["predictions"].condition.eq("clean")]
        bins = pd.DataFrame(pc.calibration_bins(p, name)).dropna()
        ax.plot([0, 1], [0, 1], "--", color=".65")
        ax.plot(bins.mean_probability, bins.malignant_fraction, color=COLORS[name], lw=1.5)
        ax.scatter(bins.mean_probability, bins.malignant_fraction, s=15 + 2*bins.patients, color=COLORS[name], alpha=.8)
        ax.set(xlabel="Mean predicted probability", ylabel="Observed malignant fraction", title=LABELS[name], xlim=(0, 1), ylim=(0, 1))
        ax.grid(alpha=.15)
    fig.suptitle("Held-out reliability by fixed bins  |  marker size reflects patient count", fontsize=12)
    fig.tight_layout()
    save_figure(fig, directory, FIGURES[2])
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    for ax, name in zip(axes, MODEL_NAMES):
        row = context["metrics"].loc[context["metrics"].model.eq(name) & context["metrics"].condition.eq("clean")].iloc[0]
        matrix = np.array([[row.tn, row.fp], [row.fn, row.tp]], dtype=int)
        ax.imshow(matrix, cmap="Blues", vmin=0, vmax=max(1, matrix.max()))
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=20,
                        color="white" if matrix[i, j] > matrix.max()*.55 else "black")
        ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"],
               xlabel="Predicted diagnosis", ylabel="Actual diagnosis", title=LABELS[name])
    fig.suptitle("Held-out clean confusion matrices at probability threshold 0.5")
    fig.tight_layout()
    save_figure(fig, directory, FIGURES[3])
    fig, ax = plt.subplots(figsize=(8.5, 3.8))
    for index, name in enumerate(MODEL_NAMES):
        for pos, condition in enumerate(COHERENT_CONDITIONS):
            row = ci.loc[ci.comparison.eq("change_from_clean") & ci.model.eq(name) & ci.condition.eq(condition) & ci.metric.eq("balanced_accuracy")].iloc[0]
            y = pos + (-.12 if index == 0 else .12)
            ax.plot([100*row.ci_low, 100*row.ci_high], [y, y], color=COLORS[name], lw=2)
            ax.scatter(100*row.estimate, y, color=COLORS[name], s=35, label=LABELS[name] if pos == 0 else None, zorder=3)
    ax.axvline(0, color=".5", ls="--", lw=1)
    ax.set(yticks=range(4), yticklabels=[COND_LABELS[c] for c in COHERENT_CONDITIONS],
           xlabel="Perturbed minus clean balanced accuracy  (percentage points)", title="Held-out coherent boundary sensitivity")
    ax.invert_yaxis()
    ax.legend(frameon=False, fontsize=9, loc="best")
    ax.grid(axis="x", alpha=.15)
    fig.tight_layout()
    save_figure(fig, directory, FIGURES[4])
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), sharey=True)
    for ax, name in zip(axes, MODEL_NAMES):
        for offset, key, label, color in ((-.18, "original", "Original method", "#89959E"), (.18, "coherent", "Coherent method", COLORS[name])):
            table = history[key].loc[history[key].model.eq(name)].set_index("condition")
            values = [100*float(table.loc[c, "delta_balanced_accuracy"]) for c in COHERENT_CONDITIONS]
            ax.bar(np.arange(4)+offset, values, .34, label=label, color=color)
        ax.axhline(0, color=".4", lw=.7)
        ax.set(xticks=range(4), xticklabels=["Mild\nerosion", "Mild\ndilation", "Moderate\nerosion", "Moderate\ndilation"], title=LABELS[name])
        ax.legend(frameon=False, fontsize=8)
    axes[0].set_ylabel("BA change  (percentage points)")
    fig.suptitle("Development sensitivity to perturbation construction  |  original clean models")
    fig.tight_layout()
    save_figure(fig, directory, FIGURES[5])
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5), sharey=True)
    for ax, name in zip(axes, MODEL_NAMES):
        table = history["selection"].loc[history["selection"].model.eq(name)].set_index("arm")
        for offset, arm, label, color in ((-.18, "all_23_matched_tuning", "All 23", "#89959E"), (.18, "stability_topk", "Stability subset", COLORS[name])):
            row = table.loc[arm]
            ax.plot(np.arange(2)+offset, [100*row.balanced_accuracy, 100*row.mean_five_condition_ba],
                    marker="o", linestyle="none", ms=8, label=label, color=color)
        ax.set(xticks=[0, 1], xticklabels=["Clean BA", "Five-condition BA"], title=LABELS[name], ylim=(75, 91))
        ax.legend(frameon=False, fontsize=8)
    axes[0].set_ylabel("Balanced accuracy  (%)")
    fig.suptitle("Nested development evaluation of the frozen stability-selection procedure")
    fig.tight_layout()
    save_figure(fig, directory, FIGURES[6])


def interval_text(context, name, metric, condition="clean", percent=True):
    row = context["intervals"].loc[context["intervals"].comparison.eq("absolute") & context["intervals"].model.eq(name)
                                    & context["intervals"].condition.eq(condition) & context["intervals"].metric.eq(metric)].iloc[0]
    factor, decimals = (100, 2) if percent else (1, 4)
    return f"{factor*row.estimate:.{decimals}f} [{factor*row.ci_low:.{decimals}f}, {factor*row.ci_high:.{decimals}f}]"


def build_sections(context, history, overlap, examples, sizes):
    clean = context["metrics"].loc[context["metrics"].condition.eq("clean")].set_index("model")
    summary, primary, manifest = context["summary"], context["primary"], context["manifest"]
    lr, svm = clean.loc["logreg"], clean.loc["svm"]
    diff = context["intervals"].loc[context["intervals"].comparison.eq("svm_minus_logreg") & context["intervals"].condition.eq("clean")
                                     & context["intervals"].metric.eq("balanced_accuracy")].iloc[0]
    total = manifest.Case.nunique()
    cohorts = []
    for split, label in ((None, "All patients"), ("development", "Development"), ("test", "Held-out test")):
        part = manifest if split is None else manifest.loc[manifest.split.eq(split)]
        patients = part.groupby("Case").Pathology.first()
        cohorts.append([label, str(len(part)), str(len(patients)), str(int(patients.eq("benign").sum())), str(int(patients.eq("malignant").sum()))])
    selection_table = [[LABELS[r.model], "All 23" if r.arm == "all_23_matched_tuning" else "Stability subset",
                        f"{100*r.balanced_accuracy:.2f}", f"{100*r.mean_five_condition_ba:.2f}"] for r in history["selection"].itertuples()]
    clean_table = [[label, interval_text(context, "logreg", metric, percent=pct), interval_text(context, "svm", metric, percent=pct)]
                   for metric, label, pct in (("balanced_accuracy", "Balanced accuracy (%)", True), ("sensitivity", "Sensitivity (%)", True),
                                             ("specificity", "Specificity (%)", True), ("roc_auc", "ROC AUC", False),
                                             ("brier_score", "Brier score", False), ("log_loss", "Log loss", False))]
    pair_table = [["LR" if r.model == "logreg" else "SVM", COND_LABELS[r.condition], f"{100*r.delta_balanced_accuracy:+.2f}",
                   f"{100*r.flip_rate:.2f}", str(int(r.new_false_negatives)), str(int(r.recovered_false_negatives))]
                  for r in context["pair_metrics"].itertuples()]
    baseline_table = [[LABELS[n], f"{100*history['baseline'][n]['balanced_accuracy']:.2f}",
                       f"{history['baseline'][n]['roc_auc']:.4f}"] for n in MODEL_NAMES]
    calibration_table = [[LABELS[r.model], f"{100*r.balanced_accuracy:.2f}", f"{100*r.sensitivity:.2f}",
                          f"{100*r.specificity:.2f}", f"{r.brier_score:.4f}"] for r in history["calibration"].itertuples()]
    costs = [[r.stage, str(int(r.base_inner_fits)), str(int(r.base_refits)), str(int(r.calibrator_fits)),
              "Not recorded" if pd.isna(r.recorded_seconds) else f"{r.recorded_seconds:.1f}"] for r in history["costs"].itertuples()]
    def section(title, paragraphs=(), table=None, figures=()):
        return {"title": title, "paragraphs": list(paragraphs), "table": table, "figures": list(figures)}
    return [
        section("Study purpose and main findings", [
            f"We studied how lesion-boundary changes affect breast-ultrasound classification using Logistic Regression (LR) and RBF SVM with 23 handcrafted features. The study includes {total:,} patients and separates development from a locked test cohort. All model selection and calibration decisions were completed before test predictions.",
            f"On {summary['held_out_patients']} held-out patients, the primary SVM clean balanced accuracy was {100*primary.estimate:.2f}% (pointwise 95% interval {100*primary.ci_low:.2f}-{100*primary.ci_high:.2f}%). LR achieved {100*lr.balanced_accuracy:.2f}%. SVM had slightly higher sensitivity and specificity, while LR had slightly better AUC, Brier score and log loss. This is not a uniform superiority result.",
            "The original mild erosion/dilation losses depended strongly on the perturbation mechanism. A separately labelled coherent alternative greatly reduced those losses on development data. The frozen stability-selection procedure did not improve the overall accuracy-robustness objective. Final held-out changes were small and sometimes raised balanced accuracy while worsening ranking or probability losses.",
            f"At the fixed 0.5 research threshold, SVM missed {int(svm.fn)} of {overlap['malignant_patients']} malignant test patients. These results support a reproducible methodological study, not clinical deployment."],
            figures=[{"file": FIGURES[0], "caption": "Figure 1. Clean held-out operating-point metrics and AUC. Bars show pointwise 95% patient-bootstrap intervals; all scores use a 0-1 axis."}]),
        section("Dataset and evaluation design", [
            "BUS-BRA version 1.0 is the public dataset associated with Gomez-Flores, Gregorio-Calas and Pereira [1,2]. We classify benign versus malignant pathology using the provided lesion delineations. BI-RADS labels are not input features in this experiment.",
            "The random seed is 42. Patients, rather than images, define the development/test separation and five locked outer folds. The original classifiers use three inner patient-stratified folds for tuning. Each patient contributes total training weight one across their views, with optional training-only class balancing. Patient scores average image scores.",
            "Images are converted to grayscale and resized with preserved aspect ratio to longest side 256. Masks use nearest-neighbor interpolation. The 23 features comprise six shape, seven intensity and ten masked texture features. Masked GLCM pairs are counted only when both pixels lie inside the lesion, using distances one and three.",
            "Shape features are area fraction, aspect ratio, eccentricity, solidity, extent and circularity. Intensity features are mean, SD, median, 10th and 90th percentiles, skewness and entropy. Texture features are contrast, dissimilarity, homogeneity, angular second moment and correlation at each of the two GLCM distances. The existing extractor fixes intensity normalization, quantization and direction averaging; its source fingerprint is preserved.",
            "LR tunes C over 0.01, 0.1, 1, 10 and 100; RBF SVM tunes C over 0.1, 1 and 10 and gamma over scale, 0.01 and 0.1. Both consider class balancing off/on. The original inner objective is patient BA, with AUC and simpler settings as tie-breakers. Later robustness-oriented searches use the equal average of clean and four coherent BAs within each inner fold, then average over the three folds. The five-condition objective gives clean performance 20% of the total weight.",
            "Clean scaling and classifier fitting never use perturbed training masks. Coherent inner-validation conditions may score the explicitly robustness-oriented candidate objective in later experiments. Final calibration uses additional patient-disjoint base/calibration splits. The held-out test is used once with frozen models, and reporting reuses its saved predictions."],
            table={"headers": ["Cohort", "Images", "Patients", "Benign", "Malignant"], "rows": cohorts}),
        section("Original and coherent development robustness", [
            "The first experiment established clean nested patient-level baselines. The original boundary experiment then assessed eight mask conditions: mild/moderate erosion and dilation, plus four translations. Unavailable translated views were retained as unavailable; their patients were excluded only from the affected paired condition metrics.",
            "Unexpectedly large mild-condition losses motivated a development diagnostic. Pseudo-random selection among boundary pixels could create scattered contour irregularities and change perimeter-dependent features. An independent coherent alternative grows localized, connected boundary edits; mild and moderate edits are nested. Their IoU targets are approximately 0.95 and 0.90.",
            "The original results were preserved. Coherent masks were audited and their features extracted separately. Clean out-of-fold predictions reproduced for both classifiers before robustness scoring. The large original mild-condition BA losses shrank with coherent edits. This supports sensitivity to perturbation construction; it does not validate either mechanism as a realistic clinical annotation-error model.",
            "The development patient-bootstrap intervals were exploratory and conditional on saved OOF fits and observed fold/diagnosis counts. They omitted retraining variability and did not establish a uniform SVM advantage. CV uncertainty requires particular caution because training sets overlap [4]."],
            table={"headers": ["Original clean baseline", "BA (%)", "AUC"], "rows": baseline_table},
            figures=[{"file": FIGURES[5], "caption": "Figure 2. Original and coherent development BA changes for the same original clean-model procedure. This is a perturbation-construction comparison, not the final calibrated test model."}]),
        section("Frozen stability feature selection experiment", [
            "Feature stability was ranked separately inside every inner training partition. Each condition's absolute image-feature shifts were averaged within patient and then equally across patients, normalized by clean training weighted SD. The score was the largest of the four condition means. Constant or near-constant features were excluded from subset candidates.",
            "The frozen subset sizes were 5, 10 and 15. Both the subset arm and the all-23 comparator tuned the same model grids and the mean of clean plus four coherent patient BAs, averaged across three inner folds. Selected subsets were reranked on outer training patients before the outer refit. No global feature list or k was selected from outer results.",
            "Both classifiers selected 15 features in each outer fold. The subset procedure did not improve the pooled five-condition objective. Its lower clean baseline can make relative BA-change comparisons look favorable even when absolute perturbed accuracy remains lower. This negative result is retained rather than altering the grid after inspecting outer outcomes."],
            table={"headers": ["Model", "Arm", "Clean BA (%)", "Five-condition BA (%)"], "rows": selection_table},
            figures=[{"file": FIGURES[6], "caption": "Figure 3. Matched nested development comparison. These are point estimates for the complete selection procedures; the plot does not imply statistical significance."}]),
        section("Patient calibration and final protocol", [
            "The final procedure keeps all 23 features. SVM was designated primary from the Step 19 development five-condition objective; LR remains the comparator. This is a post-hoc development-based decision, not a significance claim. The sigmoid method, three-member ensemble and probability threshold 0.5 were fixed before calibration evaluation.",
            "Within each outer training pool, each member reserves one of three patient folds for calibration. Its base model is independently tuned using three inner patient folds within the other two folds, then fitted only on clean base images. Calibration receives one clean score per reserved patient: logit of mean LR image probability, or mean SVM margin. A fixed regularized sigmoid with equal patient weights is fitted to those scores. This keeps base tuning, base fitting and calibration patients disjoint [3].",
            "The patient calibrator uses StandardScaler followed by LogisticRegression with C=1, max_iter=2000 and random_state=42, without class or image-count weights. LR probabilities are clipped to [1e-6, 1-1e-6] before the logit. Calibration split seeds are 2020 plus scope code; base-tuning seeds are 3000 plus ten times scope code plus member index. Scope codes are 0-4 for outer evaluation and 5 for final fitting.",
            "For prediction, image scores are aggregated within each member, calibrated at patient level, then the three probabilities are averaged. The same procedure fits the final development-only ensembles. Outer-validation patients influence none of its tuning, scaling, fitting or calibration operations.",
            "Calibration quality is measured rather than assumed. At 0.5, outer development sensitivity fell and specificity rose relative to the earlier single-refit estimators. The ensemble has different member training sizes and settings, so cross-step differences cannot be attributed solely to the sigmoid. The threshold was not changed after observing this trade-off."],
            table={"headers": ["Calibrated outer CV", "BA (%)", "Sensitivity (%)", "Specificity (%)", "Brier"], "rows": calibration_table}),
        section("Locked held out primary performance", [
            f"The primary result is SVM clean BA {100*primary.estimate:.2f}% [{100*primary.ci_low:.2f}, {100*primary.ci_high:.2f}]. The test cohort contains {overlap['benign_patients']} benign and {overlap['malignant_patients']} malignant patients. Table intervals are pointwise percentiles from 10,000 shared patient resamples within diagnosis, seed 42, conditional on the frozen models.",
            f"The clean SVM-minus-LR BA difference is {100*diff.estimate:+.2f} percentage points [{100*diff.ci_low:.2f}, {100*diff.ci_high:.2f}]. Its interval includes zero at the boundary. LR has the higher clean AUC and lower probability losses. The evidence does not support a universal claim that SVM is better on every measure.",
            "The test is an internal patient hold-out from this dataset, not an independent external clinical cohort. Secondary intervals are not adjusted for multiple comparisons. No threshold, feature or model changes follow the test results."],
            table={"headers": ["Metric", "LR estimate [95% interval]", "SVM estimate [95% interval]"], "rows": clean_table},
            figures=[{"file": FIGURES[1], "caption": "Figure 4. Clean test ROC curves from saved probabilities. AUC measures ranking over thresholds and differs from BA at the fixed operating point."}]),
        section("Probability reliability and diagnostic errors", [
            f"Clean Brier scores were {lr.brier_score:.4f} for LR and {svm.brier_score:.4f} for SVM; log losses were {lr.log_loss:.4f} and {svm.log_loss:.4f}. Lower is better for these losses. Reliability bins summarize the same held-out probabilities, with sparse bins interpreted cautiously.",
            f"At 0.5, LR had {int(lr.fn)} false negatives and {int(lr.fp)} false positives. SVM had {int(svm.fn)} false negatives and {int(svm.fp)} false positives. SVM sensitivity was {100*svm.sensitivity:.2f}% and specificity {100*svm.specificity:.2f}%. AUC near 0.91 does not remove the missed-malignancy burden at this threshold."],
            figures=[{"file": FIGURES[2], "caption": "Figure 5. Ten fixed-width reliability bins. Marker size reflects patient count; connecting lines do not establish reliable behavior in sparsely populated regions."},
                     {"file": FIGURES[3], "caption": "Figure 6. Frozen-threshold confusion matrices. Counts are patients, not individual image views."}]),
        section("Held out coherent boundary sensitivity", [
            "Every test image had all four valid coherent feature vectors. All paired comparisons retain the same complete patients and image views. Positive signed BA change means a higher perturbed BA relative to that model's own clean baseline; it does not imply that segmentation errors are beneficial.",
            "Some edits raise BA by moving a small number of scores across 0.5. At the same time, ranking AUC or probability losses can worsen. In particular, dilation reduces AUC for both classifiers in this test cohort. Report sensitivity, specificity, flips and probability quality alongside BA rather than interpreting one increase as global robustness.",
            "New FN counts identify malignant patients correctly classified clean and missed after perturbation; recovered FN counts identify the reverse transition. Balanced accuracy can be unchanged despite individual errors moving between patients."],
            table={"headers": ["Model", "Condition", "Delta BA (pp)", "Flip (%)", "New FN", "Recovered FN"], "rows": pair_table},
            figures=[{"file": FIGURES[4], "caption": "Figure 7. Held-out perturbed-minus-clean BA changes with paired pointwise 95% intervals. Zero denotes no aggregate BA change, not zero individual prediction changes."}]),
        section("Patient level failure analysis", [
            f"There were {overlap['both_wrong']} patients misclassified by both models, {overlap['lr_only_wrong']} misclassified by LR only, and {overlap['svm_only_wrong']} misclassified by SVM only; {overlap['both_correct']} were correct for both. The shared clean false-negative count was {overlap['both_false_negative']}. These overlaps use the same saved patient predictions.",
            "The example cases below are selected deterministically: up to three SVM false negatives with the lowest malignant probability, and up to three false positives with the highest probability. This is a descriptive confidence-error audit, not a blinded clinical review. The full CSV preserves every patient, image ID and per-model error flag.",
            "No image-quality, lesion-subtype or scanner-specific cause is inferred from a probability alone. Such explanations require a separate qualified image review and appropriate metadata. Confidence in an incorrect prediction is a reliability concern, not proof of a particular biological mechanism.",
            "The report preserves all failures and does not remove hard cases, optimize a new threshold, or revise the final model based on these test examples."],
            table={"headers": ["Example category", "Dataset case", "Actual", "LR probability", "SVM probability"],
                   "rows": [[r.category, str(r.Case), "Malignant" if r.actual else "Benign", f"{r.lr_probability:.4f}", f"{r.svm_probability:.4f}"] for r in examples.itertuples()]}),
        section("Computational cost and reproducibility", [
            "Fit counts document the cost of the complete procedures, including nested searches. Step 19 also used ten original-baseline reproducibility fits, separate from its twenty outer refits. The subset arm has an additional k search, so matched hyperparameter grids do not imply equal search budgets.",
            "The final held-out evaluation performed zero model fits, zero calibrator fits and no retuning. The report likewise performs no model prediction or fitting: it recomputes descriptive metrics from saved predictions, verifies the completed evaluation marker and its published-file checksums, and inspects fitted model sizes without executing inference.",
            f"The final SVM ensemble contains {int(sizes.support_vectors.fillna(0).sum())} support vectors across three members; the LR ensemble contains {int(sizes.linear_coefficients.fillna(0).sum())} base linear coefficients. The compressed combined model artifact is {context['model_bytes']/1024/1024:.2f} MiB. These are structural/size counts, not a controlled inference-speed benchmark.",
            "Only recorded runtimes are reported. Calibration and test runtimes were not retained; hardware-normalized speed, memory and energy comparisons cannot be reconstructed. Full input/output hashes, code fingerprints and provenance are supplied separately. The saved primary protocol and one-time evaluation marker remain unchanged."],
            table={"headers": ["Stage", "Inner base fits", "Base refits", "Sigmoid fits", "Recorded seconds"], "rows": costs}),
        section("Limitations and conclusions", [
            "The data come from one public dataset. Patient separation prevents view leakage but does not establish scanner-, institution- or population-independent generalization. The study uses provided lesion masks; an end-to-end system with an automatic segmenter was not evaluated.",
            "Boundary edits are synthetic and deterministic. Their IoU targets describe mask overlap, not guaranteed clinical plausibility. The large original mild-condition effect was mechanism-dependent, motivating a clearly labelled post-hoc coherent sensitivity analysis. Development-informed choices and diagnostics are not retrospective preregistration.",
            "Stability selection was evaluated only for the frozen score, candidate sizes and objectives. Selecting 15 features in every fold does not justify expanding the grid after seeing these results. The negative finding does not prove that all stability-aware methods fail.",
            "Calibration does not establish clinical risk validity. The fixed 0.5 threshold was not selected for clinical sensitivity, costs or workflow. The observed missed malignant patients preclude treating this report as a deployment recommendation. Uncertainty intervals are pointwise and conditional on the fixed fits and observed diagnosis composition; they do not include retraining or external-distribution variability.",
            "The completed project supports three conclusions: perturbation construction materially affects measured boundary robustness; the frozen stability-subset procedure did not improve the overall development objective; and the frozen calibrated SVM attained the reported held-out BA with a substantial sensitivity limitation. LR retained small advantages on AUC and probability losses. These findings should be reported together, without claiming universal SVM superiority or changing the locked evaluation after inspection."]),
        section("References and project handoff", [
            "[1] Gomez-Flores W, Gregorio-Calas MJ, Pereira WCA. BUS-BRA: A Breast Ultrasound Dataset for Assessing Computer-aided Diagnosis Systems. Medical Physics. 2024;51:3110-3123. DOI: 10.1002/mp.16812.",
            "[2] BUS-BRA version 1.0 dataset. Zenodo. https://zenodo.org/records/8231412. Dataset DOI: 10.5281/zenodo.8231412.",
            "[3] Scikit-learn documentation. Probability calibration. https://scikit-learn.org/stable/modules/calibration.html. Also: Nested versus non-nested cross-validation, https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html.",
            "[4] Bates S, Hastie T, Tibshirani R. Cross-validation: what does it estimate and how well does it do it? arXiv:2104.00673. https://arxiv.org/abs/2104.00673.",
            "Scientific evaluation is complete. No further planned experiment remains. The remaining handoff is author review, any course-specific formatting or submission requirements, and preservation of the source code, frozen model/protocol, evaluation marker, saved results and report checksums. Future work requires a separately documented evaluation plan; it must not reuse this test set for tuning."])
    ]


def report_markdown(document):
    lines = [f"# {document['title']}", "", document["authors"], "", f"Final research report | {document['date']}", ""]
    for section in document["sections"]:
        lines.extend([f"## {section['title']}", ""])
        for paragraph in section["paragraphs"]:
            lines.extend([paragraph, ""])
        if section["table"]:
            table = section["table"]
            lines.extend(["| " + " | ".join(table["headers"]) + " |", "| " + " | ".join(["---"]*len(table["headers"])) + " |"])
            lines.extend("| " + " | ".join(str(v) for v in row) + " |" for row in table["rows"])
            lines.append("")
        for figure in section["figures"]:
            lines.extend([f"![{figure['caption']}](figures/{figure['file']}.png)", "", figure["caption"], ""])
    return "\n".join(lines)


def report_html(document, output):
    parts = ["<!doctype html><html lang='en'><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'>",
             f"<title>{html.escape(document['title'])}</title><style>body{{font:16px/1.6 Georgia,serif;color:#17232d;max-width:1020px;margin:48px auto;padding:0 26px}}h1,h2{{font-family:Arial,sans-serif;color:#000}}h1{{font-size:32px;line-height:1.2}}h2{{margin-top:2.3em;font-size:23px}}p{{max-width:90ch}}table{{border-collapse:collapse;width:100%;font:14px/1.4 Arial,sans-serif;margin:22px 0}}th,td{{border:1px solid #d9d9d9;padding:9px 11px;text-align:left}}th{{background:#eaf0f4;color:#000}}tr:nth-child(even){{background:#f7f9fa}}figure{{margin:24px 0}}img{{width:100%;height:auto}}figcaption{{font:13px/1.5 Arial,sans-serif;color:#475461;margin-top:9px}}.byline{{font:14px Arial,sans-serif;color:#475461}}@media print{{body{{font-size:10pt;margin:0;max-width:none}}figure,table{{break-inside:avoid}}h2{{break-after:avoid}}@page{{margin:18mm}}}}</style>",
             f"<h1>{html.escape(document['title'])}</h1><p class='byline'>{html.escape(document['authors'])}<br>Final research report | {document['date']}</p>"]
    for section in document["sections"]:
        parts.append(f"<section><h2>{html.escape(section['title'])}</h2>")
        parts.extend(f"<p>{html.escape(p)}</p>" for p in section["paragraphs"])
        if section["table"]:
            t = section["table"]
            parts.append("<table><thead><tr>" + "".join(f"<th>{html.escape(h)}</th>" for h in t["headers"]) + "</tr></thead><tbody>")
            parts.extend("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>" for row in t["rows"])
            parts.append("</tbody></table>")
        for figure in section["figures"]:
            encoded = base64.b64encode((output / "figures" / f"{figure['file']}.png").read_bytes()).decode()
            parts.append(f"<figure><img alt='{html.escape(figure['caption'], quote=True)}' src='data:image/png;base64,{encoded}'><figcaption>{html.escape(figure['caption'])}</figcaption></figure>")
        parts.append("</section>")
    return "\n".join(parts) + "</html>"


def run(project, output):
    if output.exists():
        raise FileExistsError("Report output exists; choose a new --output directory")
    paths = source_paths(project)
    input_files = [p for key, path in paths.items() for p in ([path] if path.is_file() else path.glob("*.json"))]
    input_files += [p for key in ("heldout", "step19", "step16", "step17", "original", "calibration") for p in paths[key].glob("*.csv")]
    input_files += [paths["calibration"] / "frozen_calibrated_models.joblib"]
    before = {str(p.resolve()): pc.sha256(p) for p in set(input_files)}
    context = validate_saved_evaluation(paths)
    history = load_history(paths)
    cases, overlap, examples = failure_analysis(context["predictions"], context["manifest"])
    sizes = model_sizes(paths["calibration"])
    context["model_bytes"] = (paths["calibration"] / "frozen_calibrated_models.joblib").stat().st_size
    document = {"title": "Breast ultrasound classification under lesion boundary uncertainty",
                "authors": "Mostafa Bdeir and Hadi Fardan", "date": "2026-11-15",
                "sections": build_sections(context, history, overlap, examples, sizes)}
    output.mkdir(parents=True, exist_ok=False)
    (output / "figures").mkdir()
    make_figures(context, history, output / "figures")
    (output / "BUS_BRA_Final_Report.md").write_text(report_markdown(document), encoding="utf-8")
    (output / "BUS_BRA_Final_Report.html").write_text(report_html(document, output), encoding="utf-8")
    (output / "report_content.json").write_text(ss.canonical_json(document), encoding="utf-8")
    for filename, table in (("patient_failure_audit.csv", cases), ("illustrative_failure_cases.csv", examples),
                            ("computational_cost.csv", history["costs"]), ("final_model_sizes.csv", sizes),
                            ("heldout_clean_comparison.csv", context["metrics"].loc[context["metrics"].condition.eq("clean")]),
                            ("heldout_robustness_table.csv", context["pair_metrics"])):
        table.to_csv(output / filename, index=False, float_format="%.12g")
    if any(pc.sha256(Path(p)) != digest for p, digest in before.items()):
        raise RuntimeError("An existing study result changed during reporting")
    summary = {"step": 22, "status": "final_scientific_report_generated", "model_fits": 0, "new_predictions": 0,
               "heldout_evaluation_rerun": False, "prior_files_unchanged": True,
               "planned_experiments_remaining": 0, "figure_count": len(FIGURES), "failure_overlap": overlap,
               "primary_result": context["summary"]["primary_result"], "inputs_sha256": before,
               "source_sha256": pc.sha256(Path(__file__)),
               "outputs_sha256": {str(p.relative_to(output)): pc.sha256(p) for p in output.rglob("*") if p.is_file()},
               "remaining_handoff": "author review, course-specific formatting/submission and preservation; no further model selection"}
    (output / "report_manifest.json").write_text(ss.canonical_json(summary), encoding="utf-8")
    print(f"STEP 22 — FINAL SCIENTIFIC REPORT\nVerified complete held-out checkpoint; prior files unchanged\n"
          f"Figures: {len(FIGURES)} | Model fits: 0 | New predictions: 0\nPlanned experiments remaining: 0\nSaved: {output}")
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--project", type=Path, default=ROOT)
    parser.add_argument("--output", type=Path, default=ROOT / "reports/final_step22")
    args = parser.parse_args()
    run(args.project, args.output)


if __name__ == "__main__":
    main()
```

**`src/heldout_evaluation.py`**

```python
"""Step 21: one frozen held-out evaluation; no model fitting or selection.

--preflight verifies files/versions/models without reading test images or predicting.
The real run creates an exclusive marker in the calibration checkpoint BEFORE
opening test images. A changed output path cannot bypass that marker.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from importlib.metadata import version
from pathlib import Path
from zipfile import ZipFile

import joblib
import numpy as np
import pandas as pd

from . import patient_calibration as pc
from . import stable_selection as ss
from .coherent_features import COHERENT_CONDITIONS
from .coherent_perturbations import coherent_boundary_change
from .features import FEATURE_COLUMNS, extract_features, preprocess_pair, read_pair
from .models import ROOT
from .perturbations import TARGET_IOUS, mask_iou
from .robustness_eval import MODEL_NAMES, summarize_pair

CONDITIONS = ("clean", *COHERENT_CONDITIONS)
METRICS = ("balanced_accuracy", "sensitivity", "specificity", "roc_auc", "brier_score", "log_loss")
MARKER = "held_out_evaluation_started.json"
UNCERTAINTY_PLAN = "10000 paired patient bootstrap replicates within test diagnosis, seed 42; pointwise percentile 95%; fixed fitted models"


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, value):
    path.write_bytes(ss.canonical_json(value).encode("utf-8"))


def image_id_digest(ids):
    return hashlib.sha256(json.dumps(sorted(str(i) for i in ids)).encode()).hexdigest()


def verify_frozen(calibration, manifest_path, archive_path, audit_path):
    protocol_path, summary_path = calibration / "final_evaluation_protocol.json", calibration / "calibration_summary.json"
    protocol, summary = read_json(protocol_path), read_json(summary_path)
    model_path = calibration / "frozen_calibrated_models.joblib"
    if (summary.get("status") != "calibration_complete_final_protocol_frozen"
            or summary.get("held_out_evaluation_used") is not False
            or summary.get("final_protocol_sha256") != pc.sha256(protocol_path)
            or protocol.get("status") != "frozen_before_held_out_evaluation"
            or protocol.get("design") != pc.calibration_design()
            or protocol.get("primary_model") != "svm" or protocol.get("comparator") != "logreg"
            or protocol.get("threshold") != .5 or protocol.get("held_out_conditions") != list(CONDITIONS)
            or protocol.get("uncertainty_plan") != UNCERTAINTY_PLAN):
        raise ValueError("Frozen protocol/summary verification failed")
    if pc.sha256(model_path) != protocol.get("model_artifact_sha256") or summary.get("model_artifact_sha256") != protocol.get("model_artifact_sha256"):
        raise ValueError("Frozen model artifact hash mismatch")
    source_root = Path(pc.__file__).resolve().parent
    for filename, fingerprint in protocol["source_sha256"].items():
        if Path(filename).name != filename or pc.sha256(source_root / filename) != fingerprint:
            raise ValueError(f"Frozen source hash mismatch: {filename}")
    for package, expected in summary["versions"].items():
        if version(package) != expected:
            raise ValueError(f"Frozen package version mismatch: {package}")
    input_checks = {}
    for key, item in protocol["inputs"].items():
        path = manifest_path if key == "manifest" else Path(item["path"])
        if pc.sha256(path) != item["sha256"]:
            raise ValueError(f"Frozen training input hash mismatch: {key}")
        input_checks[key] = {"path": str(path.resolve()), "sha256": item["sha256"]}
    manifest = pd.read_csv(manifest_path, dtype={"ID": str})
    required = ["ID", "Case", "Pathology", "split", "dev_fold"]
    if (manifest[required].isna().any().any() or manifest.ID.duplicated().any()
            or not manifest.split.isin(["development", "test"]).all()
            or not manifest.Pathology.isin(["benign", "malignant"]).all()
            or manifest.groupby("Case").Pathology.nunique().ne(1).any()
            or manifest.groupby("Case").split.nunique().ne(1).any()):
        raise ValueError("Invalid locked patient manifest")
    test = manifest.loc[manifest.split.eq("test")].sort_values("ID").reset_index(drop=True)
    dev = manifest.loc[manifest.split.eq("development")]
    if (len(test) != protocol["held_out_images"] or test.Case.nunique() != protocol["held_out_patients"]
            or ss.case_digest(set(test.Case)) != protocol["held_out_case_sha256"]
            or image_id_digest(test.ID) != protocol["held_out_image_ids_sha256"]
            or set(test.Case) & set(dev.Case) or not test.dev_fold.eq(-1).all()
            or set(test.Pathology) != {"benign", "malignant"}):
        raise ValueError("Held-out cohort differs from frozen identifiers")
    archive_hash = pc.sha256(archive_path)  # Hash bytes only: no image decoding.
    if read_json(audit_path).get("archive_sha256") != archive_hash:
        raise ValueError("Dataset archive differs from the original dataset audit")
    bundle = joblib.load(model_path)  # Load only after artifact hash and version checks.
    if (bundle.get("design") != protocol["design"] or bundle.get("features") != list(FEATURE_COLUMNS)
            or bundle.get("development_case_sha256") != ss.case_digest(set(dev.Case))
            or set(bundle.get("models", {})) != set(MODEL_NAMES)):
        raise ValueError("Frozen model bundle metadata differs")
    for name, ensemble in bundle["models"].items():
        if (ensemble.get("model") != name or len(ensemble.get("members", [])) != 3
                or any(tuple(m["base_model"].columns) != FEATURE_COLUMNS for m in ensemble["members"])):
            raise ValueError("Frozen ensemble structure differs")
    checks = {"protocol_sha256": pc.sha256(protocol_path), "summary_sha256": pc.sha256(summary_path),
              "model_sha256": pc.sha256(model_path), "manifest_sha256": pc.sha256(manifest_path),
              "archive_sha256": archive_hash, "audit_sha256": pc.sha256(audit_path),
              "inputs": input_checks, "step21_source_sha256": pc.sha256(Path(__file__))}
    return protocol, bundle, test, checks


def extract_test_features(archive_path, test):
    clean_rows, changed_rows = [], []
    with ZipFile(archive_path) as archive:
        for index, row in enumerate(test.itertuples(index=False), 1):
            image, mask = preprocess_pair(*read_pair(archive, row.ID))
            initial_image, initial_mask = image.copy(), mask.copy()
            metadata = {"ID": row.ID, "Case": int(row.Case), "Pathology": row.Pathology,
                        "split": "test", "dev_fold": -1}
            clean_rows.append({**metadata, **extract_features(image, mask)})
            for condition in COHERENT_CONDITIONS:
                kind, severity = condition.split("_")
                changed = coherent_boundary_change(mask, TARGET_IOUS[severity], kind)
                if np.array_equal(changed, mask):
                    raise ValueError(f"Unavailable coherent condition: {row.ID}/{condition}")
                iou = mask_iou(mask, changed)
                if not 0 < iou < 1:
                    raise ValueError("Invalid held-out coherent IoU")
                changed_rows.append({**metadata, "condition": condition, "valid": True, "actual_iou": iou,
                                     "changed_pixels": int(np.logical_xor(mask, changed).sum()),
                                     **extract_features(image, changed)})
            if not np.array_equal(image, initial_image) or not np.array_equal(mask, initial_mask):
                raise ValueError("Original test image/mask was mutated")
            if index % 50 == 0:
                print(f"Held-out features: {index}/{len(test)} images", flush=True)
    clean, coherent = pd.DataFrame(clean_rows), pd.DataFrame(changed_rows)
    if (len(clean) != len(test) or clean.ID.duplicated().any() or set(clean.ID) != set(test.ID)
            or len(coherent) != len(test) * 4 or coherent.duplicated(["ID", "condition"]).any()
            or not np.isfinite(clean[list(FEATURE_COLUMNS)].to_numpy(float)).all()
            or not np.isfinite(coherent[list(FEATURE_COLUMNS)].to_numpy(float)).all()):
        raise ValueError("Incomplete or nonfinite held-out feature coverage")
    return clean, coherent


def predict_all(bundle, clean, coherent):
    predictions, pairs, summaries, reliability = [], [], [], []
    for name in MODEL_NAMES:
        ensemble = bundle["models"][name]
        baseline = pc.predict_ensemble(ensemble, clean)
        for condition in CONDITIONS:
            rows = baseline.copy() if condition == "clean" else pc.predict_ensemble(ensemble, coherent.loc[coherent.condition.eq(condition)])
            rows["model"], rows["condition"], rows["outer_fold"] = name, condition, -1
            predictions.append(rows)
            summaries.append({"model": name, "condition": condition, "patients": len(rows), **pc.probability_metrics(rows)})
            reliability.extend({**b, "condition": condition} for b in pc.calibration_bins(rows, name))
            if condition != "clean":
                changed = coherent.loc[coherent.condition.eq(condition)]
                p = rows.set_index("Case")
                pair = baseline[["Case", "actual", "n_images", "prob_malignant", "predicted"]].copy().rename(
                    columns={"prob_malignant": "clean_score", "predicted": "clean_predicted"})
                if set(p.index) != set(pair.Case) or not np.array_equal(pair.n_images, p.loc[pair.Case, "n_images"]):
                    raise ValueError("Missing held-out patient views")
                pair["perturbed_score"] = p.loc[pair.Case, "prob_malignant"].to_numpy(float)
                pair["perturbed_predicted"] = p.loc[pair.Case, "predicted"].to_numpy(int)
                pair["mean_iou"] = changed.groupby("Case").actual_iou.mean().loc[pair.Case].to_numpy(float)
                pair["outer_fold"], pair["eligible"], pair["model"], pair["condition"] = -1, True, name, condition
                pairs.append(pair)
    predictions, pairs = pd.concat(predictions, ignore_index=True), pd.concat(pairs, ignore_index=True)
    n = clean.Case.nunique()
    if (len(predictions) != n * 10 or predictions.duplicated(["model", "condition", "Case"]).any()
            or len(pairs) != n * 8 or pairs.duplicated(["model", "condition", "Case"]).any()):
        raise ValueError("Incomplete held-out prediction coverage")
    pair_summary = [summarize_pair(p, model=name, condition=condition)
                    for (name, condition), p in pairs.groupby(["model", "condition"])]
    return predictions, pairs, pd.DataFrame(summaries), pd.DataFrame(pair_summary), pd.DataFrame(reliability)


def bootstrap_weights(y, *, resamples=10000, seed=42):
    if set(y) != {0, 1} or resamples < 2:
        raise ValueError("Patient bootstrap requires both diagnoses")
    rng = np.random.default_rng(seed)
    counts = np.zeros((resamples, len(y)), dtype=np.int32)
    for label in (0, 1):
        ix = np.flatnonzero(y == label)
        counts[:, ix] = rng.multinomial(len(ix), np.full(len(ix), 1 / len(ix)), size=resamples)
    return counts


def auc_from_counts(y, scores, counts):
    order = np.argsort(scores, kind="stable")
    starts = np.r_[0, np.flatnonzero(np.diff(scores[order]) != 0) + 1]
    positive = np.add.reduceat(counts[:, order] * y[order], starts, axis=1)
    negative = np.add.reduceat(counts[:, order] * (1 - y[order]), starts, axis=1)
    numerator = np.sum(positive * (np.cumsum(negative, axis=1) - .5 * negative), axis=1)
    denominator = positive.sum(axis=1) * negative.sum(axis=1)
    if (denominator == 0).any():
        raise ValueError("Bootstrap AUC sample lost a diagnosis")
    return numerator / denominator


def metric_draws(y, probability, counts):
    p = np.asarray(probability, dtype=float)
    if not np.isfinite(p).all() or not ((p >= 0) & (p <= 1)).all():
        raise ValueError("Invalid bootstrap probability")
    pred = (p >= .5).astype(int)
    malignant, benign = counts @ y, counts @ (1 - y)
    sensitivity = (counts @ (y * pred)) / malignant
    specificity = (counts @ ((1 - y) * (1 - pred))) / benign
    n = counts.sum(axis=1)
    eps = np.finfo(float).eps
    loss = -(y * np.log(np.clip(p, eps, 1 - eps)) + (1 - y) * np.log(np.clip(1 - p, eps, 1 - eps)))
    return {"balanced_accuracy": .5 * (sensitivity + specificity), "sensitivity": sensitivity,
            "specificity": specificity, "roc_auc": auc_from_counts(y, p, counts),
            "brier_score": (counts @ ((p - y) ** 2)) / n, "log_loss": (counts @ loss) / n}


def paired_intervals(predictions, *, resamples=10000, seed=42):
    groups = {(name, condition): predictions.loc[predictions.model.eq(name) & predictions.condition.eq(condition)]
              .sort_values("Case").reset_index(drop=True) for name in MODEL_NAMES for condition in CONDITIONS}
    first = groups[("svm", "clean")]
    if first.Case.duplicated().any():
        raise ValueError("Duplicate bootstrap patient")
    for frame in groups.values():
        if (frame.Case.duplicated().any() or not np.array_equal(frame[["Case", "actual", "n_images"]], first[["Case", "actual", "n_images"]])):
            raise ValueError("Bootstrap comparisons are not patient-paired")
    y = first.actual.to_numpy(int)
    counts = bootstrap_weights(y, resamples=resamples, seed=seed)
    draws, points = {}, {}
    for key, frame in groups.items():
        draws[key] = metric_draws(y, frame.prob_malignant.to_numpy(float), counts)
        points[key] = pc.probability_metrics(frame)
    rows = []
    def emit(kind, model, condition, metric, point, distribution):
        low, high = np.quantile(distribution, [.025, .975], method="linear")
        rows.append({"comparison": kind, "model": model, "condition": condition, "metric": metric,
                     "estimate": float(point), "ci_low": float(low), "ci_high": float(high),
                     "patients": len(y), "resamples": resamples, "seed": seed,
                     "primary": kind == "absolute" and model == "svm" and condition == "clean" and metric == "balanced_accuracy",
                     "higher_is_better": metric not in ("brier_score", "log_loss", "flip_rate", "new_false_negative_rate"),
                     "interval_scope": "pointwise_percentile_conditional_on_frozen_models"})
    for name in MODEL_NAMES:
        for condition in CONDITIONS:
            key, base = (name, condition), (name, "clean")
            for metric in METRICS:
                emit("absolute", name, condition, metric, points[key][metric], draws[key][metric])
                if condition != "clean":
                    emit("change_from_clean", name, condition, metric,
                         points[key][metric] - points[base][metric], draws[key][metric] - draws[base][metric])
            if condition != "clean":
                clean_pred, changed_pred = groups[base].predicted.to_numpy(int), groups[key].predicted.to_numpy(int)
                flip = (clean_pred != changed_pred).astype(int)
                new_fn = y * (clean_pred == 1) * (changed_pred == 0)
                emit("paired_event", name, condition, "flip_rate", flip.mean(), (counts @ flip) / len(y))
                emit("paired_event", name, condition, "new_false_negative_rate", new_fn.sum() / y.sum(), (counts @ new_fn) / y.sum())
        composite_point = np.mean([points[(name, c)]["balanced_accuracy"] for c in CONDITIONS])
        composite_draw = np.mean([draws[(name, c)]["balanced_accuracy"] for c in CONDITIONS], axis=0)
        emit("five_condition_mean", name, "five_condition_mean", "balanced_accuracy", composite_point, composite_draw)
    for condition in CONDITIONS:
        for metric in METRICS:
            difference = points[("svm", condition)][metric] - points[("logreg", condition)][metric]
            distribution = draws[("svm", condition)][metric] - draws[("logreg", condition)][metric]
            emit("svm_minus_logreg", "both", condition, metric, difference, distribution)
            if condition != "clean":
                base_diff = points[("svm", "clean")][metric] - points[("logreg", "clean")][metric]
                base_draw = draws[("svm", "clean")][metric] - draws[("logreg", "clean")][metric]
                emit("svm_minus_logreg_change", "both", condition, metric, difference - base_diff, distribution - base_draw)
    emit("svm_minus_logreg_five_condition_mean", "both", "five_condition_mean", "balanced_accuracy",
         np.mean([points[("svm", c)]["balanced_accuracy"] - points[("logreg", c)]["balanced_accuracy"] for c in CONDITIONS]),
         np.mean([draws[("svm", c)]["balanced_accuracy"] - draws[("logreg", c)]["balanced_accuracy"] for c in CONDITIONS], axis=0))
    return pd.DataFrame(rows), counts, first[["Case", "actual"]]


def update_marker(path, record):
    temporary = path.with_suffix(".tmp")
    write_json(temporary, record)
    temporary.replace(path)


def run(calibration, manifest, archive, audit, output):
    if output.exists():
        raise FileExistsError("Held-out output exists; preserve it and do not rerun")
    marker = calibration / MARKER
    if marker.exists():
        raise FileExistsError("This frozen checkpoint already started its held-out evaluation; inspect the existing marker/results")
    protocol, bundle, test, checks = verify_frozen(calibration, manifest, archive, audit)
    record = {"status": "started", "output": str(output.resolve()), "phase": "feature_extraction", **checks}
    # Exclusive creation protects against concurrent runs and alternate output paths.
    with marker.open("x", encoding="utf-8") as handle:
        handle.write(ss.canonical_json(record))
    try:
        clean, coherent = extract_test_features(archive, test)
        record["phase"] = "prediction"
        update_marker(marker, record)
        predictions, pairs, summaries, pair_summary, reliability = predict_all(bundle, clean, coherent)
        record["phase"] = "bootstrap_and_publication"
        update_marker(marker, record)
        intervals, counts, patients = paired_intervals(predictions)  # Exactly frozen 10,000/42.
        if pc.sha256(archive) != checks["archive_sha256"] or pc.sha256(manifest) != checks["manifest_sha256"]:
            raise RuntimeError("Dataset or manifest changed during held-out evaluation")
        if pc.sha256(calibration / "frozen_calibrated_models.joblib") != checks["model_sha256"] or pc.sha256(calibration / "final_evaluation_protocol.json") != checks["protocol_sha256"]:
            raise RuntimeError("Frozen models or protocol changed during evaluation")
        primary = intervals.loc[intervals.primary].to_dict("records")
        if len(primary) != 1 or len(intervals) != 181:
            raise RuntimeError("Primary endpoint or interval coverage differs")
        result = {"step": 21, "status": "locked_held_out_evaluation_complete", "held_out_patients": int(test.Case.nunique()),
                  "held_out_images": len(test), "clean_feature_vectors": len(clean), "coherent_feature_vectors": len(coherent),
                  "patient_prediction_rows": len(predictions), "paired_patient_rows": len(pairs),
                  "primary_endpoint": protocol["primary_endpoint"], "primary_result": primary[0],
                  "threshold": .5, "model_fits": 0, "calibrator_fits": 0, "selection_or_retuning": False,
                  "bootstrap": {"resamples": 10000, "seed": 42, "method": "paired diagnosis-stratified multinomial patient counts; percentile 95%"},
                  "provenance": checks, "conditions": list(CONDITIONS), "clean_metrics": summaries.loc[summaries.condition.eq("clean")].to_dict("records"),
                  "limitations": "Single dataset research evaluation; posthoc development model choice; fixed research threshold; pointwise intervals conditional on fitted models; no clinical or multiplicity-adjusted claim."}
        output.mkdir(parents=True, exist_ok=False)
        for filename, table in (("held_out_clean_features.csv", clean), ("held_out_coherent_features.csv", coherent),
                                ("held_out_patient_predictions.csv", predictions), ("held_out_paired_predictions.csv", pairs),
                                ("held_out_condition_metrics.csv", summaries), ("held_out_paired_summary.csv", pair_summary),
                                ("held_out_reliability_bins.csv", reliability), ("held_out_intervals.csv", intervals)):
            table.to_csv(output / filename, index=False, float_format="%.12g")
        np.savez_compressed(output / "paired_bootstrap_weights.npz", counts=counts,
                            Case=patients.Case.to_numpy(), actual=patients.actual.to_numpy())
        write_json(output / "held_out_summary.json", result)
        write_json(output / "frozen_protocol_copy.json", protocol)
        result_hash = pc.sha256(output / "held_out_summary.json")
        record.update(status="complete", phase="complete", summary_sha256=result_hash,
                      published_files_sha256={p.name: pc.sha256(p) for p in sorted(output.iterdir()) if p.is_file()})
        update_marker(marker, record)
        print(summaries.loc[summaries.condition.eq("clean"), ["model", "balanced_accuracy", "sensitivity", "specificity", "roc_auc", "brier_score", "log_loss"]].to_string(index=False))
        p = primary[0]
        print(f"Primary SVM clean BA: {100*p['estimate']:.2f}% | pointwise 95% CI [{100*p['ci_low']:.2f}%, {100*p['ci_high']:.2f}%]\n"
              f"Frozen models only | Fits: 0 | Retuning: NONE\nSaved: {output}", flush=True)
        return result
    except Exception as error:
        record.update(status="failed", error_type=type(error).__name__, error=str(error))
        update_marker(marker, record)
        raise


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--calibration", type=Path, default=ROOT / "results/robustness/calibration_step20")
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--zip", type=Path, default=ROOT / "data/raw/BUSBRA.zip")
    parser.add_argument("--audit", type=Path, default=ROOT / "results/audit/summary.json")
    parser.add_argument("--output", type=Path, default=ROOT / "results/robustness/heldout_step21")
    parser.add_argument("--preflight", action="store_true", help="Integrity check only; no image decoding, predictions or marker creation")
    args = parser.parse_args()
    if args.preflight:
        _, _, test, checks = verify_frozen(args.calibration, args.manifest, args.zip, args.audit)
        print(f"STEP 21 PREFLIGHT: PASSED | {test.Case.nunique()} held-out patients | {len(test)} images\n"
              f"Protocol/models/sources/versions/archive: MATCHED\nImages decoded: 0 | Predictions: 0 | Marker created: NO\n"
              f"Protocol SHA256: {checks['protocol_sha256']}")
    else:
        run(args.calibration, args.manifest, args.zip, args.audit, args.output)


if __name__ == "__main__":
    main()
```

**`src/models.py`**

```python
"""Leakage-aware baseline for BUS-BRA breast-lesion classification.

Place this file at src/models.py and run from repository root:
    .\\.venv\\Scripts\\python.exe -m src.models

Uses only data/processed/development_features.csv and the LOCKED image split.
Evaluates a tuned Logistic Regression with 5 outer patient folds and 3 inner
patient folds. Predictions are averaged to one per patient before scoring.
No held-out test labels or features are read, and no test model is evaluated.

The 0.50 threshold is a *provisional baseline* operating point. Calibration,
clinical operating-threshold selection and SVM comparison come later.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .features import FEATURE_COLUMNS

ROOT = Path(__file__).resolve().parents[1]
SEED = 42
CS = (0.01, 0.1, 1.0, 10.0, 100.0)
BALANCE_OPTIONS = (False, True)
NUM_OUTER_FOLDS = 5
NUM_INNER_FOLDS = 3


def load_development(feature_csv: Path, manifest_csv: Path) -> pd.DataFrame:
    """Verify feature columns and that every row belongs to locked development."""
    df = pd.read_csv(feature_csv)
    manifest = pd.read_csv(manifest_csv)
    id_cols = ["ID", "Case", "Pathology", "split", "dev_fold"]
    required = id_cols + list(FEATURE_COLUMNS)
    if list(df.columns) != required:
        raise ValueError("Feature table schema/order differs from validated 23-feature extractor")
    if df[id_cols].isna().any().any() or df["ID"].duplicated().any():
        raise ValueError("Missing identifiers or duplicate image IDs")
    if not df["split"].eq("development").all():
        raise ValueError("Feature table contains non-development patients")
    if not df["Pathology"].isin(("benign", "malignant")).all():
        raise ValueError("Unexpected pathology class")
    if not np.isfinite(df[list(FEATURE_COLUMNS)].to_numpy(dtype=float)).all():
        raise ValueError("Missing or nonfinite feature values")
    if (df.groupby("Case")["Pathology"].nunique() != 1).any():
        raise ValueError("A patient has multiple pathology labels")
    if (df.groupby("Case")["dev_fold"].nunique() != 1).any():
        raise ValueError("A patient occurs in multiple development folds")
    folds = set(df["dev_fold"].astype(int).unique())
    if folds != set(range(NUM_OUTER_FOLDS)):
        raise ValueError(f"Expected development fold IDs 0..4, found {folds}")

    manifest = manifest.loc[manifest["split"] == "development", id_cols]
    if manifest["ID"].duplicated().any():
        raise ValueError("Manifest has duplicate development image IDs")
    left = df[id_cols].sort_values("ID").reset_index(drop=True)
    right = manifest.sort_values("ID").reset_index(drop=True)
    try:
        pd.testing.assert_frame_equal(left, right, check_dtype=False)
    except AssertionError as exc:
        raise ValueError("Feature table differs from locked development image manifest") from exc
    return df


def patient_image_weights(df: pd.DataFrame, balance: bool = False) -> np.ndarray:
    """Make each patient contribute total weight 1 (then optional case balancing)."""
    n_per_case = df.groupby("Case")["ID"].transform("size").to_numpy(dtype=float)
    weights = 1.0 / n_per_case
    if balance:
        patients = df.groupby("Case")["Pathology"].first()
        class_counts = patients.value_counts()
        if len(class_counts) != 2:
            raise ValueError("Class weighting requires two classes")
        n_cases = len(patients)
        case_factors = {label: n_cases / (2.0 * int(count)) for label, count in class_counts.items()}
        weights *= df["Pathology"].map(case_factors).to_numpy(dtype=float)
    return weights


def make_model(C: float) -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=float(C), max_iter=2000, random_state=SEED)),
    ])


def fit_model(df: pd.DataFrame, C: float, balance: bool) -> Pipeline:
    if df["Pathology"].nunique() != 2:
        raise ValueError("Training fold must contain both classes")
    w = patient_image_weights(df, balance=balance)
    model = make_model(C)
    X = df[list(FEATURE_COLUMNS)].to_numpy(dtype=float)
    y = df["Pathology"].eq("malignant").astype(int).to_numpy()
    # Training-only weighted scaling and training-only weighted classifier.
    model.fit(X, y, scaler__sample_weight=w, classifier__sample_weight=w)
    return model


def patient_predictions(model: Pipeline, df: pd.Data, fold: int | None = None) -> pd.DataFrame:
    X = df[list(FEATURE_COLUMNS)].to_numpy(dtype=float)
    probs = model.predict_proba(X)[:, 1]
    image_frame = pd.DataFrame({
        "Case": df["Case"].to_numpy(),
        "actual": df["Pathology"].eq("malignant").astype(int).to_numpy(),
        "prob_malignant": probs,
    })
    if (image_frame.groupby("Case")["actual"].nunique() != 1).any():
        raise ValueError("Inconsistent patient labels in prediction set")
    patient = image_frame.groupby("Case", as_index=False).agg(
        actual=("actual", "first"),
        prob_malignant=("prob_malignant", "mean"),
        n_images=("prob_malignant", "size"),
    )
    patient["predicted"] = (patient["prob_malignant"] >= 0.50).astype(int)
    if fold is not None:
        patient["outer_fold"] = int(fold)
    return patient


def score_patients(patients: pd.DataFrame) -> dict[str, float | int]:
    y = patients["actual"].to_numpy(dtype=int)
    yhat = patients["predicted"].to_numpy(dtype=int)
    probs = patients["prob_malignant"].to_numpy(dtype=float)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Validation patients must contain both diagnosis classes")
    tn, fp, fn, tp = (int(x) for x in confusion_matrix(y, yhat, labels=[0, 1]).ravel())
    return {
        "patients": len(patients),
        "balanced_accuracy": float(balanced_accuracy_score(y, yhat)),
        "sensitivity": float(tp / (tp + fn)),
        "specificity": float(tn / (tn + fp)),
        "roc_auc": float(roc_auc_score(y, probs)),
        "brier_score": float(brier_score_loss(y, probs)),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def inner_patient_folds(train: pd.DataFrame, seed: int, n_splits: int = NUM_INNER_FOLDS):
    """Construct stratified folds from UNIQUE PATIENTS, never image rows."""
    cases = train.groupby("Case", as_index=False)["Pathology"].first().sort_values("Case")
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for train_idx, val_idx in splitter.split(cases[["Case"]], cases["Pathology"]):
        train_cases = set(cases.iloc[train_idx]["Case"])
        val_cases = set(cases.iloc[val_idx]["Case"])
        assert train_cases.isdisjoint(val_cases)
        fit_rows = train[train["Case"].isin(train_cases)]
        val_rows = train[train["Case"].isin(val_cases)]
        yield fit_rows, val_rows


def choose_settings(train: pd.DataFrame, *, seed: int = SEED,
                    cs=CS, balances=BALANCE_OPTIONS,
                    n_splits: int = NUM_INNER_FOLDS) -> dict:
    """Choose hyperparameters by inner-fold patient BA (not outer test fold)."""
    inner = list(inner_patient_folds(train, seed=seed, n_splits=n_splits))
    ranking = []
    for C in cs:
        for balance in balances:
            scores = []
            aucs = []
            for fitting, validation in inner:
                model = fit_model(fitting, C=C, balance=balance)
                scored = score_patients(patient_predictions(model, validation))
                scores.append(scored["balanced_accuracy"])
                aucs.append(scored["roc_auc"])
            ranking.append({
                "C": float(C), "balance": bool(balance),
                "mean_inner_ba": float(np.mean(scores)),
                "mean_inner_auc": float(np.mean(aucs)),
            })
    # Transparent deterministic tie-break: AUC, then simpler regularized model.
    # On equal BA/AUC, prefer smaller C and no class balancing.
    return max(ranking, key=lambda x: (
        x["mean_inner_ba"], x["mean_inner_auc"],
        -x["C"], -int(x["balance"]),
    ))


def nested_logreg_cv(development: pd.DataFrame, *, cs=CS,
                     balances=BALANCE_OPTIONS) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    predictions = []
    fold_stats = []
    begin = perf_counter()
    for outer in range(NUM_OUTER_FOLDS):
        train = development[development["dev_fold"].ne(outer)]
        validation = development[development["dev_fold"].eq(outer)]
        assert set(train["Case"]).isdisjoint(set(validation["Case"]))
        selected = choose_settings(train, seed=SEED + outer,
                                   cs=cs, balances=balances)
        model = fit_model(train, C=selected["C"], balance=selected["balance"])
        predicted = patient_predictions(model, validation, fold=outer)
        metrics = score_patients(predicted)
        predictions.append(predicted)
        fold_stats.append({"outer_fold": outer, **selected, **metrics})
        print(f"Outer fold {outer}: {metrics['patients']} patients | "
              f"C={selected['C']:g}, balanced={selected['balance']} | "
              f"BA={metrics['balanced_accuracy']:.3f}, "
              f"AUC={metrics['roc_auc']:.3f}, FN={metrics['fn']}", flush=True)
    oof = pd.concat(predictions, ignore_index=True).sort_values("Case").reset_index(drop=True)
    if oof["Case"].duplicated().any() or len(oof) != development["Case"].nunique():
        raise RuntimeError("OOF predictions must contain each development patient exactly once")
    overall = score_patients(oof)
    overall.update({
        "method": "nested grouped CV (5 locked outer folds, 3 inner patient-stratified folds)",
        "threshold": 0.50,
        "threshold_status": "provisional (not clinically calibrated)",
        "features": len(FEATURE_COLUMNS),
        "test_patients_used": 0,
        "elapsed_cv_seconds": round(perf_counter() - begin, 3),
    })
    return oof, pd.DataFrame(fold_stats), overall


def run(feature_csv: Path, manifest_csv: Path, output_dir: Path, model_dir: Path) -> dict:
    development = load_development(feature_csv, manifest_csv)
    print("BUS-BRA — LOGISTIC REGRESSION BASELINE")
    print("=" * 45)
    print(f"Development: {len(development)} images, {development['Case'].nunique()} patients, "
          f"{len(FEATURE_COLUMNS)} features")
    print("Protocol: 5 outer folds, 3 inner folds; patient-level metrics")
    print("Held-out test features and labels: NOT USED")
    oof, folds, result = nested_logreg_cv(development)
    print(f"Overall OOF BA={result['balanced_accuracy']:.4f}, "
          f"sensitivity={result['sensitivity']:.4f}, specificity={result['specificity']:.4f}, "
          f"AUC={result['roc_auc']:.4f}, FN={result['fn']}")

    # After honest OOF evaluation, select the settings for a development-only
    # reference model with 5-fold patient CV; do not evaluate on held-out test.
    settings = choose_settings(development, seed=SEED, n_splits=NUM_OUTER_FOLDS)
    reference = fit_model(development, C=settings["C"], balance=settings["balance"])
    output_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    oof.to_csv(output_dir / "logreg_nested_cv_predictions.csv", index=False)
    folds.to_csv(output_dir / "logreg_nested_cv_folds.csv", index=False)
    result["development_reference_model_settings"] = settings
    (output_dir / "logreg_nested_cv_summary.json").write_text(
        json.dumps(result, indent=2) + "\n", encoding="utf-8")
    joblib.dump({"pipeline": reference, "features": list(FEATURE_COLUMNS),
                 "settings": settings, "status": "development_only_provisional"},
                model_dir / "logreg_development_candidate.joblib")
    print(f"Saved evaluation results: {output_dir}")
    print(f"Saved provisional development-only model: {model_dir / 'logreg_development_candidate.joblib'}")
    return result


def main():
    parser = argparse.ArgumentParser(description="Patient-level nested CV Logistic Regression baseline")
    parser.add_argument("--features", type=Path, default=ROOT / "data/processed/development_features.csv")
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--output", type=Path, default=ROOT / "results/modeling")
    parser.add_argument("--models", type=Path, default=ROOT / "models")
    args = parser.parse_args()
    run(args.features, args.manifest, args.output, args.models)


if __name__ == "__main__":
    main()
```

**`src/outlier_review.py`**

```python
"""Review the most extreme BUS-BRA development feature values, without modifying data.

Copy to src/outlier_review.py and run from repository root:
    .\\.venv\\Scripts\\python.exe -m src.outlier_review

Reads only DEVELOPMENT cases from the saved feature table. Recomputes the 23
features from source images/masks, checks agreement with cached features,
and creates a contact sheet of the highest-ranked unusual observations.
"""
from __future__ import annotations

from pathlib import Path
from zipfile import ZipFile
import math

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps

from .features import FEATURE_COLUMNS, preprocess_pair, extract_features, read_pair

ROOT = Path(__file__).resolve().parents[1]
ZIP_PATH = ROOT / 'data' / 'raw' / 'BUSBRA.zip'
FEATURES_PATH = ROOT / 'data' / 'processed' / 'development_features.csv'
FLAGS_PATH = ROOT / 'results' / 'feature_audit' / 'potential_outliers.csv'
OUT_DIR = ROOT / 'results' / 'feature_audit'
CONTACT_SHEET = OUT_DIR / 'outlier_review.png'
REVIEW_CSV = OUT_DIR / 'outlier_review.csv'
TOP_N = 12
COLUMNS = 3
CELL_W, CELL_H = 370, 322
DISPLAY_W, DISPLAY_H = 345, 241


def render_tile(image: np.ndarray, mask: np.ndarray, info: dict) -> Image.Image:
    im = Image.fromarray(image.astype(np.uint8), mode='L')
    ma = Image.fromarray(mask.astype(np.uint8) * 255, mode='L')
    view = ImageOps.contain(im, (DISPLAY_W, DISPLAY_H), method=Image.Resampling.BILINEAR)
    ma = ma.resize(view.size, Image.Resampling.NEAREST)
    inside = np.asarray(ma) > 0
    gray = np.asarray(view)
    rgb = np.repeat(gray[:, :, None], 3, axis=2).astype(np.float32)
    rgb[inside] = 0.76 * rgb[inside] + 0.24 * np.array([36, 222, 85], dtype=np.float32)
    border = inside.copy()
    border[1:-1, 1:-1] &= ~(
        inside[:-2, 1:-1] & inside[2:, 1:-1]
        & inside[1:-1, :-2] & inside[1:-1, 2:]
    )
    rgb[border] = np.array([245, 55, 48], dtype=np.float32)
    preview = Image.fromarray(rgb.clip(0, 255).astype(np.uint8), 'RGB')
    tile = Image.new('RGB', (CELL_W, CELL_H), '#FFFFFF')
    tile.paste(preview, ((CELL_W - view.width) // 2, 4))
    d = ImageDraw.Draw(tile)
    y = DISPLAY_H + 12
    d.text((8, y), f"{info['ID']}  |  {info['Pathology']}", fill='#1A1A1A')
    d.text((8, y + 18), f"Top flag: {info['top_feature']}  (z={info['max_robust_z']:.1f})", fill='#333333')
    d.text((8, y + 36), f"ASM d3: {info['asm_d3']:.3f}  |  Entropy: {info['int_entropy']:.2f}", fill='#333333')
    d.text((8, y + 54), f"Dominant gray bin: {info['top_bin_fraction']:.0%}  |  Mask area: {info['mask_area_pixels']} px", fill='#555555')
    return tile


def main() -> None:
    for p in (ZIP_PATH, FEATURES_PATH, FLAGS_PATH):
        if not p.is_file():
            raise FileNotFoundError(f'Missing expected file: {p}')
    feats = pd.read_csv(FEATURES_PATH, dtype={'ID': str})
    flags = pd.read_csv(FLAGS_PATH, dtype={'ID': str})
    if not feats['split'].eq('development').all():
        raise ValueError('Feature table contains nondevelopment images')
    if feats.ID.duplicated().any():
        raise ValueError('Duplicate IDs in feature table')
    needed = {'ID', 'feature', 'robust_z'}
    if not needed.issubset(flags.columns):
        raise ValueError(f'Outlier report missing columns: {needed - set(flags.columns)}')
    # Cover the largest outliers AND a variety of feature families, rather than
    # reviewing only ASM examples because they dominate the highest z-scores.
    ranked = flags.sort_values('robust_z', ascending=False).drop_duplicates('ID')
    selected_ids = list(ranked.head(4)['ID'])
    for _, candidate in flags.sort_values('robust_z', ascending=False).drop_duplicates('feature').iterrows():
        identifier = str(candidate['ID'])
        if identifier not in selected_ids:
            selected_ids.append(identifier)
        if len(selected_ids) == TOP_N:
            break
    for identifier in ranked['ID']:
        if len(selected_ids) == TOP_N:
            break
        if identifier not in selected_ids:
            selected_ids.append(identifier)
    chosen = flags.sort_values('robust_z', ascending=False).drop_duplicates('ID').set_index('ID').loc[selected_ids].reset_index()
    joined = chosen.merge(feats, on='ID', how='left', validate='one_to_one', suffixes=('', '_cached'))
    if joined.Pathology.isna().any():
        raise ValueError('Flag contains image ID missing from locked development data')

    rows: list[dict] = []
    tiles: list[Image.Image] = []
    with ZipFile(ZIP_PATH) as zf:
        for _, r in joined.iterrows():
            image, lesion = read_pair(zf, str(r['ID']))
            image, lesion = preprocess_pair(image, lesion)
            recalculated = extract_features(image, lesion)
            stored = np.array([float(r[name]) for name in FEATURE_COLUMNS])
            fresh = np.array([recalculated[name] for name in FEATURE_COLUMNS])
            delta = float(np.max(np.abs(stored - fresh)))
            if not np.allclose(stored, fresh, rtol=1e-6, atol=1e-8):
                raise AssertionError(f"Cached features do not reproduce for {r['ID']}: max abs error {delta}")
            gray32 = np.minimum(image[lesion].astype(np.uint16) * 32 // 256, 31)
            bins = np.bincount(gray32, minlength=32)
            info = {
                'ID': str(r['ID']),
                'Case': int(r['Case']),
                'Pathology': r['Pathology'],
                'top_feature': r['feature'],
                'max_robust_z': float(r['robust_z']),
                'asm_d3': float(recalculated['glcm_asm_d3']),
                'int_entropy': float(recalculated['int_entropy']),
                'top_bin_fraction': float(bins.max() / bins.sum()),
                'mask_area_pixels': int(lesion.sum()),
                'max_recomputed_feature_difference': delta,
            }
            rows.append(info)
            tiles.append(render_tile(image, lesion, info))

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(REVIEW_CSV, index=False, float_format='%.9g')
    nrows = math.ceil(len(tiles) / COLUMNS)
    canvas = Image.new('RGB', (COLUMNS * CELL_W, 42 + nrows * CELL_H), '#F1F3F5')
    draw = ImageDraw.Draw(canvas)
    draw.text((15, 13), 'BUS-BRA: extreme-value review (green mask, red boundary; DEV only)', fill='#111111')
    for j, tile in enumerate(tiles):
        canvas.paste(tile, ((j % COLUMNS) * CELL_W, 42 + (j // COLUMNS) * CELL_H))
    canvas.save(CONTACT_SHEET)

    print('\nBUS-BRA OUTLIER VISUAL REVIEW')
    print('=' * 44)
    print(f'Reviewed unique development images: {len(rows)}')
    print('Cached feature recomputation: MATCHED for all reviewed images')
    print('Original ZIP and extracted feature table: UNCHANGED')
    print('Highest-flagged examples:')
    print(pd.DataFrame(rows)[['ID', 'top_feature', 'max_robust_z', 'asm_d3', 'int_entropy', 'top_bin_fraction']]
          .to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    print(f'Contact sheet: {CONTACT_SHEET}')
    print(f'Review CSV:    {REVIEW_CSV}')
    print('Check each mask alignment and lesion appearance manually before closing audit.')


if __name__ == '__main__':
    main()
```

**`src/patient_calibration.py`**

```python
"""Step 20: nested, patient-level sigmoid calibration and final protocol freeze.

Base tuning/training and calibration use disjoint patients. All 23 features;
three-member calibrated ensemble per classifier; fixed probability threshold .5.
No held-out feature/image access or evaluation, and no final test results here.
"""
from __future__ import annotations

import argparse
import hashlib
import itertools
import json
import warnings
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from . import stable_evaluation as se
from . import stable_selection as ss
from . import features as feature_module
from . import coherent_features as coherent_feature_module
from . import coherent_perturbations as coherent_geometry_module
from . import perturbations as original_geometry_module
from . import models as model_module
from . import svm_model as svm_module
from . import robustness_eval as robustness_module
from .coherent_eval import load_coherent, sha256, validate_manifest_isolation
from .coherent_features import COHERENT_CONDITIONS, verify_coherent_geometry
from .features import FEATURE_COLUMNS
from .models import ROOT, inner_patient_folds, load_development
from .robustness_eval import MODEL_NAMES, _predict, metrics, summarize_pair

PROBABILITY_CLIP = 1e-6


def calibration_design():
    return {"step": 20, "version": "patient_sigmoid_ensemble_v1", "features": list(FEATURE_COLUMNS),
            "arm": "all_23_matched_tuning", "primary_model": "svm", "comparator": "logreg",
            "model_choice_basis": "posthoc Step 19 development five-condition BA; not a significance claim",
            "calibration": "three-member split ensemble; each member tuned on base patients and sigmoid-fit on separate clean patients",
            "calibration_folds": 3, "calibration_split_seed": "2020 + scope_code (outer=0..4, final=5)",
            "base_tuning_folds": 3, "base_tuning_seed": "3000 + 10 * scope_code + member_index",
            "base_tuning_objective_and_grids": "unchanged Step 18 all-23 robust objective and grids",
            "patient_score": {"logreg": "logit of mean image malignancy probability, clipped to [1e-6,1-1e-6]",
                              "svm": "mean image decision margin"},
            "calibrator": "training-calibration-patient StandardScaler then LogisticRegression(C=1,max_iter=2000,random_state=42); no class or image weights",
            "calibration_data": "one row per clean calibration patient; no coherent images used for calibration fit",
            "ensemble_prediction": "mean member calibrated patient probabilities, after patient aggregation within each member",
            "threshold": 0.5, "threshold_status": "fixed research threshold; not clinically optimized or validated",
            "method_selection_after_outer_results": "forbidden; no isotonic/threshold/model switching after calibration results",
            "test_access": "no held-out features/images/predictions before Step 21",
            "interpretation": "posthoc development analysis; calibration quality must be measured, not assumed"}


def patient_raw(model, rows, name):
    result = _predict(model, rows, name, 0)
    key = "prob_malignant" if name == "logreg" else "decision_score"
    result = result.rename(columns={key: "raw_score"}).drop(columns="outer_fold")
    result["calibration_input"] = (logit(np.clip(result.raw_score.to_numpy(float), PROBABILITY_CLIP, 1 - PROBABILITY_CLIP))
                                   if name == "logreg" else result.raw_score.to_numpy(float))
    return result


def fit_sigmoid(patient_scores):
    if (patient_scores.Case.duplicated().any() or set(patient_scores.actual) != {0, 1}
            or not np.isfinite(patient_scores.calibration_input).all()):
        raise ValueError("Invalid one-row-per-patient calibration sample")
    calibrator = Pipeline([("scaler", StandardScaler()),
                           ("sigmoid", LogisticRegression(C=1.0, max_iter=2000, random_state=42))])
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        calibrator.fit(patient_scores[["calibration_input"]].to_numpy(float), patient_scores.actual.to_numpy(int))
    return calibrator


def fit_ensemble(pool, coherent, name, *, excluded_cases, scope_code, protocol):
    """No pool patient can also belong to the outer validation/test scope."""
    pool_cases, excluded_cases = set(pool.Case), set(excluded_cases)
    if pool_cases & excluded_cases:
        raise ValueError("Leakage: ensemble pool intersects excluded patients")
    members, audit, candidates = [], [], []
    for member_index, (base, calibration) in enumerate(inner_patient_folds(pool, seed=2020 + scope_code, n_splits=3)):
        base_cases, cal_cases = set(base.Case), set(calibration.Case)
        if base_cases & cal_cases or base_cases | cal_cases != pool_cases:
            raise ValueError("Leakage or missing patient in base/calibration partition")
        contexts = []
        for inner, (train, val) in enumerate(inner_patient_folds(base, seed=3000 + 10 * scope_code + member_index, n_splits=3)):
            train_cases, val_cases = set(train.Case), set(val.Case)
            if (train_cases & val_cases or (train_cases | val_cases) != base_cases
                    or (train_cases | val_cases) & (cal_cases | excluded_cases)):
                raise ValueError("Leakage in nested base-model tuning")
            scope = {"scope_code": scope_code, "member": member_index, "inner_fold": inner,
                     "training_case_sha256": ss.case_digest(train_cases), "validation_case_sha256": ss.case_digest(val_cases)}
            contexts.append({"train": train,
                             "pack": se.validation_pack(val, coherent.loc[coherent.Case.isin(val_cases)]),
                             "ranking": None, "scope": scope})
        chosen, candidate_rows, details = se.tune_arm(contexts, name, se.ARMS[0], protocol)
        # Model selection has never observed a calibration or excluded patient's features/labels.
        model = se.fit_subset(base, name, FEATURE_COLUMNS, chosen["settings"])
        scores = patient_raw(model, calibration, name)
        calibrator = fit_sigmoid(scores)
        members.append({"base_model": model, "calibrator": calibrator, "settings": chosen["settings"]})
        audit.append({"scope_code": scope_code, "model": name, "member": member_index,
                      "base_cases": sorted(int(c) for c in base_cases), "calibration_cases": sorted(int(c) for c in cal_cases),
                      "excluded_cases": sorted(int(c) for c in excluded_cases), "chosen": chosen,
                      "inner_fits": len(details), "patient_overlap": 0})
        candidates.extend({"scope_code": scope_code, "member": member_index, "model": name, **row} for row in candidate_rows)
    return {"model": name, "members": members}, audit, candidates


def predict_ensemble(ensemble, rows):
    aligned, probabilities, raw = None, [], []
    for member in ensemble["members"]:
        score = patient_raw(member["base_model"], rows, ensemble["model"]).sort_values("Case").reset_index(drop=True)
        if aligned is None:
            aligned = score[["Case", "actual", "n_images"]].copy()
        elif not np.array_equal(aligned, score[["Case", "actual", "n_images"]]):
            raise ValueError("Ensemble patient aggregation differs across members")
        probabilities.append(member["calibrator"].predict_proba(score[["calibration_input"]].to_numpy(float))[:, 1])
        raw.append(score.raw_score.to_numpy(float))
    aligned["prob_malignant"] = np.mean(probabilities, axis=0)
    aligned["raw_score"] = np.mean(raw, axis=0)
    aligned["predicted"] = aligned.prob_malignant.ge(.5).astype(int)
    aligned["raw_predicted"] = aligned.raw_score.ge(.5 if ensemble["model"] == "logreg" else 0.0).astype(int)
    return aligned


def probability_metrics(patients):
    result = metrics(patients, "prob_malignant", "predicted")
    p, y = patients.prob_malignant.to_numpy(float), patients.actual.to_numpy(int)
    if not ((p >= 0) & (p <= 1)).all():
        raise ValueError("Calibrated probability outside [0,1]")
    return {**result, "brier_score": float(brier_score_loss(y, p)),
            "log_loss": float(log_loss(y, np.column_stack([1 - p, p]), labels=[0, 1]))}


def calibration_bins(patients, name):
    result = []
    # Fixed-width bins, including the endpoint probability one.
    bins = np.minimum((patients.prob_malignant.to_numpy(float) * 10).astype(int), 9)
    for bin_index in range(10):
        part = patients.loc[bins == bin_index]
        result.append({"model": name, "bin": bin_index, "lower": bin_index / 10, "upper": (bin_index + 1) / 10,
                       "patients": len(part), "mean_probability": float(part.prob_malignant.mean()) if len(part) else None,
                       "malignant_fraction": float(part.actual.mean()) if len(part) else None})
    return result


def calibrated_pairs(ensemble, clean_predictions, changed, fold):
    pert = predict_ensemble(ensemble, changed).set_index("Case")
    out = clean_predictions[["Case", "actual", "n_images", "prob_malignant", "predicted"]].copy()
    if set(out.Case) != set(pert.index):
        raise ValueError("Incomplete calibrated condition patient pairing")
    if not np.array_equal(out.n_images.to_numpy(), pert.loc[out.Case, "n_images"].to_numpy()):
        raise ValueError("Calibrated condition missing patient image views")
    if not np.array_equal(out.actual.to_numpy(), pert.loc[out.Case, "actual"].to_numpy()):
        raise ValueError("Calibrated condition patient diagnoses differ")
    out = out.rename(columns={"prob_malignant": "clean_score", "predicted": "clean_predicted"})
    out["perturbed_score"] = pert.loc[out.Case, "prob_malignant"].to_numpy(float)
    out["perturbed_predicted"] = pert.loc[out.Case, "predicted"].to_numpy(int)
    out["outer_fold"], out["eligible"], out["model"] = fold, True, ensemble["model"]
    out["condition"] = changed.condition.iloc[0]
    out["mean_iou"] = changed.groupby("Case").actual_iou.mean().loc[out.Case].to_numpy(float)
    return out


def nested_calibration(dev, coherent, protocol):
    clean_parts, paired_parts, audit, candidates = [], [], [], []
    for fold in range(5):
        pool, val = dev.loc[dev.dev_fold.ne(fold)], dev.loc[dev.dev_fold.eq(fold)]
        for name in MODEL_NAMES:
            ensemble, checks, search = fit_ensemble(pool, coherent, name, excluded_cases=set(val.Case), scope_code=fold, protocol=protocol)
            clean = predict_ensemble(ensemble, val)
            clean["outer_fold"], clean["model"] = fold, name
            clean_parts.append(clean)
            audit.extend(checks)
            candidates.extend(search)
            for condition in COHERENT_CONDITIONS:
                changed = coherent.loc[coherent.dev_fold.eq(fold) & coherent.condition.eq(condition)]
                paired_parts.append(calibrated_pairs(ensemble, clean, changed, fold))
            print(f"Outer {fold} | {name.upper()}: three patient-disjoint calibration members complete", flush=True)
    clean, pairs = pd.concat(clean_parts, ignore_index=True), pd.concat(paired_parts, ignore_index=True)
    if (len(clean) != dev.Case.nunique() * 2 or clean.duplicated(["model", "Case"]).any()
            or len(pairs) != dev.Case.nunique() * 8 or pairs.duplicated(["model", "condition", "Case"]).any()):
        raise RuntimeError("Incomplete calibrated outer patient coverage")
    summaries = [{"model": name, "patients": len(part), **probability_metrics(part)} for name, part in clean.groupby("model")]
    pair_summary = [{**summarize_pair(part, model=name, condition=condition)}
                    for (name, condition), part in pairs.groupby(["model", "condition"])]
    bins = [row for name, part in clean.groupby("model") for row in calibration_bins(part, name)]
    return clean, pairs, summaries, pair_summary, bins, audit, candidates


def verify_step19(directory, fingerprints, dev):
    summary = json.loads((directory / "evaluation_summary.json").read_text(encoding="utf-8"))
    if (summary.get("status") != "exploratory_nested_development_evaluation"
            or summary.get("held_out_evaluation_used") is not False
            or summary.get("original_clean_oof") != {n: "MATCHED" for n in MODEL_NAMES}
            or summary.get("source_sha256") != sha256(Path(se.__file__))):
        raise ValueError("Step 19 checkpoint/source not verified")
    for key in ("clean_features", "manifest", "coherent_features", "geometry", "protocol", "design_summary"):
        if summary.get("inputs", {}).get(key, {}).get("sha256") != fingerprints[key]["sha256"]:
            raise ValueError(f"Step 19 input hash mismatch: {key}")
    clean = pd.read_csv(directory / "clean_oof_predictions.csv")
    pairs = pd.read_csv(directory / "coherent_paired_predictions.csv")
    if (clean.duplicated(["model", "arm", "Case"]).any()
            or pairs.duplicated(["model", "arm", "condition", "Case"]).any()):
        raise ValueError("Duplicate Step 19 OOF predictions")
    records = summary.get("clean_summary", [])
    if len(records) != 4 or {(r["model"], r["arm"]) for r in records} != set(itertools.product(MODEL_NAMES, se.ARMS)):
        raise ValueError("Step 19 summary requires four arms/models")
    patient_metadata = dev.groupby("Case").agg(outer_fold=("dev_fold", "first"), n_images=("ID", "size"),
                                               Pathology=("Pathology", "first"))
    patient_metadata["actual"] = patient_metadata.Pathology.eq("malignant").astype(int)
    if len(clean) != len(dev.groupby("Case")) * 4 or len(pairs) != len(dev.groupby("Case")) * 16:
        raise ValueError("Step 19 prediction table coverage mismatch")
    for row in records:
        part = clean.loc[clean.model.eq(row["model"]) & clean.arm.eq(row["arm"])]
        if len(part) != summary["development_patients"]:
            raise ValueError("Step 19 cohort coverage mismatch")
        if set(part.Case) != set(patient_metadata.index):
            raise ValueError("Step 19 patient IDs differ from development")
        for key in ("actual", "outer_fold", "n_images"):
            if not np.array_equal(part[key], patient_metadata.loc[part.Case, key]):
                raise ValueError("Step 19 locked patient metadata mismatch")
        for key, value in metrics(part, "score", "predicted").items():
            if not np.isclose(row[key], value, rtol=1e-8, atol=1e-9):
                raise ValueError("Step 19 clean summary mismatch")
        bas = []
        for condition in COHERENT_CONDITIONS:
            p = pairs.loc[pairs.model.eq(row["model"]) & pairs.arm.eq(row["arm"]) & pairs.condition.eq(condition)]
            se.require_pair_metadata(part, p)
            ordered = p.set_index("Case").loc[part.Case]
            if (not np.allclose(part.score.to_numpy(float), ordered.clean_score.to_numpy(float), rtol=1e-6, atol=1e-7)
                    or not np.array_equal(part.predicted, ordered.clean_predicted)):
                raise ValueError("Step 19 clean/paired prediction mismatch")
            if not p.eligible.astype(str).str.lower().eq("true").all():
                raise ValueError("Step 19 coherent patient unavailable")
            bas.append(summarize_pair(p.assign(eligible=True), model=row["model"], condition=condition)["pert_balanced_accuracy"])
        if not np.isclose(row["mean_five_condition_ba"], (row["balanced_accuracy"] + sum(bas)) / 5, atol=1e-9):
            raise ValueError("Step 19 five-condition summary mismatch")
    return summary


def run(clean_path, manifest, coherent_path, geometry, design, step19, output):
    if output.exists():
        raise FileExistsError("Step 20 output exists; choose a fresh --output directory")
    paths = {"clean_features": clean_path, "manifest": manifest, "coherent_features": coherent_path, "geometry": geometry,
             "protocol": design / "selection_protocol.json", "design_summary": design / "selection_design_summary.json",
             **{f"step19_{name}": step19 / filename for name, filename in
                (("summary", "evaluation_summary.json"), ("clean", "clean_oof_predictions.csv"), ("paired", "coherent_paired_predictions.csv"))}}
    hashes = {key: {"path": str(path.resolve()), "sha256": sha256(path)} for key, path in paths.items()}
    protocol = se.load_frozen_design(design, hashes)
    dev = load_development(clean_path, manifest)
    validate_manifest_isolation(manifest, dev)
    prior = verify_step19(step19, hashes, dev)
    coherent = load_coherent(coherent_path, dev)
    verify_coherent_geometry(coherent, pd.read_csv(geometry, dtype={"ID": str}), dev)
    if prior["development_patients"] != dev.Case.nunique() or prior["development_images"] != len(dev):
        raise ValueError("Step 19 development cohort changed")
    frozen_design = calibration_design()  # Fixed before calibration CV results.
    print(f"STEP 20 — PATIENT CALIBRATION AND FINAL PROTOCOL FREEZE\n"
          f"Development: {len(dev)} images | {dev.Case.nunique()} patients\n"
          "Primary: SVM | Comparator: LR | All 23 features | Fixed probability threshold: 0.5", flush=True)
    clean, pairs, scores, pair_scores, bins, audit, candidates = nested_calibration(dev, coherent, protocol)
    final_models = {}
    manifest_ids = pd.read_csv(manifest, usecols=["ID", "Case", "split"])
    test_ids = manifest_ids.loc[manifest_ids.split.eq("test")]
    if test_ids.empty:
        raise ValueError("Locked manifest has no held-out test identifiers")
    for name in MODEL_NAMES:
        ensemble, checks, search = fit_ensemble(dev, coherent, name, excluded_cases=set(test_ids.Case), scope_code=5, protocol=protocol)
        final_models[name] = ensemble
        audit.extend(checks)
        candidates.extend(search)
        print(f"Final {name.upper()}: three development-only calibrated members fitted", flush=True)
    if any(sha256(path) != hashes[key]["sha256"] for key, path in paths.items()):
        raise RuntimeError("Locked inputs changed during Step 20")
    source_hashes = {Path(module.__file__).name: sha256(Path(module.__file__)) for module in
                     (se, ss, feature_module, coherent_feature_module, coherent_geometry_module,
                      original_geometry_module, model_module, svm_module)}
    source_hashes[Path(robustness_module.__file__).name] = sha256(Path(robustness_module.__file__))
    source_hashes[Path(__file__).name] = sha256(Path(__file__))
    output.mkdir(parents=True, exist_ok=False)
    bundle = {"version": frozen_design["version"], "design": frozen_design, "features": list(FEATURE_COLUMNS),
              "models": final_models, "development_case_sha256": ss.case_digest(set(dev.Case))}
    model_path = output / "frozen_calibrated_models.joblib"
    joblib.dump(bundle, model_path, compress=3)
    final_protocol = {"step": 20, "status": "frozen_before_held_out_evaluation", "design": frozen_design,
                      "primary_model": "svm", "comparator": "logreg", "threshold": .5,
                      "primary_endpoint": "SVM clean patient balanced accuracy at fixed probability threshold 0.5",
                      "secondary_endpoints": ["LR clean metrics and paired LR/SVM clean difference", "four coherent-condition paired robustness metrics",
                                              "clean and perturbed probability Brier/log loss", "five-condition mean BA"],
                      "held_out_conditions": ["clean", *COHERENT_CONDITIONS],
                      "held_out_patients": int(test_ids.Case.nunique()), "held_out_images": len(test_ids),
                      "held_out_case_sha256": ss.case_digest(set(test_ids.Case)),
                      "held_out_image_ids_sha256": hashlib.sha256(json.dumps(sorted(test_ids.ID.astype(str))).encode()).hexdigest(),
                      "test_once_policy": "evaluate both fixed ensembles once; no retraining, model switching, threshold tuning or feature changes after seeing test results",
                      "uncertainty_plan": "10000 paired patient bootstrap replicates within test diagnosis, seed 42; pointwise percentile 95%; fixed fitted models",
                      "missing_coherent_policy": "fail checkpoint if any test image lacks a valid coherent condition; do not silently drop or replace views",
                      "original_perturbations": "preserved development experiment; not regenerated for primary held-out evaluation",
                      "model_artifact_sha256": sha256(model_path), "inputs": hashes, "source_sha256": source_hashes,
                      "limitations": "Posthoc development-based model choice; fixed threshold is research-only, not a clinical decision rule."}
    protocol_path = output / "final_evaluation_protocol.json"
    protocol_path.write_bytes(ss.canonical_json(final_protocol).encode())
    summary = {"step": 20, "status": "calibration_complete_final_protocol_frozen", "held_out_evaluation_used": False,
               "development_images": len(dev), "development_patients": int(dev.Case.nunique()),
               "calibration_design": frozen_design, "clean_cv_metrics": scores,
               "final_protocol_sha256": sha256(protocol_path), "model_artifact_sha256": sha256(model_path),
               "base_inner_fits": sum(a["inner_fits"] for a in audit), "base_refits": len(audit), "sigmoid_fits": len(audit),
               "inputs": hashes, "versions": {pkg: version(pkg) for pkg in ("numpy", "pandas", "scikit-learn", "scipy", "joblib", "pillow", "scikit-image")},
               "limitations": "Outer CV evaluates this calibrated ensemble, a different estimator from Step 19 single refits; no assumption that calibration improves scores."}
    tables = {"calibrated_clean_oof.csv": clean, "calibrated_coherent_pairs.csv": pairs,
              "calibrated_clean_summary.csv": pd.DataFrame(scores), "calibrated_coherent_summary.csv": pd.DataFrame(pair_scores),
              "clean_reliability_bins.csv": pd.DataFrame(bins)}
    for filename, table in tables.items():
        table.to_csv(output / filename, index=False, float_format="%.12g")
    for filename, value in (("calibration_scope_audit.json", audit), ("base_candidate_search.json", candidates),
                            ("calibration_summary.json", summary), ("calibration_design.json", frozen_design)):
        (output / filename).write_text(ss.canonical_json(value), encoding="utf-8")
    print(pd.DataFrame(scores)[["model", "balanced_accuracy", "sensitivity", "specificity", "roc_auc", "brier_score", "log_loss"]].to_string(index=False))
    print(f"Final protocol: FROZEN | Held-out evaluation: NOT USED\nSaved: {output}", flush=True)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--clean", type=Path, default=ROOT / "data/processed/development_features.csv")
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--coherent", type=Path, default=ROOT / "data/processed/development_coherent_features.csv")
    parser.add_argument("--geometry", type=Path, default=ROOT / "results/robustness/coherent_geometry_development.csv")
    parser.add_argument("--design", type=Path, default=ROOT / "results/robustness/selection_step18")
    parser.add_argument("--step19", type=Path, default=ROOT / "results/robustness/selection_step19")
    parser.add_argument("--output", type=Path, default=ROOT / "results/robustness/calibration_step20")
    args = parser.parse_args()
    run(args.clean, args.manifest, args.coherent, args.geometry, args.design, args.step19, args.output)


if __name__ == "__main__":
    main()
```

**`src/perturbations.py`**

```python
"""BUS-BRA: deterministic, label-blind lesion mask perturbations.

Usage from repository root:
    .\\.venv\\Scripts\\python.exe -m src.perturbations --limit 12

This is the GEOMETRY AUDIT / smoke test only. No models, test-set data, or
labels are used to *select* perturbations. The image remains unchanged.
Eight prescribed cases: partial boundary erosion/dilation at ~.95/.90
original-mask IoU, and translations left/right/up/down at ~.90 IoU.

Partial erosion/dilation removes/adds boundary pixels in order of distance
to the existing contour, with deterministic tie-breaking within each distance
band. This avoids identical mild and moderate variants on small lesions.
These are synthetic approximations, not actual radiologist annotations.

IoU targets are approximate because masks are pixel grids. Record actual
IoU, and invalid/unavailable variants; do not silently substitute identity.
"""
from __future__ import annotations

import argparse
import hashlib
from dataclasses import dataclass
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from scipy import ndimage as ndi

from .features import PROJECT_ROOT, preprocess_pair, read_pair

# These are protocol hypotheses, not empirical severity estimates.
TARGET_IOUS = {"mild": 0.95, "moderate": 0.90}
MAX_TRANSLATION = 24
MIN_PIXELS = 15


@dataclass(frozen=True)
class Variant:
    condition: str
    mask: np.ndarray
    iou: float
    changed_pixels: int
    area_ratio: float
    amount_pixels: int


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)
    if a.ndim != 2 or a.shape != b.shape:
        raise ValueError("Masks must have same 2-D shape")
    union = np.logical_or(a, b).sum()
    if not union:
        raise ValueError("Both masks empty")
    return float(np.logical_and(a, b).sum() / union)


def partial_boundary_change(mask: np.ndarray, target_iou: float,
                            kind: str) -> np.ndarray:
    """Modify approximately 5% or 10% of area near the lesion contour.

    Unlike integer-radius disk morphology, this produces distinct mild/moderate
    alterations for small masks. Candidate pixel order uses distance-to-contour
    with deterministic tie breaks based on original mask bytes. The stricter
    perturbation is a superset of the gentler perturbation's changed pixels.
    """
    original = np.asarray(mask, dtype=bool)
    if kind not in {"erosion", "dilation"}:
        raise ValueError("Expected erosion or dilation")
    if not (0 < target_iou < 1):
        raise ValueError("target_iou must lie strictly between 0 and 1")
    area = int(original.sum())
    if area < MIN_PIXELS:
        raise ValueError("Original mask is too small")
    # IoU(A\B,A) = (area - removed)/area for removal;
    # IoU(A,A∪B) = area/(area + added) for addition.
    n_change = (round(area*(1-target_iou)) if kind == "erosion"
                else round(area*(1/target_iou-1)))
    n_change = max(1, int(n_change))
    candidates = original if kind == "erosion" else ~original
    locations = np.flatnonzero(candidates)
    if kind == "erosion" and area-n_change < MIN_PIXELS:
        raise ValueError("Erosion would make mask too small")
    if n_change > len(locations):
        raise ValueError("Insufficient surrounding pixels for dilation")
    distances = (ndi.distance_transform_edt(original) if kind == "erosion"
                 else ndi.distance_transform_edt(~original))
    seed = int.from_bytes(hashlib.blake2b(original.tobytes(), digest_size=8).digest(),
                          byteorder="little")
    noise = np.random.default_rng(seed).random(len(locations))
    # Proximity to contour is the primary sort key; reproducible pseudo-random
    # ordering breaks ties within a one-pixel boundary ring.
    sorted_locations = locations[np.lexsort((noise, distances.flat[locations]))]
    chosen = sorted_locations[:n_change]
    updated = original.copy()
    updated.flat[chosen] = (kind == "dilation")
    return updated


def translate_no_wrap(mask: np.ndarray, dy: int, dx: int) -> tuple[np.ndarray, int]:
    """Zero-padded shift, never wraps around; returns number of clipped pixels."""
    mask = np.asarray(mask, dtype=bool)
    if mask.ndim != 2:
        raise ValueError("Expected 2-D mask")
    h, w = mask.shape
    result = np.zeros_like(mask)
    if abs(dy) >= h or abs(dx) >= w:
        return result, int(mask.sum())
    sy0, sy1 = max(0, -dy), min(h, h-dy)
    sx0, sx1 = max(0, -dx), min(w, w-dx)
    if sy0 < sy1 and sx0 < sx1:
        result[sy0+dy:sy1+dy, sx0+dx:sx1+dx] = mask[sy0:sy1, sx0:sx1]
    return result, int(mask.sum() - result.sum())


def _closest(mask: np.ndarray, name: str, target: float, candidates) -> Variant | None:
    chosen = None
    for amount, candidate in candidates:
        if candidate.sum() < MIN_PIXELS:
            continue
        overlap = mask_iou(mask, candidate)
        # No identity fallback; all accepted changes must modify the boundary.
        if overlap >= 1.0:
            continue
        comparison = (abs(overlap-target), amount)
        if chosen is None or comparison < chosen[0]:
            chosen = (comparison, amount, candidate.copy(), overlap)
    if chosen is None:
        return None
    _, amount, altered, overlap = chosen
    return Variant(name, altered, overlap,
                   int(np.logical_xor(mask, altered).sum()),
                   float(altered.sum()/mask.sum()), int(amount))


def make_variants(mask: np.ndarray) -> dict[str, Variant | None]:
    """Return all eight named variants; None means impossible without clipping/empty mask.

    Approximate target IoU is optimized purely from each mask's geometry.
    No outcome labels, model scores, or held-out samples influence parameters.
    """
    original = np.asarray(mask, dtype=bool)
    if original.ndim != 2 or original.sum() < MIN_PIXELS:
        raise ValueError("Invalid or too-small original lesion mask")
    result: dict[str, Variant | None] = {}
    for name, target in TARGET_IOUS.items():
        for kind in ("erosion", "dilation"):
            condition = f"{kind}_{name}"
            altered = partial_boundary_change(original, target, kind)
            result[condition] = Variant(
                condition, altered, mask_iou(original, altered),
                int(np.logical_xor(original, altered).sum()),
                float(altered.sum()/original.sum()),
                int(np.logical_xor(original, altered).sum()),
            )
    for direction, dyunit, dxunit in (
        ("left", 0, -1), ("right", 0, 1),
        ("up", -1, 0), ("down", 1, 0)
    ):
        candidates = []
        for distance in range(1, MAX_TRANSLATION+1):
            shifted, clipped = translate_no_wrap(original, dyunit*distance, dxunit*distance)
            if clipped == 0:
                candidates.append((distance, shifted))
        result[f"shift_{direction}_moderate"] = _closest(
            original, f"shift_{direction}_moderate", TARGET_IOUS["moderate"], candidates)
    assert len(result) == 8
    return result


def audit_variants(zip_path: Path, manifest_path: Path, output_path: Path,
                   limit: int | None) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_path)
    necessary = {"ID", "Case", "Pathology", "split", "dev_fold"}
    if not necessary.issubset(manifest.columns):
        raise ValueError("Manifest missing expected split columns")
    subset = manifest.loc[manifest["split"].eq("development")].sort_values("ID")
    if limit is not None:
        if limit <= 0:
            raise ValueError("Positive --limit required")
        subset = subset.head(limit)
    if subset.empty:
        raise ValueError("No development images found")
    all_rows = []
    with ZipFile(zip_path) as zf:
        for n, record in enumerate(subset.itertuples(index=False), 1):
            image, mask = preprocess_pair(*read_pair(zf, str(record.ID)))
            # The same 'image' is intentionally reused by all variants.
            assert image.shape == mask.shape
            variants = make_variants(mask)
            for condition, variation in variants.items():
                row = {"ID": str(record.ID), "Case": record.Case,
                       "condition": condition, "valid": variation is not None}
                if variation is not None:
                    row.update({"actual_iou": variation.iou,
                                "changed_pixels": variation.changed_pixels,
                                "area_ratio": variation.area_ratio,
                                "amount_pixels": variation.amount_pixels})
                all_rows.append(row)
            if n % 100 == 0:
                print(f"Geometry-checked {n}/{len(subset)} development images...", flush=True)
    df = pd.DataFrame(all_rows)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print("BUS-BRA PERTURBATION GEOMETRY AUDIT")
    print("="*44)
    print(f"Development images: {subset.shape[0]} | Prespecified variants: 8")
    print(f"Valid variants: {int(df.valid.sum())}/{len(df)}")
    print(df.groupby("condition", sort=False).agg(
        valid=("valid", "sum"),
        median_iou=("actual_iou", "median"),
        min_iou=("actual_iou", "min"),
        max_iou=("actual_iou", "max")
    ).round(3).to_string())
    print(f"Saved: {output_path}")
    print("These are geometry checks, not classifier robustness results.")
    return df


def main() -> None:
    parser = argparse.ArgumentParser(description="BUS-BRA eight-mask geometry audit")
    parser.add_argument("--zip", type=Path, default=PROJECT_ROOT/"data/raw/BUSBRA.zip")
    parser.add_argument("--manifest", type=Path, default=PROJECT_ROOT/"splits/image_splits.csv")
    parser.add_argument("--limit", type=int, default=12)
    parser.add_argument("--all-development", action="store_true")
    args = parser.parse_args()
    limit = None if args.all_development else args.limit
    output = PROJECT_ROOT/"results/robustness"/(
        "perturbation_geometry_development.csv" if limit is None else
        f"perturbation_geometry_smoke_{limit}.csv")
    audit_variants(args.zip, args.manifest, output, limit)


if __name__ == "__main__":
    main()
```

**`src/perturbed_features.py`**

```python
"""Step 13: extract lesion-mask-perturbed BUS-BRA development features.

Commands from project root:
  python -m src.perturbed_features --limit 12
  python -m src.perturbed_features --all-development

Uses the EXACT preprocessed grayscale image for all masks of each image.
Preserves all eight intended rows per development image, including genuinely
unavailable translations (with valid=False and missing numerical features).
Verifies geometry against the frozen Step 12 audit; test split stays untouched.
No classifier is fit or evaluated here.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd

from .features import FEATURE_COLUMNS, PROJECT_ROOT, extract_features, preprocess_pair, read_pair
from .perturbations import make_variants

INFO_COLUMNS = (
    "ID", "Case", "Pathology", "split", "dev_fold", "condition",
    "valid", "failure_reason", "actual_iou", "changed_pixels", "area_ratio", "amount_pixels"
)
VARIANT_NAMES = (
    "erosion_mild", "dilation_mild", "erosion_moderate", "dilation_moderate",
    "shift_left_moderate", "shift_right_moderate",
    "shift_up_moderate", "shift_down_moderate"
)


def image_rows(image_id: str, case: int, pathology: str, dev_fold: int,
               image: np.ndarray, original_mask: np.ndarray) -> list[dict]:
    """Generate eight rows without changing the image or original mask."""
    if image.shape != original_mask.shape:
        raise ValueError("Image/mask dimensions differ")
    base_image = image.copy()
    base_mask = original_mask.copy()
    variants = make_variants(original_mask)
    if tuple(variants) != VARIANT_NAMES:
        raise AssertionError("Perturbation condition order/schema changed")
    rows = []
    for name, var in variants.items():
        row = {
            "ID": str(image_id), "Case": int(case), "Pathology": str(pathology),
            "split": "development", "dev_fold": int(dev_fold), "condition": name,
            "valid": var is not None, "failure_reason": "" if var is not None else "translation_would_clip_lesion",
            "actual_iou": float(var.iou) if var is not None else np.nan,
            "changed_pixels": int(var.changed_pixels) if var is not None else np.nan,
            "area_ratio": float(var.area_ratio) if var is not None else np.nan,
            "amount_pixels": int(var.amount_pixels) if var is not None else np.nan,
        }
        if var is None:
            row.update({f: np.nan for f in FEATURE_COLUMNS})
        else:
            if np.array_equal(var.mask, original_mask) or not np.array_equal(image, base_image):
                raise AssertionError("Mask did not change or ultrasound image was altered")
            try:
                row.update(extract_features(image, var.mask))
            except (ValueError, IndexError, ArithmeticError) as exc:
                raise RuntimeError(f"Feature extraction failed for {image_id} / {name}") from exc
        rows.append(row)
    if not np.array_equal(image, base_image) or not np.array_equal(original_mask, base_mask):
        raise AssertionError("Function mutated original ultrasound image or mask")
    return rows


def check_geometry_match(generated: pd.DataFrame, audited: pd.DataFrame) -> None:
    """Fail closed when recomputed eligibility/IoU diverges from Step 12 audit."""
    keys = ["ID", "condition"]
    columns = ["ID", "condition", "valid", "actual_iou", "changed_pixels", "area_ratio", "amount_pixels"]
    if not set(columns).issubset(audited.columns):
        raise ValueError(f"Geometry audit missing columns: {set(columns)-set(audited.columns)}")
    if generated.duplicated(keys).any() or audited.duplicated(keys).any():
        raise ValueError("Duplicate ID-condition pair")
    m = generated[columns].merge(audited[columns], on=keys, how="outer", suffixes=("_new", "_audit"), indicator=True)
    if not m._merge.eq("both").all() or len(m) != len(generated):
        raise ValueError("Geometry audit and feature rows have different IDs or conditions")
    if not (m.valid_new.astype(bool) == m.valid_audit.astype(bool)).all():
        raise ValueError("Perturbation availability disagrees with Step 12 geometry audit")
    for col in ("actual_iou", "changed_pixels", "area_ratio", "amount_pixels"):
        a = pd.to_numeric(m[f"{col}_new"], errors="coerce").to_numpy(float)
        b = pd.to_numeric(m[f"{col}_audit"], errors="coerce").to_numpy(float)
        if not np.allclose(a, b, rtol=1e-8, atol=1e-8, equal_nan=True):
            raise ValueError(f"Step 12 geometry audit mismatch: {col}")


def extract_perturbed_development(zip_path: Path, manifest_path: Path, clean_path: Path,
                                  geometry_path: Path, output_path: Path,
                                  summary_path: Path, limit: int | None = 12) -> pd.DataFrame:
    manifest = pd.read_csv(manifest_path, dtype={"ID": str})
    required = {"ID", "Case", "Pathology", "split", "dev_fold"}
    if not required.issubset(manifest):
        raise ValueError(f"Split manifest missing: {required - set(manifest)}")
    if manifest.ID.duplicated().any():
        raise ValueError("Duplicated image IDs")
    dev = manifest.loc[manifest.split.eq("development")].sort_values("ID")
    if limit is not None:
        if limit <= 0:
            raise ValueError("--limit must be positive")
        dev = dev.head(limit)
    if len(dev) == 0 or not dev.dev_fold.between(0, 4).all():
        raise ValueError("Invalid development-only subset")
    clean = pd.read_csv(clean_path, dtype={"ID": str})
    if clean["ID"].duplicated().any():
        raise ValueError("Duplicate image IDs in clean feature table")
    clean = clean.set_index("ID")
    if not set(dev.ID).issubset(clean.index):
        raise ValueError("Missing original clean features for selected development images")
    for rec in dev.itertuples(index=False):
        prev = clean.loc[rec.ID]
        if (int(prev.Case) != int(rec.Case) or str(prev.Pathology) != str(rec.Pathology)
                or str(prev.split) != "development" or int(prev.dev_fold) != int(rec.dev_fold)):
            raise ValueError(f"Clean features conflict with manifest for {rec.ID}")
    audited = pd.read_csv(geometry_path, dtype={"ID": str})
    audited = audited.loc[audited.ID.isin(set(dev.ID))].copy()
    if len(audited) != 8 * len(dev):
        raise ValueError("Geometry audit does not contain eight rows for every selected development image")
    rows = []
    with ZipFile(zip_path) as archive:
        for n, rec in enumerate(dev.itertuples(index=False), 1):
            image, mask = preprocess_pair(*read_pair(archive, str(rec.ID)))
            # Ensure identical preprocessing/extractor to previous clean-mask experiment.
            recomputed_clean = extract_features(image, mask)
            cached = clean.loc[rec.ID, list(FEATURE_COLUMNS)].to_numpy(dtype=float)
            if not np.allclose(cached, np.array(list(recomputed_clean.values())), rtol=1e-8, atol=1e-8):
                raise ValueError(f"Original clean features no longer reproduce for {rec.ID}")
            rows.extend(image_rows(rec.ID, rec.Case, rec.Pathology, rec.dev_fold, image, mask))
            if n % 100 == 0:
                print(f"Extracted perturbed features for {n}/{len(dev)} development images...", flush=True)
    result = pd.DataFrame(rows, columns=list(INFO_COLUMNS) + list(FEATURE_COLUMNS))
    if len(result) != 8*len(dev) or result.ID.nunique() != len(dev):
        raise AssertionError("Incomplete perturbation output")
    valid = result.valid.astype(bool)
    if not result.split.eq("development").all():
        raise AssertionError("Test image leaked into perturbation extraction")
    if not np.isfinite(result.loc[valid, list(FEATURE_COLUMNS)].to_numpy(dtype=float)).all():
        raise AssertionError("Nonfinite valid feature measurement")
    if not result.loc[~valid, list(FEATURE_COLUMNS)].isna().all().all():
        raise AssertionError("Unavailable masks must not be represented by invented features")
    check_geometry_match(result, audited)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, float_format="%.12g")
    summary = {
        "development_images": int(len(dev)), "unique_patients": int(dev.Case.nunique()),
        "conditions": len(VARIANT_NAMES), "intended_rows": int(len(result)),
        "valid_rows": int(valid.sum()), "unavailable_rows": int((~valid).sum()),
        "numerical_features": len(FEATURE_COLUMNS), "test_images_seen": 0,
        "geometry_audit_matched": True,
        "by_condition": {name: {"valid": int(g.valid.sum()), "unavailable": int((~g.valid).sum()),
                                "iou_min": (None if g.actual_iou.isna().all() else float(g.actual_iou.min())),
                                "iou_median": (None if g.actual_iou.isna().all() else float(g.actual_iou.median()))}
                         for name, g in result.groupby("condition", sort=False)}
    }
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print("BUS-BRA PERTURBED FEATURE EXTRACTION")
    print("="*44)
    print(f"Development images: {len(dev)} | Patients: {dev.Case.nunique()} | Features: {len(FEATURE_COLUMNS)}")
    print(f"Available features: {int(valid.sum())}/{len(result)} variants")
    print(f"Unavailable variants: {int((~valid).sum())} (explicitly flagged; no substitute clean masks)")
    print("Original clean features: MATCHED")
    print("Step 12 geometry report: MATCHED")
    print("Test set: NOT USED")
    print(f"Saved feature table: {output_path}")
    print(f"Saved summary: {summary_path}")
    return result


def main():
    p = argparse.ArgumentParser(description="Extract unchanged-image, changed-mask BUS-BRA features")
    p.add_argument("--zip", type=Path, default=PROJECT_ROOT / "data/raw/BUSBRA.zip")
    p.add_argument("--manifest", type=Path, default=PROJECT_ROOT / "splits/image_splits.csv")
    p.add_argument("--clean", type=Path, default=PROJECT_ROOT / "data/processed/development_features.csv")
    p.add_argument("--geometry", type=Path, default=None)
    p.add_argument("--output", type=Path, default=None)
    p.add_argument("--summary", type=Path, default=None)
    p.add_argument("--limit", type=int, default=12)
    p.add_argument("--all-development", action="store_true")
    args = p.parse_args()
    if args.all_development and args.limit != 12:
        p.error("Do not combine --all-development with --limit")
    limit = None if args.all_development else args.limit
    base = PROJECT_ROOT / "results/robustness"
    geometry = args.geometry or base / ("perturbation_geometry_development.csv" if limit is None else f"perturbation_geometry_smoke_{limit}.csv")
    output = args.output or PROJECT_ROOT / "data/processed" / ("development_perturbed_features.csv" if limit is None else f"perturbed_features_smoke_{limit}.csv")
    summary = args.summary or base / ("perturbed_feature_summary.json" if limit is None else f"perturbed_feature_smoke_{limit}_summary.json")
    for path in (args.zip, args.manifest, args.clean, geometry):
        if not path.is_file():
            raise FileNotFoundError(path)
    extract_perturbed_development(args.zip, args.manifest, args.clean, geometry, output, summary, limit)


if __name__ == "__main__":
    main()
```

**`src/robustness_diagnostics.py`**

```python
"""Post-hoc development-only diagnostic for the surprising Step 14 severity pattern.

Run from the project root:
    python -m src.robustness_diagnostics

No retraining and NO test images. Reads development feature tables and saved
patient predictions. Saves feature-shift, patient flip-pattern and contour
inspection artifacts; it does not modify the perturbation protocol or models.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from zipfile import ZipFile

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .features import FEATURE_COLUMNS, PROJECT_ROOT, preprocess_pair, read_pair, extract_features
from .perturbations import make_variants
from .perturbed_features import VARIANT_NAMES

COMPARE = (("erosion_mild", "erosion_moderate"), ("dilation_mild", "dilation_moderate"))
REVIEW = ("clean", "erosion_mild", "erosion_moderate", "dilation_mild", "dilation_moderate")


def parse_flags(column: pd.Series) -> pd.Series:
    """Strict CSV bool parser: bool('False') incorrectly evaluates to True."""
    values = column.astype('string').str.strip().str.lower()
    if not values.isin(['true', 'false']).all():
        raise ValueError('Expected only True/False availability values')
    return values.eq('true').astype(bool)


def check_tables(clean: pd.DataFrame, perturbed: pd.DataFrame) -> pd.DataFrame:
    """Join valid perturbed rows to original, development-only features."""
    fields = {'ID', 'Case', 'split', 'dev_fold'} | set(FEATURE_COLUMNS)
    if not fields.issubset(clean.columns) or not (fields | {'condition', 'valid'}).issubset(perturbed.columns):
        raise ValueError('Missing required columns in feature table')
    if clean.ID.duplicated().any() or perturbed.duplicated(['ID', 'condition']).any():
        raise ValueError('Duplicate image or image-condition key')
    if not clean.split.eq('development').all() or not perturbed.split.eq('development').all():
        raise ValueError('This analysis must use development data only')
    if not set(perturbed.ID).issubset(set(clean.ID)):
        raise ValueError('Perturbed image not present in clean development table')
    valid = parse_flags(perturbed.valid)
    part = perturbed.loc[valid, ['ID', 'Case', 'dev_fold', 'condition', *FEATURE_COLUMNS]].copy()
    orig = clean[['ID', 'Case', 'dev_fold', *FEATURE_COLUMNS]]
    merged = part.merge(orig, on='ID', how='inner', suffixes=('', '_clean'), validate='many_to_one')
    if len(merged) != len(part) or not (merged['Case'] == merged['Case_clean']).all() or not (merged['dev_fold'] == merged['dev_fold_clean']).all():
        raise ValueError('Missing images or patient/fold metadata mismatch')
    return merged


def feature_drift(clean: pd.DataFrame, paired: pd.DataFrame) -> pd.DataFrame:
    """Descriptive drift standardized by the ORIGINAL development feature std.

    This is NOT preprocessing for any classifier, nor a fitted model.
    """
    std = clean[list(FEATURE_COLUMNS)].std(ddof=0).replace(0, np.nan)
    if std.isna().any():
        raise ValueError('Constant feature prevents drift standardization')
    records = []
    for condition, rows in paired.groupby('condition', sort=False):
        for feature in FEATURE_COLUMNS:
            delta = rows[feature].to_numpy(float) - rows[feature+'_clean'].to_numpy(float)
            if not np.isfinite(delta).all():
                raise ValueError('Non-finite feature drift')
            records.append({
                'condition': condition, 'feature': feature, 'valid_images': len(rows),
                'median_signed_delta': float(np.median(delta)),
                'median_abs_delta': float(np.median(np.abs(delta))),
                'median_abs_delta_dev_sd': float(np.median(np.abs(delta)) / std[feature]),
                'mean_abs_delta_dev_sd': float(np.mean(np.abs(delta)) / std[feature]),
                'p90_abs_delta_dev_sd': float(np.percentile(np.abs(delta), 90) / std[feature]),
            })
    return pd.DataFrame(records)


def flip_discordance(preds: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    fields = {'model', 'condition', 'Case', 'actual', 'eligible', 'clean_predicted', 'perturbed_predicted'}
    if not fields.issubset(preds.columns):
        raise ValueError('Missing model prediction columns')
    if preds.duplicated(['model', 'condition', 'Case']).any():
        raise ValueError('Duplicate model-condition-patient record')
    preds = preds.copy()
    preds['eligible'] = parse_flags(preds.eligible)
    summaries, patient_rows = [], []
    for model, data in preds.groupby('model', sort=True):
        for mild, moderate in COMPARE:
            a = data.loc[data.condition.eq(mild)].copy()
            b = data.loc[data.condition.eq(moderate)].copy()
            p = a.merge(b, on='Case', how='inner', validate='one_to_one', suffixes=('_mild', '_moderate'))
            if len(p) != len(a) or len(p) != len(b):
                raise ValueError('Mild/moderate patient coverage differs')
            if not (p.actual_mild.eq(p.actual_moderate) & p.clean_predicted_mild.eq(p.clean_predicted_moderate)).all():
                raise ValueError('Mild/moderate rows have inconsistent clean reference')
            p = p.loc[p.eligible_mild & p.eligible_moderate].copy()
            if not len(p):
                raise ValueError('No patients eligible for both severities')
            mild_flip = p.perturbed_predicted_mild.ne(p.clean_predicted_mild)
            moderate_flip = p.perturbed_predicted_moderate.ne(p.clean_predicted_moderate)
            mild_only = mild_flip & ~moderate_flip
            moderate_only = moderate_flip & ~mild_flip
            summaries.append({
                'model': model, 'family': mild.removesuffix('_mild'),
                'eligible_both': int(len(p)),
                'mild_flip_count': int(mild_flip.sum()),
                'moderate_flip_count': int(moderate_flip.sum()),
                'mild_only_flips': int(mild_only.sum()),
                'moderate_only_flips': int(moderate_only.sum()),
                'both_flip': int((mild_flip & moderate_flip).sum()),
                'neither_flip': int((~mild_flip & ~moderate_flip).sum()),
                'mild_only_malignant': int((mild_only & p.actual_mild.eq(1)).sum()),
            })
            for ix, row in p.loc[mild_only | moderate_only].iterrows():
                patient_rows.append({
                    'model': model, 'family': mild.removesuffix('_mild'),
                    'Case': int(row.Case), 'actual': int(row.actual_mild),
                    'mild_only_flip': bool(mild_only.loc[ix]),
                    'moderate_only_flip': bool(moderate_only.loc[ix]),
                    'clean_predicted': int(row.clean_predicted_mild),
                    'mild_predicted': int(row.perturbed_predicted_mild),
                    'moderate_predicted': int(row.perturbed_predicted_moderate),
                })
    return pd.DataFrame(summaries), pd.DataFrame(patient_rows)


def create_gallery(zip_path: Path, paired: pd.DataFrame, output: Path) -> list[str]:
    """Compare identical grayscale images with five original/changed masks."""
    small = paired.loc[paired.condition.isin(['erosion_mild', 'erosion_moderate', 'dilation_mild', 'dilation_moderate']),
                       ['ID', 'condition', 'shape_circularity', 'shape_circularity_clean']].copy()
    small['shift'] = (small.shape_circularity - small.shape_circularity_clean).abs()
    tab = small.pivot_table(index='ID', columns='condition', values='shift', aggfunc='first')
    required = ['erosion_mild', 'erosion_moderate', 'dilation_mild', 'dilation_moderate']
    tab = tab.dropna(subset=required)
    # Larger positive scores mean mild changes alter circularity more than moderate changes.
    score = ((tab.erosion_mild - tab.erosion_moderate).clip(lower=0)
             + (tab.dilation_mild - tab.dilation_moderate).clip(lower=0))
    chosen = list(score.sort_values(ascending=False).head(4).index)
    if not chosen:
        raise ValueError('No complete erosion/dilation examples for contour gallery')
    fig, axes = plt.subplots(len(chosen), len(REVIEW), figsize=(14, 2.8 * len(chosen)), squeeze=False)
    with ZipFile(zip_path) as archive:
        for i, image_id in enumerate(chosen):
            image, mask = preprocess_pair(*read_pair(archive, image_id))
            variants = make_variants(mask)
            for j, name in enumerate(REVIEW):
                ax = axes[i, j]
                selected = mask if name == 'clean' else variants[name].mask
                ax.imshow(image, cmap='gray', vmin=0, vmax=255)
                ax.contour(selected.astype(float), levels=[0.5], colors=['#f97316'], linewidths=1.0)
                if j:
                    ax.contour(mask.astype(float), levels=[0.5], colors=['#34d399'], linewidths=.65, linestyles='dotted')
                circ = extract_features(image, selected)['shape_circularity']
                iou = '1.000' if name == 'clean' else f'{variants[name].iou:.3f}'
                ax.set_title(f'{image_id}\n{name}\nIoU={iou}, circ={circ:.3f}', fontsize=8)
                ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle('Orange = evaluated mask; green dotted = original mask (development only)', fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=165, bbox_inches='tight')
    plt.close(fig)
    return chosen


def run(project_root: Path, *, show_gallery=True, pert_path: Path | None = None,
        predictions_path: Path | None = None) -> dict:
    root = Path(project_root)
    clean_path = root/'data/processed/development_features.csv'
    pert_path = pert_path or root/'data/processed/development_perturbed_features.csv'
    predictions_path = predictions_path or root/'results/robustness/robustness_paired_patient_predictions.csv'
    output_dir = root/'results/robustness'
    for fp in (clean_path, pert_path, predictions_path):
        if not fp.is_file():
            raise FileNotFoundError(fp)
    clean = pd.read_csv(clean_path, dtype={'ID': str})
    pert = pd.read_csv(pert_path, dtype={'ID': str}, low_memory=False)
    preds = pd.read_csv(predictions_path)
    if (len(pert) != len(clean) * len(VARIANT_NAMES)
            or pert.ID.nunique() != clean.ID.nunique()
            or set(pert.condition) != set(VARIANT_NAMES)
            or not pert.groupby('ID').condition.nunique().eq(len(VARIANT_NAMES)).all()):
        raise ValueError('Expected all eight perturbation records for every development image')
    paired = check_tables(clean, pert)
    drift = feature_drift(clean, paired)
    discord, patients = flip_discordance(preds)
    output_dir.mkdir(parents=True, exist_ok=True)
    drift.to_csv(output_dir/'diagnostics_feature_shift_by_condition.csv', index=False, float_format='%.10g')
    discord.to_csv(output_dir/'diagnostics_mild_moderate_flip_patterns.csv', index=False)
    patients.to_csv(output_dir/'diagnostics_severity_discordant_patients.csv', index=False)
    gallery_ids=[]
    if show_gallery:
        gallery_ids=create_gallery(root/'data/raw/BUSBRA.zip', paired, output_dir/'diagnostics_boundary_gallery.png')
    summary={
        'purpose':'Post-hoc diagnosis of non-monotonic mild/moderate mask robustness results',
        'development_images':int(clean.ID.nunique()),
        'perturbation_rows':int(len(pert)),
        'valid_perturbation_rows':int(len(paired)),
        'test_images_used':0, 'model_retraining':False,
        'gallery_image_ids':gallery_ids,
        'comparison':[x for x in discord.to_dict(orient='records')],
    }
    (output_dir/'diagnostics_summary.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
    print('BUS-BRA STEP 14.1 — SEVERITY / FEATURE DRIFT AUDIT')
    print('='*56)
    print(f"Development: {len(clean)} images | Valid perturbation rows: {len(paired)} | Test patients used: 0")
    print('\nCIRCULARITY feature shift (original feature units):')
    circ=drift.loc[drift.feature.eq('shape_circularity') & drift.condition.isin(
        ['erosion_mild','erosion_moderate','dilation_mild','dilation_moderate'])]
    print(circ[['condition','median_signed_delta','median_abs_delta','median_abs_delta_dev_sd']].to_string(index=False,float_format=lambda n:f'{n:.4f}'))
    print('\nPREDICTION FLIP OVERLAP (same eligible patients for both severities):')
    print(discord[['model','family','eligible_both','mild_flip_count','moderate_flip_count','mild_only_flips','moderate_only_flips','mild_only_malignant']].to_string(index=False))
    print('\nSaved diagnostic CSVs and summary in:',output_dir)
    if show_gallery:
        print('Mask comparison gallery:',output_dir/'diagnostics_boundary_gallery.png')
    print('Important: descriptive post-hoc diagnostic; do not change experimental protocol based on this alone.')
    return summary


def main():
    p=argparse.ArgumentParser(description='Audit severity paradox in development-only robustness results')
    p.add_argument('--project-root',type=Path,default=PROJECT_ROOT)
    p.add_argument('--no-gallery',action='store_true')
    args=p.parse_args()
    run(args.project_root,show_gallery=not args.no_gallery)


if __name__=='__main__':
    main()
```

**`src/robustness_eval.py`**

```python
"""Step 14: held-out-outer-fold robustness against synthetic lesion-mask changes.

From repository root:
    python -m src.robustness_eval

Reuses LOCKED inner-selected hyperparameters recorded by Steps 8 and 9.
Refits each outer training fold using *clean* data only, reproduces the
saved clean OOF scores, then applies those fitted models to unchanged-image,
perturbed-mask features of the SAME outer validation patients.

For a perturbation condition, a patient is eligible ONLY when every one of
that patient's images has a valid perturbed mask. Ineligible patients are
retained in the prediction output with no perturbed score; paired metrics are
computed on the matching eligible patients. No held-out test data are read.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score

from .features import FEATURE_COLUMNS
from .models import ROOT, NUM_OUTER_FOLDS, fit_model, load_development, patient_predictions
from .perturbed_features import INFO_COLUMNS, VARIANT_NAMES
from .svm_model import fit_svm, patient_svm_predictions

ID_COLUMNS = ("ID", "Case", "Pathology", "split", "dev_fold")
MODEL_NAMES = ("logreg", "svm")


def parse_boolean_column(series: pd.Series) -> pd.Series:
    """Parse CSV bools explicitly: astype(bool) would make 'False' truthy."""
    values = series.astype("string").str.strip().str.lower()
    if not values.isin(["true", "false"]).all():
        raise ValueError("Expected only True/False in perturbed-feature 'valid' column")
    return values.eq("true").astype(bool)


def load_perturbations(path: Path, clean: pd.DataFrame) -> pd.DataFrame:
    """Require one valid/unavailable row for EVERY development ID and condition."""
    pert = pd.read_csv(path, dtype={"ID": str}, low_memory=False)
    expected = list(INFO_COLUMNS) + list(FEATURE_COLUMNS)
    if list(pert.columns) != expected:
        raise ValueError("Perturbed feature table schema/order differs from Step 13")
    if len(pert) != len(clean) * len(VARIANT_NAMES):
        raise ValueError("Missing/extra perturbation condition records")
    if pert[list(ID_COLUMNS) + ["condition", "valid"]].isna().any().any():
        raise ValueError("Missing perturbation ID, label, condition, or availability")
    if pert.duplicated(["ID", "condition"]).any():
        raise ValueError("Duplicate ID-condition pair in perturbed features")
    if set(pert["condition"]) != set(VARIANT_NAMES):
        raise ValueError("Unexpected/missing perturbation condition")
    if not pert.groupby("ID")["condition"].nunique().eq(len(VARIANT_NAMES)).all():
        raise ValueError("An image does not have all eight perturbations")
    if not pert["split"].eq("development").all():
        raise ValueError("Non-development rows in perturbed feature table")
    if set(pert["ID"]) != set(clean["ID"]):
        raise ValueError("Perturbation images differ from clean development images")

    # Metadata must agree with the frozen split, not just be plausible.
    metadata = clean[list(ID_COLUMNS)].set_index("ID")
    compare = pert[list(ID_COLUMNS)].set_index("ID")
    for column in ID_COLUMNS[1:]:
        expected_values = metadata.loc[compare.index, column].to_numpy()
        if not np.array_equal(compare[column].to_numpy(), expected_values):
            raise ValueError(f"Perturbation metadata differs from clean manifest: {column}")

    pert["valid"] = parse_boolean_column(pert["valid"])
    cols = list(FEATURE_COLUMNS)
    matrix = pert[cols].to_numpy(dtype=float)
    v = pert["valid"].to_numpy(dtype=bool)
    if not np.isfinite(matrix[v]).all():
        raise ValueError("Valid perturbation has missing/nonfinite features")
    if not np.isnan(matrix[~v]).all():
        raise ValueError("Unavailable perturbation contains fabricated numeric features")
    if pert.loc[v, "actual_iou"].isna().any() or not pert.loc[~v, "actual_iou"].isna().all():
        raise ValueError("IoU missing for valid variant or present for unavailable variant")
    if not pert.loc[v, "actual_iou"].between(0, 1).all():
        raise ValueError("Invalid perturbation IoU")
    if len(pert) != len(clean) * len(VARIANT_NAMES):
        raise AssertionError("Perturbation coverage changed unexpectedly")
    return pert


def load_fold_settings(path: Path, model_name: str) -> dict[int, dict]:
    fold_table = pd.read_csv(path)
    if ("outer_fold" not in fold_table or "C" not in fold_table
            or "balance" not in fold_table):
        raise ValueError(f"Missing recorded {model_name} hyperparameters")
    if fold_table["outer_fold"].duplicated().any() or set(fold_table.outer_fold) != set(range(NUM_OUTER_FOLDS)):
        raise ValueError(f"Expected one {model_name} setting per outer fold")
    if model_name == "svm" and "gamma" not in fold_table:
        raise ValueError("Recorded SVM settings missing gamma")
    parsed = {}
    for r in fold_table.to_dict("records"):
        fold = int(r["outer_fold"])
        if not np.isfinite(float(r["C"])) or float(r["C"]) <= 0:
            raise ValueError("Invalid regularization C")
        b = str(r["balance"]).strip().lower()
        if b not in ("true", "false"):
            raise ValueError("Invalid class-balance setting")
        setting = {"C": float(r["C"]), "balance": b == "true"}
        if model_name == "svm":
            gamma = str(r["gamma"]).strip()
            if gamma != "scale":
                gamma = float(gamma)
                if not np.isfinite(gamma) or gamma <= 0:
                    raise ValueError("Invalid SVM gamma")
            setting["gamma"] = gamma
        parsed[fold] = setting
    return parsed


def validate_clean_reference(recomputed: pd.DataFrame, reference_path: Path,
                             model_name: str) -> None:
    """Fail closed if refitting does not reproduce saved Step 8/9 clean OOF."""
    previous = pd.read_csv(reference_path)
    if (len(previous) != len(recomputed) or previous.Case.duplicated().any()
            or set(previous.Case) != set(recomputed.Case)):
        raise ValueError(f"Saved {model_name} OOF patient coverage differs")
    left = recomputed.sort_values("Case").reset_index(drop=True)
    right = previous.sort_values("Case").reset_index(drop=True)
    for column in ("Case", "actual", "predicted", "outer_fold", "n_images"):
        if column not in right or not np.array_equal(left[column], right[column]):
            raise ValueError(f"Saved {model_name} OOF mismatch: {column}")
    score_key = "prob_malignant" if model_name == "logreg" else "decision_score"
    if score_key not in right or not np.allclose(
        left[score_key].to_numpy(float), right[score_key].to_numpy(float), rtol=1e-6, atol=1e-7
    ):
        raise ValueError(f"Saved {model_name} OOF score mismatch; do not continue with robustness")


def eligible_patient_ids(validation: pd.DataFrame, variants: pd.DataFrame) -> set:
    """Exclude the whole patient if even one of their images is unavailable."""
    if set(validation.ID) != set(variants.ID):
        raise ValueError("Validation image coverage differs between clean and perturbed")
    missing_ids = set(variants.loc[~variants.valid, "ID"])
    missing_cases = set(validation.loc[validation.ID.isin(missing_ids), "Case"])
    return set(validation.Case) - missing_cases


def metrics(df: pd.DataFrame, score_col: str, pred_col: str) -> dict:
    if df.empty or not set(df["actual"]) == {0, 1}:
        raise ValueError("Paired metric sample requires both diagnoses")
    y = df["actual"].to_numpy(dtype=int)
    p = df[pred_col].to_numpy(dtype=int)
    s = df[score_col].to_numpy(dtype=float)
    if not np.isfinite(s).all():
        raise ValueError("Nonfinite patient score")
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y, p, labels=[0, 1]).ravel())
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return {"balanced_accuracy": float((sensitivity + specificity) / 2),
            "sensitivity": float(sensitivity), "specificity": float(specificity),
            "roc_auc": float(roc_auc_score(y, s)),
            "tn": tn, "fp": fp, "fn": fn, "tp": tp}


def _predict(model, frames: pd.DataFrame, model_name: str, fold: int) -> pd.DataFrame:
    return (patient_predictions(model, frames, fold=fold) if model_name == "logreg"
            else patient_svm_predictions(model, frames, outer_fold=fold))


def compare_one_condition(model, model_name: str, validation: pd.DataFrame,
                          pert_one: pd.DataFrame, fold: int) -> pd.DataFrame:
    """One row per held-out patient. Missing variants yield NA, never fallback."""
    baseline = _predict(model, validation, model_name, fold)
    score_key = "prob_malignant" if model_name == "logreg" else "decision_score"
    eligible_cases = eligible_patient_ids(validation, pert_one)
    out = baseline[["Case", "actual", "outer_fold", "n_images", score_key, "predicted"]].copy()
    out = out.rename(columns={score_key: "clean_score", "predicted": "clean_predicted"})
    out["eligible"] = out.Case.isin(eligible_cases)
    out["perturbed_score"] = np.nan
    out["perturbed_predicted"] = np.nan
    out["mean_iou"] = np.nan
    if eligible_cases:
        imgs = pert_one.loc[pert_one.Case.isin(eligible_cases)].copy()
        if not imgs.valid.all():
            raise AssertionError("Unavailable image would be evaluated")
        pert_pred = _predict(model, imgs, model_name, fold)
        elig_clean = validation.loc[validation.Case.isin(eligible_cases)]
        clean_counts = elig_clean.groupby("Case").size().sort_index()
        pert_counts = imgs.groupby("Case").size().sort_index()
        if not clean_counts.equals(pert_counts):
            raise AssertionError("Eligible patients must have all original image views")
        pert_pred = pert_pred.set_index("Case")
        ious = imgs.groupby("Case")["actual_iou"].mean()
        ix = out["eligible"]
        cases = out.loc[ix, "Case"]
        out.loc[ix, "perturbed_score"] = pert_pred.loc[cases, score_key].to_numpy(float)
        out.loc[ix, "perturbed_predicted"] = pert_pred.loc[cases, "predicted"].to_numpy(int)
        out.loc[ix, "mean_iou"] = ious.loc[cases].to_numpy(float)
    out["condition"] = pert_one.condition.iloc[0]
    out["model"] = model_name
    out["missing_reason"] = np.where(out["eligible"], "", "at_least_one_image_mask_unavailable")
    return out[["model", "condition", "Case", "actual", "outer_fold", "n_images", "eligible",
                "missing_reason", "mean_iou", "clean_score", "perturbed_score",
                "clean_predicted", "perturbed_predicted"]]


def summarize_pair(preds: pd.DataFrame, *, model: str, condition: str, fold=None) -> dict:
    n_total = len(preds)
    if preds.Case.duplicated().any():
        raise ValueError("Repeated patient in paired predictions")
    paired = preds.loc[preds.eligible].copy()
    if paired.empty:
        raise ValueError("No eligible validation patients")
    if paired[["clean_score", "perturbed_score", "clean_predicted", "perturbed_predicted"]].isna().any().any():
        raise ValueError("Missing score within eligible pair")
    clean_m = metrics(paired, "clean_score", "clean_predicted")
    pert_m = metrics(paired, "perturbed_score", "perturbed_predicted")
    flip = paired.clean_predicted.to_numpy(int) != paired.perturbed_predicted.to_numpy(int)
    malignant = paired.actual.eq(1).to_numpy()
    clean_pred = paired.clean_predicted.to_numpy(int)
    pert_pred = paired.perturbed_predicted.to_numpy(int)
    output = {
        "model": model, "condition": condition,
        "eligible_patients": len(paired), "excluded_patients": n_total - len(paired),
        "eligible_images": int(paired.n_images.sum()),
        "mean_patient_iou": float(paired.mean_iou.mean()),
        "clean_balanced_accuracy": clean_m["balanced_accuracy"],
        "pert_balanced_accuracy": pert_m["balanced_accuracy"],
        "delta_balanced_accuracy": pert_m["balanced_accuracy"] - clean_m["balanced_accuracy"],
        "clean_sensitivity": clean_m["sensitivity"],
        "pert_sensitivity": pert_m["sensitivity"],
        "delta_sensitivity": pert_m["sensitivity"] - clean_m["sensitivity"],
        "clean_specificity": clean_m["specificity"],
        "pert_specificity": pert_m["specificity"],
        "delta_specificity": pert_m["specificity"] - clean_m["specificity"],
        "clean_roc_auc": clean_m["roc_auc"],
        "pert_roc_auc": pert_m["roc_auc"],
        "delta_roc_auc": pert_m["roc_auc"] - clean_m["roc_auc"],
        "flip_count": int(flip.sum()),
        "flip_rate": float(flip.mean()),
        "new_false_negatives": int((malignant & (clean_pred == 1) & (pert_pred == 0)).sum()),
        "recovered_false_negatives": int((malignant & (clean_pred == 0) & (pert_pred == 1)).sum()),
        "clean_false_negatives": clean_m["fn"],
        "pert_false_negatives": pert_m["fn"],
        "mean_abs_score_change": float(np.mean(np.abs(paired.perturbed_score - paired.clean_score))),
    }
    if fold is not None:
        output["outer_fold"] = int(fold)
    return output


def run(clean_path: Path, manifest_path: Path, perturbed_path: Path,
        results_dir: Path, output_dir: Path) -> dict:
    dev = load_development(clean_path, manifest_path)
    pert = load_perturbations(perturbed_path, dev)
    print("BUS-BRA — PAIRED BOUNDARY ROBUSTNESS (DEVELOPMENT ONLY)")
    print("=" * 56)
    print(f"Development: {len(dev)} images | {dev.Case.nunique()} patients | {len(FEATURE_COLUMNS)} features")
    print(f"Perturbations: {len(pert)} records; {int(pert.valid.sum())} valid; {(~pert.valid).sum()} unavailable")
    print("Five locked outer folds; original inner-chosen hyperparameters; no retraining on perturbed masks")
    print("Unavailable variants: exclude affected patient from condition-specific paired metrics")
    print("Test patients: NOT USED")

    all_preds = []
    fold_stats = []
    reference_checks = {}
    training_seconds = {}
    start = perf_counter()
    for model_name in MODEL_NAMES:
        selected = load_fold_settings(results_dir / f"{model_name}_nested_cv_folds.csv", model_name)
        clean_parts = []
        model_out = []
        model_fit_seconds = 0.0
        for fold in range(NUM_OUTER_FOLDS):
            train = dev.loc[dev.dev_fold.ne(fold)]
            validation = dev.loc[dev.dev_fold.eq(fold)]
            if set(train.Case) & set(validation.Case):
                raise AssertionError("Leakage: training/validation share patient")
            setup = selected[fold]
            t = perf_counter()
            if model_name == "logreg":
                model = fit_model(train, C=setup["C"], balance=setup["balance"])
            else:
                model = fit_svm(train, C=setup["C"], gamma=setup["gamma"], balance=setup["balance"])
            model_fit_seconds += perf_counter() - t
            baseline = _predict(model, validation, model_name, fold)
            clean_parts.append(baseline)
            for condition in VARIANT_NAMES:
                variants = pert.loc[pert.condition.eq(condition) & pert.dev_fold.eq(fold)]
                rec = compare_one_condition(model, model_name, validation, variants, fold)
                fold_stats.append(summarize_pair(rec, model=model_name, condition=condition, fold=fold))
                model_out.append(rec)
            print(f"{model_name.upper()} outer fold {fold}: {len(baseline)} held-out patients; 8 conditions", flush=True)
        recomputed = pd.concat(clean_parts, ignore_index=True)
        if recomputed.Case.duplicated().any() or len(recomputed) != dev.Case.nunique():
            raise RuntimeError("Clean OOF missing or repeated patient")
        validate_clean_reference(recomputed, results_dir / f"{model_name}_nested_cv_predictions.csv", model_name)
        reference_checks[model_name] = "MATCHED"
        training_seconds[model_name] = round(model_fit_seconds, 3)
        all_preds.extend(model_out)
        print(f"{model_name.upper()} saved clean OOF predictions: MATCHED", flush=True)
    predictions = pd.concat(all_preds, ignore_index=True)
    expected = dev.Case.nunique() * len(VARIANT_NAMES) * len(MODEL_NAMES)
    if len(predictions) != expected or predictions.duplicated(["model", "condition", "Case"]).any():
        raise AssertionError("Missing/duplicate model-condition-patient records")
    summary_rows = []
    for model_name in MODEL_NAMES:
        for condition in VARIANT_NAMES:
            subset = predictions.loc[predictions.model.eq(model_name) & predictions.condition.eq(condition)]
            summary_rows.append(summarize_pair(subset, model=model_name, condition=condition))
    summary = pd.DataFrame(summary_rows)
    folds = pd.DataFrame(fold_stats)
    output_dir.mkdir(parents=True, exist_ok=True)
    predictions = predictions.sort_values(["model", "condition", "Case"])
    predictions.to_csv(output_dir / "robustness_paired_patient_predictions.csv", index=False, float_format="%.12g")
    summary.to_csv(output_dir / "robustness_paired_condition_summary.csv", index=False, float_format="%.12g")
    folds.to_csv(output_dir / "robustness_outer_fold_metrics.csv", index=False, float_format="%.12g")
    result = {
        "protocol": "Refit fold-specific clean models with recorded inner-CV parameters; compare on matched held-out patients",
        "development_patients": int(dev.Case.nunique()), "development_images": int(len(dev)),
        "conditions": list(VARIANT_NAMES), "model_names": list(MODEL_NAMES),
        "perturbation_records": int(len(pert)), "valid_variations": int(pert.valid.sum()),
        "unavailable_variations": int((~pert.valid).sum()),
        "patient_rows_saved": len(predictions), "test_patients_seen": 0,
        "missing_condition_policy": "Exclude entire affected patient from that condition's paired metrics; retain NA record",
        "clean_reference_oof": reference_checks,
        "fold_training_seconds_not_comparable_benchmark": training_seconds,
        "elapsed_total_seconds": round(perf_counter() - start, 3),
        "result_rows": summary.to_dict("records"),
    }
    (output_dir / "robustness_paired_summary.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print("\nMODEL | CONDITION              | ELIGIBLE | DELTA BA | FLIP % | NEW FN")
    print("-" * 75)
    for r in summary.itertuples():
        print(f"{r.model:6} | {r.condition:22} | {r.eligible_patients:8} | {r.delta_balanced_accuracy:+.4f} | "
              f"{100*r.flip_rate:5.2f}% | {r.new_false_negatives:6}")
    print(f"Saved: {output_dir}")
    print("Both clean reference OOF tables: MATCHED | Held-out test: NOT USED")
    return result


def main() -> None:
    p = argparse.ArgumentParser(description="Patient-matched held-out-fold boundary robustness evaluation")
    p.add_argument("--clean", type=Path, default=ROOT / "data/processed/development_features.csv")
    p.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    p.add_argument("--perturbed", type=Path, default=ROOT / "data/processed/development_perturbed_features.csv")
    p.add_argument("--modeling", type=Path, default=ROOT / "results/modeling")
    p.add_argument("--output", type=Path, default=ROOT / "results/robustness")
    args = p.parse_args()
    for path in (args.clean, args.manifest, args.perturbed,
                 args.modeling / "logreg_nested_cv_folds.csv",
                 args.modeling / "logreg_nested_cv_predictions.csv",
                 args.modeling / "svm_nested_cv_folds.csv",
                 args.modeling / "svm_nested_cv_predictions.csv"):
        if not path.is_file():
            raise FileNotFoundError(path)
    run(args.clean, args.manifest, args.perturbed, args.modeling, args.output)


if __name__ == "__main__":
    main()
```

**`src/robustness_uncertainty.py`**

```python
"""Step 17: exploratory, paired patient bootstrap of saved development OOF results.

Intervals condition on the existing OOF fits and observed fold/diagnosis counts.
They do NOT estimate retraining variability or provide confirmatory CV inference.
No fitting, feature selection, threshold changes or held-out evaluation occurs.
"""
from __future__ import annotations

import argparse
import json
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd

from .coherent_eval import PROTOCOL, ORIGINAL_PROTOCOL, require_same_clean, sha256
from .coherent_features import COHERENT_CONDITIONS
from .models import ROOT
from .robustness_eval import MODEL_NAMES, parse_boolean_column, summarize_pair, validate_clean_reference

METRICS = ("delta_balanced_accuracy", "delta_sensitivity", "delta_specificity",
           "flip_rate", "new_false_negative_rate_among_malignant")
LIMITATION = ("Exploratory pointwise percentile intervals conditional on saved OOF models and "
              "observed fold/diagnosis counts. No retraining variance, simultaneous coverage, "
              "multiplicity correction or confirmatory significance claim.")


def load_patients(path: Path) -> pd.DataFrame:
    manifest = pd.read_csv(path, usecols=["ID", "Case", "Pathology", "split", "dev_fold"])
    if manifest[["ID", "Case", "split"]].isna().any().any() or manifest.ID.duplicated().any():
        raise ValueError("Invalid manifest identifiers")
    dev = manifest.loc[manifest.split.eq("development")].copy()
    if dev.empty or dev[["Pathology", "dev_fold"]].isna().any().any():
        raise ValueError("Missing development manifest metadata")
    if set(dev.Case) & set(manifest.loc[manifest.split.ne("development"), "Case"]):
        raise ValueError("Leakage: patient shared with held-out split")
    if (not dev.Pathology.isin(["benign", "malignant"]).all()
            or not dev.dev_fold.isin(range(5)).all()
            or set(dev.dev_fold) != set(range(5))):
        raise ValueError("Invalid development diagnosis/fold")
    if (dev.groupby("Case")[["Pathology", "dev_fold"]].nunique() != 1).any().any():
        raise ValueError("Patient crosses folds or diagnoses")
    patients = dev.groupby("Case", as_index=False).agg(
        Pathology=("Pathology", "first"), outer_fold=("dev_fold", "first"), n_images=("ID", "size"))
    patients["actual"] = patients.Pathology.eq("malignant").astype(int)
    return patients.sort_values("Case").reset_index(drop=True)


def load_predictions(path: Path, patients: pd.DataFrame, *, coherent: bool) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {"model", "condition", "Case", "actual", "outer_fold", "n_images", "eligible",
                "clean_score", "perturbed_score", "clean_predicted", "perturbed_predicted", "mean_iou"}
    if not required.issubset(frame):
        raise ValueError("Missing prediction columns")
    if coherent:
        if ("protocol" not in frame or not frame.protocol.eq(PROTOCOL).all()
                or set(frame.condition) != set(COHERENT_CONDITIONS)):
            raise ValueError("Unexpected coherent protocol/conditions")
    frame = frame.loc[frame.condition.isin(COHERENT_CONDITIONS)].copy()
    if (frame[list(required - {"perturbed_score", "perturbed_predicted", "mean_iou"})].isna().any().any()
            or set(frame.model) != set(MODEL_NAMES)
            or frame.duplicated(["model", "condition", "Case"]).any()):
        raise ValueError("Missing metadata or duplicate model-condition-patient")
    frame["eligible"] = parse_boolean_column(frame.eligible)
    if coherent and not frame.eligible.all():
        raise ValueError("Completed Step 16 must have all coherent patients eligible")
    indexed = patients.set_index("Case")
    for name in MODEL_NAMES:
        for condition in COHERENT_CONDITIONS:
            part = frame.loc[frame.model.eq(name) & frame.condition.eq(condition)]
            if len(part) != len(patients) or set(part.Case) != set(patients.Case):
                raise ValueError("Prediction patient coverage differs from locked development")
            for col in ("actual", "outer_fold", "n_images"):
                if not np.array_equal(part[col], indexed.loc[part.Case, col]):
                    raise ValueError(f"Prediction metadata mismatch: {col}")
            if not np.isfinite(part.clean_score).all() or not part.clean_predicted.isin([0, 1]).all():
                raise ValueError("Invalid clean prediction")
            eligible = part.loc[part.eligible]
            if (not np.isfinite(eligible[["perturbed_score", "mean_iou"]]).all().all()
                    or not eligible.perturbed_predicted.isin([0, 1]).all()
                    or not eligible.mean_iou.between(0, 1).all()):
                raise ValueError("Invalid eligible prediction")
            if part.loc[~part.eligible, ["perturbed_score", "perturbed_predicted"]].notna().any().any():
                raise ValueError("Ineligible patient has fabricated perturbed prediction")
            for prefix in ("clean", "perturbed"):
                sample = part if prefix == "clean" else eligible
                score = sample[f"{prefix}_score"].to_numpy(float)
                if name == "logreg" and not ((score >= 0) & (score <= 1)).all():
                    raise ValueError("LR score outside probability range")
                expected = (score >= (0.5 if name == "logreg" else 0.0)).astype(int)
                if not np.array_equal(sample[f"{prefix}_predicted"], expected):
                    raise ValueError("Prediction disagrees with locked threshold")
    return frame


def validate_references(original: pd.DataFrame, coherent: pd.DataFrame, modeling: Path,
                        step16_summary: dict, hashes: dict, patients: pd.DataFrame) -> None:
    if (step16_summary.get("protocol") != PROTOCOL
            or step16_summary.get("clean_reference_oof") != {n: "MATCHED" for n in MODEL_NAMES}
            or step16_summary.get("step14_clean_predictions") != "MATCHED"
            or step16_summary.get("held_out_features_images_predictions_used") is not False
            or step16_summary.get("development_patients") != len(patients)
            or step16_summary.get("development_images") != int(patients.n_images.sum())):
        raise ValueError("Step 16 checkpoint not verified or cohort changed")
    for key in ("manifest", "step14_predictions", "logreg_predictions", "svm_predictions"):
        if step16_summary.get("inputs", {}).get(key, {}).get("sha256") != hashes[key]["sha256"]:
            raise ValueError(f"Step 16 recorded input hash mismatch: {key}")
    rows = step16_summary.get("results", [])
    if len(rows) != 8:
        raise ValueError("Step 16 summary must contain eight model-condition rows")
    saved = pd.DataFrame(rows)
    if saved.duplicated(["model", "condition"]).any():
        raise ValueError("Duplicate Step 16 summary row")
    for name in MODEL_NAMES:
        for condition in COHERENT_CONDITIONS:
            old = original.loc[original.model.eq(name) & original.condition.eq(condition)]
            new = coherent.loc[coherent.model.eq(name) & coherent.condition.eq(condition)]
            require_same_clean(old, new)
            baseline = new[["Case", "actual", "outer_fold", "n_images", "clean_predicted", "clean_score"]].rename(
                columns={"clean_predicted": "predicted", "clean_score": "prob_malignant" if name == "logreg" else "decision_score"})
            validate_clean_reference(baseline, modeling / f"{name}_nested_cv_predictions.csv", name)
            recalculated = summarize_pair(new, model=name, condition=condition)
            match = saved.loc[saved.model.eq(name) & saved.condition.eq(condition)]
            if len(match) != 1:
                raise ValueError("Missing Step 16 summary row")
            for key, value in recalculated.items():
                if isinstance(value, (int, float)):
                    if key not in match or not np.isclose(float(match.iloc[0][key]), value, rtol=1e-8, atol=1e-9):
                        raise ValueError(f"Step 16 saved summary mismatch: {key}")


def patient_values(part: pd.DataFrame) -> np.ndarray:
    """Per-patient contributions whose means equal the five target metrics."""
    y = part.actual.to_numpy(int)
    clean = part.clean_predicted.to_numpy(int)
    pert = part.perturbed_predicted.to_numpy(int)
    if set(y) != {0, 1}:
        raise ValueError("Common patient population requires both diagnoses")
    n = len(y)
    malignant, benign = y.sum(), (1 - y).sum()
    delta_correct = (pert == y).astype(float) - (clean == y).astype(float)
    return np.column_stack((delta_correct * n / (2 * np.where(y == 1, malignant, benign)),
                            y * (pert - clean) * n / malignant,
                            (1 - y) * (clean - pert) * n / benign,
                            (pert != clean).astype(float),
                            y * (clean == 1) * (pert == 0) * n / malignant))


def paired_draws(values: np.ndarray, labels: np.ndarray, folds: np.ndarray,
                 *, n_resamples: int, seed: int) -> np.ndarray:
    """Same patient resampling indices for every model/protocol/metric column."""
    if (values.ndim != 2 or len(values) != len(labels) or len(labels) != len(folds)
            or not np.isfinite(values).all() or set(labels) != {0, 1}
            or not np.isin(folds, range(5)).all() or n_resamples < 2 or seed < 0):
        raise ValueError("Invalid paired bootstrap inputs")
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero((folds == fold) & (labels == label)) for fold in range(5) for label in (0, 1)]
    if any(len(group) == 0 for group in groups):
        raise ValueError("Common patient cohort needs both diagnoses in every locked fold")
    draws = np.empty((n_resamples, values.shape[1]), dtype=float)
    for start in range(0, n_resamples, 256):
        stop = min(start + 256, n_resamples)
        batch = np.zeros((stop - start, values.shape[1]))
        for group in groups:
            ix = group[rng.integers(0, len(group), size=(stop - start, len(group)))]
            batch += values[ix].sum(axis=1)
        draws[start:stop] = batch / len(values)
    return draws


def analyze_condition(original: pd.DataFrame, coherent: pd.DataFrame, condition: str,
                      *, n_resamples: int, seed: int) -> tuple[list[dict], list[dict]]:
    parts = {(protocol, name): frame.loc[frame.condition.eq(condition) & frame.model.eq(name)].copy()
             for protocol, frame in ((ORIGINAL_PROTOCOL, original), (PROTOCOL, coherent)) for name in MODEL_NAMES}
    common = set.intersection(*(set(part.loc[part.eligible, "Case"]) for part in parts.values()))
    if not common:
        raise ValueError("No common eligible patients")
    aligned = {key: part.loc[part.Case.isin(common)].sort_values("Case").reset_index(drop=True)
               for key, part in parts.items()}
    first = next(iter(aligned.values()))
    for part in aligned.values():
        if not np.array_equal(part[["Case", "actual", "outer_fold", "n_images"]],
                              first[["Case", "actual", "outer_fold", "n_images"]]):
            raise ValueError("Unpaired patient metadata")
    keys = list(aligned)
    values = np.column_stack([patient_values(aligned[key]) for key in keys])
    draws = paired_draws(values, first.actual.to_numpy(int), first.outer_fold.to_numpy(int),
                         n_resamples=n_resamples, seed=seed)
    point = values.mean(axis=0)
    def vector(protocol, name):
        weights = np.zeros(len(keys))
        weights[keys.index((protocol, name))] = 1
        return weights
    contrasts = []
    for name in MODEL_NAMES:
        contrasts.append(("coherent_change", name, PROTOCOL, vector(PROTOCOL, name)))
        contrasts.append(("protocol_difference_coherent_minus_original", name, "both",
                          vector(PROTOCOL, name) - vector(ORIGINAL_PROTOCOL, name)))
    for protocol in (ORIGINAL_PROTOCOL, PROTOCOL):
        contrasts.append(("model_difference_svm_minus_logreg", "both", protocol,
                          vector(protocol, "svm") - vector(protocol, "logreg")))
    contrasts.append(("change_in_svm_advantage_coherent_minus_original", "both", "both",
                      vector(PROTOCOL, "svm") - vector(PROTOCOL, "logreg")
                      - vector(ORIGINAL_PROTOCOL, "svm") + vector(ORIGINAL_PROTOCOL, "logreg")))
    rows = []
    for kind, name, protocol, weights in contrasts:
        for metric_ix, metric in enumerate(METRICS):
            distribution = draws[:, metric_ix::len(METRICS)] @ weights
            low, high = np.quantile(distribution, [0.025, 0.975], method="linear")
            rows.append({"comparison": kind, "model": name, "protocol": protocol,
                         "condition": condition, "metric": metric,
                         "estimate": float(point[metric_ix::len(METRICS)] @ weights),
                         "ci_low": float(low), "ci_high": float(high),
                         "common_patients": len(common), "malignant_patients": int(first.actual.sum()),
                         "excluded_from_common": len(next(iter(parts.values()))) - len(common),
                         "n_resamples": n_resamples, "condition_seed": seed,
                         "interval_scope": "exploratory_pointwise_conditional_on_saved_OOF"})
    coverage = [{"condition": condition, "outer_fold": fold, "actual": label,
                 "common_patients": int(((first.outer_fold == fold) & (first.actual == label)).sum())}
                for fold in range(5) for label in (0, 1)]
    return rows, coverage


def run(manifest: Path, original_path: Path, coherent_dir: Path, modeling: Path,
        output: Path, *, n_resamples: int = 10000, seed: int = 42) -> dict:
    if n_resamples < 200 or seed < 0:
        raise ValueError("Use at least 200 resamples and a nonnegative seed")
    # New directory only; no pre-existing checkpoint can be overwritten.
    if output.exists():
        raise FileExistsError("Step 17 output already exists; choose a new --output directory")
    paths = {"manifest": manifest, "step14_predictions": original_path,
             "step16_predictions": coherent_dir / "coherent_paired_patient_predictions.csv",
             "step16_summary": coherent_dir / "coherent_summary.json",
             **{f"{name}_predictions": modeling / f"{name}_nested_cv_predictions.csv" for name in MODEL_NAMES}}
    hashes = {key: {"path": str(path.resolve()), "sha256": sha256(path)} for key, path in paths.items()}
    patients = load_patients(manifest)
    original = load_predictions(original_path, patients, coherent=False)
    coherent = load_predictions(paths["step16_predictions"], patients, coherent=True)
    step16 = json.loads(paths["step16_summary"].read_text(encoding="utf-8"))
    validate_references(original, coherent, modeling, step16, hashes, patients)
    rows, coverage = [], []
    print(f"STEP 17 — PAIRED PATIENT UNCERTAINTY\nDevelopment patients: {len(patients)} | Resamples: {n_resamples}", flush=True)
    for offset, condition in enumerate(COHERENT_CONDITIONS):
        rec, counts = analyze_condition(original, coherent, condition, n_resamples=n_resamples, seed=seed + offset)
        rows.extend(rec)
        coverage.extend(counts)
        print(f"{condition}: {rec[0]['common_patients']} common patients; paired draws complete", flush=True)
    intervals = pd.DataFrame(rows)
    if any(sha256(path) != hashes[key]["sha256"] for key, path in paths.items()):
        raise RuntimeError("Saved inputs changed during Step 17")
    result = {"step": 17, "status": "exploratory_posthoc_uncertainty",
              "method": "paired patient bootstrap stratified by locked outer fold and diagnosis; percentile 95%",
              "n_resamples": n_resamples, "seed": seed,
              "condition_seeds": {c: seed + i for i, c in enumerate(COHERENT_CONDITIONS)},
              "development_patients": len(patients), "interval_rows": len(intervals),
              "population": "per-condition intersection across both models and original/coherent protocols",
              "limitations": LIMITATION, "refitting_performed": False, "held_out_evaluation_used": False,
              "clean_references": "MATCHED to nested OOF and Step 14",
              "step16_summary_recomputed": "MATCHED", "inputs": hashes,
              "source_sha256": sha256(Path(__file__)),
              "versions": {pkg: version(pkg) for pkg in ("numpy", "pandas", "scikit-learn")}}
    output.mkdir(parents=True, exist_ok=False)
    intervals.to_csv(output / "paired_bootstrap_intervals.csv", index=False, float_format="%.12g")
    pd.DataFrame(coverage).to_csv(output / "common_patient_coverage.csv", index=False)
    (output / "uncertainty_summary.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    main = intervals.loc[intervals.comparison.eq("model_difference_svm_minus_logreg")
                         & intervals.protocol.eq(PROTOCOL) & intervals.metric.eq("delta_balanced_accuracy")]
    print(main[["condition", "estimate", "ci_low", "ci_high"]].to_string(index=False))
    print(f"{LIMITATION}\nClean references: MATCHED | Held-out evaluation: NOT USED\nSaved: {output}")
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--original", type=Path, default=ROOT / "results/robustness/robustness_paired_patient_predictions.csv")
    parser.add_argument("--coherent", type=Path, default=ROOT / "results/robustness/coherent_step16_run2")
    parser.add_argument("--modeling", type=Path, default=ROOT / "results/modeling")
    parser.add_argument("--output", type=Path, default=ROOT / "results/robustness/uncertainty_step17")
    parser.add_argument("--resamples", type=int, default=10000)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    run(args.manifest, args.original, args.coherent, args.modeling, args.output,
        n_resamples=args.resamples, seed=args.seed)


if __name__ == "__main__":
    main()
```

**`src/splits.py`**

```python
"""Create leakage-safe, deterministic patient-level BUS-BRA splits.

Place at src/splits.py and run from the project root:
    .\\.venv\\Scripts\\python.exe -m src.splits

Reads data/raw/BUSBRA.zip directly; writes CSV/JSON under splits/.
The split files are small and SHOULD be committed to Git for reproducibility.
This script does not extract or modify BUS-BRA images or masks.
"""

from __future__ import annotations

import argparse
import hashlib
import io
import json
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SEED = 42
TEST_FRACTION = 0.20
CV_FOLDS = 5


def safe_write_csv(data: pd.DataFrame, path: Path) -> None:
    """Prevent accidental changes to previously locked split assignments."""
    if path.exists():
        existing = pd.read_csv(path)
        try:
            pd.testing.assert_frame_equal(
                existing.reset_index(drop=True),
                data.reset_index(drop=True),
                check_dtype=False,
            )
        except AssertionError as exc:
            raise RuntimeError(
                f"Locked split file differs from today's output: {path}. "
                "Do not overwrite it without investigating the cause."
            ) from exc
        print(f"Verified existing locked file: {path}")
    else:
        data.to_csv(path, index=False)
        print(f"Created: {path}")


def create_splits(zip_path: Path, output_dir: Path) -> dict:
    if not zip_path.is_file():
        raise FileNotFoundError(f"Dataset not found: {zip_path}")

    with ZipFile(zip_path) as zf:
        csv_bytes = zf.read("BUSBRA/bus_data.csv")
    metadata = pd.read_csv(io.BytesIO(csv_bytes))
    required = {"Case", "ID", "Pathology"}
    if not required.issubset(metadata.columns):
        raise ValueError(f"Missing columns: {sorted(required - set(metadata.columns))}")
    if metadata[list(required)].isna().any().any():
        raise ValueError("Metadata contains missing Case, ID or Pathology values")
    if metadata["ID"].duplicated().any():
        raise ValueError("Duplicate image IDs; investigate before splitting")

    metadata = metadata.copy()
    metadata["Pathology"] = metadata["Pathology"].str.strip().str.lower()
    if not metadata["Pathology"].isin({"benign", "malignant"}).all():
        raise ValueError("Unexpected pathology value")
    if (metadata.groupby("Case")["Pathology"].nunique() != 1).any():
        raise ValueError("One patient has inconsistent pathology labels")

    # Each patient appears exactly once in this table.
    patients = (
        metadata.groupby("Case", as_index=False)
        .agg(Pathology=("Pathology", "first"), n_images=("ID", "size"))
        .sort_values("Case")
        .reset_index(drop=True)
    )
    if len(patients) != 1064 or len(metadata) != 1875:
        raise ValueError("Unexpected BUS-BRA sample counts; verify dataset version")

    development, test = train_test_split(
        patients,
        test_size=TEST_FRACTION,
        stratify=patients["Pathology"],
        shuffle=True,
        random_state=SEED,
    )
    patients["split"] = "test"
    patients["dev_fold"] = -1
    patients.loc[development.index, "split"] = "development"

    # Fold assignment is performed on patients, never on image rows.
    dev_for_cv = patients.loc[patients["split"] == "development"].copy()
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    for fold, (_train_idx, valid_idx) in enumerate(
        cv.split(dev_for_cv[["Case"]], dev_for_cv["Pathology"])
    ):
        patient_row_indices = dev_for_cv.iloc[valid_idx].index
        patients.loc[patient_row_indices, "dev_fold"] = fold

    assert len(patients) == patients["Case"].nunique()
    assert set(patients.loc[patients["split"] == "development", "Case"]).isdisjoint(
        set(patients.loc[patients["split"] == "test", "Case"])
    )
    assert (patients.loc[patients["split"] == "test", "dev_fold"] == -1).all()
    assert set(patients.loc[patients["split"] == "development", "dev_fold"]) == set(range(CV_FOLDS))

    # Image manifest is for convenience; it inherits patient assignments exactly.
    images = metadata[["ID", "Case", "Pathology"]].merge(
        patients[["Case", "split", "dev_fold"]],
        on="Case", how="left", validate="many_to_one"
    ).sort_values(["Case", "ID"]).reset_index(drop=True)
    assert len(images) == len(metadata)
    assert (images.groupby("Case")["split"].nunique() == 1).all()
    assert (images.groupby("Case")["dev_fold"].nunique() == 1).all()
    assert not images[["split", "dev_fold"]].isna().any().any()

    output_dir.mkdir(parents=True, exist_ok=True)
    patients = patients[["Case", "Pathology", "n_images", "split", "dev_fold"]]
    safe_write_csv(patients, output_dir / "patient_splits.csv")
    safe_write_csv(images, output_dir / "image_splits.csv")

    def counts(df: pd.DataFrame) -> dict:
        return {
            "patients": int(len(df)),
            "images": int(df["n_images"].sum()),
            "benign_patients": int((df["Pathology"] == "benign").sum()),
            "malignant_patients": int((df["Pathology"] == "malignant").sum()),
        }

    summary = {
        "random_seed": SEED,
        "test_fraction_requested": TEST_FRACTION,
        "cv_folds": CV_FOLDS,
        "dataset_csv_sha256": hashlib.sha256(csv_bytes).hexdigest(),
        "all": counts(patients),
        "development": counts(patients[patients["split"] == "development"]),
        "test": counts(patients[patients["split"] == "test"]),
        "development_validation_folds": {
            str(f): counts(patients[(patients["split"] == "development") & (patients["dev_fold"] == f)])
            for f in range(CV_FOLDS)
        },
    }
    summary_path = output_dir / "split_summary.json"
    serialized = json.dumps(summary, indent=2, sort_keys=True) + "\n"
    if summary_path.exists() and summary_path.read_text(encoding="utf-8") != serialized:
        raise RuntimeError(f"Existing split summary differs: {summary_path}")
    summary_path.write_text(serialized, encoding="utf-8")

    print("\nBUS-BRA PATIENT SPLIT VERIFICATION")
    print("=" * 45)
    for label in ("all", "development", "test"):
        c = summary[label]
        print(f"{label.title():12} {c['patients']:4} patients | {c['images']:4} images | "
              f"benign {c['benign_patients']:3} | malignant {c['malignant_patients']:3}")
    for fold, c in summary["development_validation_folds"].items():
        print(f"Dev fold {fold}: {c['patients']} validation patients "
              f"(benign {c['benign_patients']}, malignant {c['malignant_patients']})")
    print("Patient leakage: 0")
    print("Fold overlap: 0")
    print(f"Split files: {output_dir}")
    return summary


def main() -> None:
    parser = argparse.ArgumentParser(description="Create reproducible patient-level BUS-BRA splits")
    parser.add_argument("--zip", type=Path, default=PROJECT_ROOT / "data/raw/BUSBRA.zip")
    parser.add_argument("--output", type=Path, default=PROJECT_ROOT / "splits")
    args = parser.parse_args()
    create_splits(args.zip, args.output)


if __name__ == "__main__":
    main()
```

**`src/stability.py`**

```python

```

**`src/stable_evaluation.py`**

```python
"""Step 19: nested patient CV for the frozen Step 18 stability-selection design.

Refits clean models only. Coherent INNER-validation predictions score candidates;
OUTER-validation patients never affect ranking, scaling, fitting or selection.
"""
from __future__ import annotations

import argparse
import itertools
import json
import warnings
from dataclasses import dataclass
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score

from . import stable_selection as ss
from .coherent_eval import load_coherent, sha256, validate_manifest_isolation
from .coherent_features import COHERENT_CONDITIONS, verify_coherent_geometry
from .features import FEATURE_COLUMNS
from .models import ROOT, inner_patient_folds, load_development, make_model, patient_image_weights
from .robustness_eval import (MODEL_NAMES, _predict, compare_one_condition, load_fold_settings,
                             metrics, summarize_pair, validate_clean_reference)
from .svm_model import build_svm

ARMS = ("all_23_matched_tuning", "stability_topk")


@dataclass
class SubsetModel:
    pipeline: object
    columns: tuple

    def matrix(self, full_matrix):
        matrix = np.asarray(full_matrix, dtype=float)
        if matrix.ndim != 2 or matrix.shape[1] != len(FEATURE_COLUMNS):
            raise ValueError("Subset adapter expects the full canonical 23-column input")
        return matrix[:, [FEATURE_COLUMNS.index(c) for c in self.columns]]

    def predict_proba(self, full_matrix):
        return self.pipeline.predict_proba(self.matrix(full_matrix))

    def decision_function(self, full_matrix):
        return self.pipeline.decision_function(self.matrix(full_matrix))


def fit_subset(train, name, columns, settings):
    columns = tuple(columns)
    if (name not in MODEL_NAMES or not columns or len(set(columns)) != len(columns)
            or columns != tuple(c for c in FEATURE_COLUMNS if c in columns)):
        raise ValueError("Invalid classifier or noncanonical subset columns")
    if (not train.split.eq("development").all() or train.ID.duplicated().any()
            or set(train.Pathology) != {"benign", "malignant"}):
        raise ValueError("Invalid clean fitting partition")
    matrix = train[list(columns)].to_numpy(float)
    if not np.isfinite(matrix).all():
        raise ValueError("Nonfinite clean training matrix")
    weights = patient_image_weights(train, balance=settings["balance"])
    pipeline = (make_model(settings["C"]) if name == "logreg"
                else build_svm(settings["C"], settings["gamma"]))
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        pipeline.fit(matrix, train.Pathology.eq("malignant").to_numpy(int),
                     scaler__sample_weight=weights, classifier__sample_weight=weights)
    return SubsetModel(pipeline, columns)


def validation_pack(clean, coherent):
    """Stack five conditions for one model call while preserving patient means."""
    frames = [clean]
    for condition in COHERENT_CONDITIONS:
        part = coherent.loc[coherent.condition.eq(condition)]
        if (len(part) != len(clean) or set(part.ID) != set(clean.ID)
                or not part.valid.all() or part.ID.duplicated().any()):
            raise ValueError("Incomplete inner-validation pairing")
        part = part.set_index("ID").loc[clean.ID].reset_index()
        for col in ("Case", "Pathology", "dev_fold"):
            if not np.array_equal(part[col], clean[col]):
                raise ValueError("Inner-validation metadata mismatch")
        frames.append(part)
    return {
        "X": np.concatenate([f[list(FEATURE_COLUMNS)].to_numpy(float) for f in frames]),
        "Case": np.tile(clean.Case.to_numpy(), 5),
        "actual": np.tile(clean.Pathology.eq("malignant").to_numpy(int), 5),
        "condition_index": np.repeat(np.arange(5), len(clean)),
    }


def evaluate_inner(model, name, pack):
    scores = (model.predict_proba(pack["X"])[:, 1] if name == "logreg"
              else model.decision_function(pack["X"]))
    if not np.isfinite(scores).all():
        raise ValueError("Nonfinite inner-validation score")
    frame = pd.DataFrame({k: pack[k] for k in ("Case", "actual", "condition_index")})
    frame["score"] = scores
    if frame.groupby("Case").actual.nunique().ne(1).any():
        raise ValueError("Inconsistent validation patient diagnoses")
    patient = frame.groupby(["condition_index", "Case"], as_index=False).agg(
        actual=("actual", "first"), score=("score", "mean"))
    patient["predicted"] = patient.score.ge(0.5 if name == "logreg" else 0.0).astype(int)
    bas = []
    for condition in range(5):
        p = patient.loc[patient.condition_index.eq(condition)]
        y, pred = p.actual.to_numpy(int), p.predicted.to_numpy(int)
        if set(y) != {0, 1}:
            raise ValueError("Inner validation needs both patient diagnoses")
        bas.append(float(0.5 * ((pred[y == 1] == 1).mean() + (pred[y == 0] == 0).mean())))
    clean = patient.loc[patient.condition_index.eq(0)]
    return {"objective": float(np.mean(bas)), "clean_ba": bas[0],
            "clean_auc": float(roc_auc_score(clean.actual, clean.score)),
            **{f"ba_{condition}": bas[i + 1] for i, condition in enumerate(COHERENT_CONDITIONS)}}


def build_contexts(dev, coherent, outer):
    outer_train = dev.loc[dev.dev_fold.ne(outer)]
    outer_val = set(dev.loc[dev.dev_fold.eq(outer), "Case"])
    contexts, audit, rankings = [], [], []
    for inner, (train, val) in enumerate(inner_patient_folds(outer_train, seed=42 + outer)):
        forbidden = outer_val | set(val.Case)
        cases = set(train.Case)
        if cases & forbidden or cases | forbidden != set(dev.Case):
            raise ValueError("Leakage or missing patient in inner scope")
        ranking = ss.rank_training_features(dev, coherent, training_cases=cases, forbidden_cases=forbidden)
        scope = {"outer_fold": outer, "inner_fold": inner, "context": "inner_training",
                 "training_case_sha256": ss.case_digest(cases), "excluded_case_sha256": ss.case_digest(forbidden),
                 "training_patients": len(cases), "excluded_patients": len(forbidden)}
        contexts.append({"train": train, "pack": validation_pack(val, coherent.loc[coherent.Case.isin(set(val.Case))]),
                         "ranking": ranking, "scope": scope})
        audit.append(scope)
        rankings.append(ranking.assign(outer_fold=outer, inner_fold=inner, context="inner_training"))
    return contexts, audit, rankings


def setting_grid(name, protocol):
    grid = protocol[name]
    for C, balance in itertools.product(grid["C"], grid["balance"]):
        for gamma_index, gamma in enumerate(grid["gamma"] if name == "svm" else [None]):
            setting = {"C": C, "balance": balance}
            if name == "svm":
                setting["gamma"] = gamma
            yield setting, gamma_index


def candidate_key(row):
    return (row["mean_objective"], row["mean_clean_ba"], row["mean_clean_auc"],
            -row["k"], -row["settings"]["C"], -int(row["settings"]["balance"]), -row["gamma_index"])


def tune_arm(contexts, name, arm, protocol):
    ks = [23] if arm == ARMS[0] else protocol["top_k_candidates"]
    candidates, details = [], []
    for k in ks:
        if arm == ARMS[1] and any(int(ctx["ranking"].eligible.sum()) < k for ctx in contexts):
            continue
        columns = [FEATURE_COLUMNS if arm == ARMS[0] else ss.selected_columns(ctx["ranking"], k) for ctx in contexts]
        for settings, gamma_index in setting_grid(name, protocol):
            scores = []
            for inner, ctx in enumerate(contexts):
                start = perf_counter()
                model = fit_subset(ctx["train"], name, columns[inner], settings)
                fit_seconds = perf_counter() - start
                score = evaluate_inner(model, name, ctx["pack"])
                scores.append(score)
                details.append({**ctx["scope"], "model": name, "arm": arm, "k": k,
                                "settings": json.dumps(settings, sort_keys=True), "features": json.dumps(columns[inner]),
                                **score, "fit_seconds": fit_seconds})
            candidates.append({"model": name, "arm": arm, "k": k, "settings": settings,
                               "gamma_index": gamma_index,
                               **{f"mean_{key}": float(np.mean([s[key] for s in scores]))
                                  for key in ("objective", "clean_ba", "clean_auc")}})
    if not candidates:
        raise ValueError("No valid frozen candidate in all three inner-training folds")
    return max(candidates, key=candidate_key), candidates, details


def require_pair_metadata(left, right):
    a, b = left.sort_values("Case"), right.sort_values("Case")
    if a.Case.duplicated().any() or b.Case.duplicated().any():
        raise ValueError("Duplicate patient in arm comparison")
    if not np.array_equal(a[["Case", "actual", "outer_fold", "n_images"]], b[["Case", "actual", "outer_fold", "n_images"]]):
        raise ValueError("Patient metadata differs across comparison arms")


def arm_comparisons(predictions):
    rows = []
    for name in MODEL_NAMES:
        for condition in COHERENT_CONDITIONS:
            groups = [predictions.loc[predictions.model.eq(name) & predictions.arm.eq(arm)
                                     & predictions.condition.eq(condition)] for arm in ARMS]
            require_pair_metadata(*groups)
            a, b = [summarize_pair(p, model=name, condition=condition) for p in groups]
            rows.append({"model": name, "condition": condition, "patients": b["eligible_patients"],
                         **{f"{key}_subset_minus_all23": b[key] - a[key] for key in
                            ("clean_balanced_accuracy", "pert_balanced_accuracy", "delta_balanced_accuracy",
                             "pert_sensitivity", "pert_specificity", "flip_rate")},
                         "new_false_negatives_all23": a["new_false_negatives"],
                         "new_false_negatives_subset": b["new_false_negatives"]})
    return pd.DataFrame(rows)


def nested_evaluate(dev, coherent, protocol):
    clean_parts, paired_parts, selections, scopes, rankings = [], [], [], [], []
    candidate_rows, inner_rows, fold_clean, fold_pairs = [], [], [], []
    begin = perf_counter()
    for outer in range(5):
        contexts, audit, ranks = build_contexts(dev, coherent, outer)
        scopes.extend(audit)
        rankings.extend(ranks)
        train, val = dev.loc[dev.dev_fold.ne(outer)], dev.loc[dev.dev_fold.eq(outer)]
        ranking = ss.rank_training_features(dev, coherent, training_cases=set(train.Case), forbidden_cases=set(val.Case))
        rankings.append(ranking.assign(outer_fold=outer, inner_fold=-1, context="outer_refit_only"))
        scopes.append({"outer_fold": outer, "inner_fold": -1, "context": "outer_refit_only",
                       "training_case_sha256": ss.case_digest(set(train.Case)), "excluded_case_sha256": ss.case_digest(set(val.Case)),
                       "training_patients": train.Case.nunique(), "excluded_patients": val.Case.nunique()})
        for name in MODEL_NAMES:
            for arm in ARMS:
                start = perf_counter()
                chosen, candidates, details = tune_arm(contexts, name, arm, protocol)
                tuning_seconds = perf_counter() - start
                columns = FEATURE_COLUMNS if arm == ARMS[0] else ss.selected_columns(ranking, chosen["k"])
                start = perf_counter()
                model = fit_subset(train, name, columns, chosen["settings"])
                refit_seconds = perf_counter() - start
                selection = {"outer_fold": outer, **chosen, "features": list(columns),
                             "training_case_sha256": ss.case_digest(set(train.Case)),
                             "tuning_seconds": tuning_seconds, "outer_refit_seconds": refit_seconds}
                selections.append(selection)
                for record in candidates:
                    candidate_rows.append({"outer_fold": outer, **record, "selected": record == chosen})
                inner_rows.extend(details)
                baseline = _predict(model, val, name, outer).rename(columns={
                    "prob_malignant" if name == "logreg" else "decision_score": "score"})
                baseline["model"], baseline["arm"] = name, arm
                clean_parts.append(baseline)
                fold_clean.append({"outer_fold": outer, "model": name, "arm": arm,
                                   **metrics(baseline, "score", "predicted")})
                for condition in COHERENT_CONDITIONS:
                    variants = coherent.loc[coherent.dev_fold.eq(outer) & coherent.condition.eq(condition)]
                    paired = compare_one_condition(model, name, val, variants, outer)
                    paired["arm"] = arm
                    paired_parts.append(paired)
                    fold_pairs.append({"outer_fold": outer, "arm": arm,
                                       **summarize_pair(paired, model=name, condition=condition)})
                print(f"Outer {outer} | {name} | {arm}: k={chosen['k']} | "
                      f"inner objective={chosen['mean_objective']:.4f} | clean training only", flush=True)
    clean, pairs = pd.concat(clean_parts, ignore_index=True), pd.concat(paired_parts, ignore_index=True)
    n = dev.Case.nunique()
    if (len(clean) != n * 4 or len(pairs) != n * 16
            or clean.duplicated(["model", "arm", "Case"]).any()
            or pairs.duplicated(["model", "arm", "condition", "Case"]).any() or not pairs.eligible.all()):
        raise RuntimeError("Incomplete/duplicate OOF patient predictions")
    pooled_pairs = pd.DataFrame([{"arm": arm, **summarize_pair(part, model=name, condition=condition)}
                                for (name, arm, condition), part in pairs.groupby(["model", "arm", "condition"])])
    pooled_clean = []
    for (name, arm), part in clean.groupby(["model", "arm"]):
        clean_metrics = metrics(part, "score", "predicted")
        changed = pooled_pairs.loc[pooled_pairs.model.eq(name) & pooled_pairs.arm.eq(arm)]
        pooled_clean.append({"model": name, "arm": arm, "patients": len(part), **clean_metrics,
                             "mean_five_condition_ba": float((clean_metrics["balanced_accuracy"] + changed.pert_balanced_accuracy.sum()) / 5)})
    frequency = [{"model": name, "arm": arm, "feature": feature,
                  "selected_outer_folds": sum(s["model"] == name and s["arm"] == arm and feature in s["features"] for s in selections)}
                 for name in MODEL_NAMES for arm in ARMS for feature in FEATURE_COLUMNS]
    flat_candidates = [{**r, "settings": json.dumps(r["settings"], sort_keys=True)} for r in candidate_rows]
    return {"tables": {"clean_oof_predictions.csv": clean, "coherent_paired_predictions.csv": pairs,
                       "clean_summary.csv": pd.DataFrame(pooled_clean), "coherent_summary.csv": pooled_pairs,
                       "clean_outer_fold_metrics.csv": pd.DataFrame(fold_clean),
                       "coherent_outer_fold_metrics.csv": pd.DataFrame(fold_pairs),
                       "arm_comparison.csv": arm_comparisons(pairs),
                       "inner_candidate_summary.csv": pd.DataFrame(flat_candidates),
                       "inner_fold_scores.csv": pd.DataFrame(inner_rows),
                       "recomputed_training_rankings.csv": pd.concat(rankings, ignore_index=True),
                       "feature_selection_frequency.csv": pd.DataFrame(frequency)},
            "selections": selections, "scopes": scopes,
            "fit_counts": {"inner": len(inner_rows), "outer_refits": len(selections)},
            "elapsed_seconds": perf_counter() - begin}


def load_frozen_design(directory, fingerprints):
    protocol_path, summary_path = directory / "selection_protocol.json", directory / "selection_design_summary.json"
    protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if (summary.get("status") != "design_frozen_and_training_scopes_audited"
            or summary.get("protocol_sha256") != sha256(protocol_path)
            or summary.get("source_sha256") != sha256(Path(ss.__file__))
            or summary.get("held_out_evaluation_used") is not False
            or summary.get("all_scopes_patient_disjoint") is not True or summary.get("training_scopes") != 20
            or protocol != ss.locked_protocol()):
        raise ValueError("Step 18 frozen protocol/source verification failed")
    for key in ("clean_features", "manifest", "coherent_features", "geometry"):
        if summary.get("inputs", {}).get(key, {}).get("sha256") != fingerprints[key]["sha256"]:
            raise ValueError(f"Step 18 input hash mismatch: {key}")
    return protocol


def check_original_oof(dev, modeling):
    for name in MODEL_NAMES:
        settings = load_fold_settings(modeling / f"{name}_nested_cv_folds.csv", name)
        pieces = []
        for fold in range(5):
            train, val = dev.loc[dev.dev_fold.ne(fold)], dev.loc[dev.dev_fold.eq(fold)]
            model = fit_subset(train, name, FEATURE_COLUMNS, settings[fold])
            pieces.append(_predict(model, val, name, fold))
        validate_clean_reference(pd.concat(pieces, ignore_index=True), modeling / f"{name}_nested_cv_predictions.csv", name)
        print(f"Original {name.upper()} clean OOF: MATCHED", flush=True)


def run(clean_path, manifest, coherent_path, geometry, design, modeling, output):
    if output.exists():
        raise FileExistsError("Step 19 output exists; choose a fresh --output directory")
    paths = {"clean_features": clean_path, "manifest": manifest, "coherent_features": coherent_path, "geometry": geometry,
             "protocol": design / "selection_protocol.json", "design_summary": design / "selection_design_summary.json",
             **{f"{name}_{kind}": modeling / f"{name}_nested_cv_{kind}.csv"
                for name in MODEL_NAMES for kind in ("folds", "predictions")}}
    fingerprints = {key: {"path": str(path.resolve()), "sha256": sha256(path)} for key, path in paths.items()}
    protocol = load_frozen_design(design, fingerprints)
    dev = load_development(clean_path, manifest)
    validate_manifest_isolation(manifest, dev)
    coherent = load_coherent(coherent_path, dev)
    verify_coherent_geometry(coherent, pd.read_csv(geometry, dtype={"ID": str}), dev)
    print(f"STEP 19 — NESTED STABILITY-SELECTION EVALUATION\nDevelopment: {len(dev)} images | {dev.Case.nunique()} patients\n"
          "Frozen Step 18 design: VERIFIED | Held-out evaluation: NOT USED", flush=True)
    check_original_oof(dev, modeling)
    result = nested_evaluate(dev, coherent, protocol)
    if any(sha256(path) != fingerprints[key]["sha256"] for key, path in paths.items()):
        raise RuntimeError("Locked inputs changed during Step 19")
    summary = {"step": 19, "status": "exploratory_nested_development_evaluation",
               "development_images": len(dev), "development_patients": int(dev.Case.nunique()),
               "arms": list(ARMS), "model_names": list(MODEL_NAMES), "conditions": list(COHERENT_CONDITIONS),
               "protocol_sha256": fingerprints["protocol"]["sha256"], "inputs": fingerprints,
               "original_clean_oof": {name: "MATCHED" for name in MODEL_NAMES},
               "held_out_evaluation_used": False, "training": "clean partition only; inner training reranking; inner validation candidate scoring",
               "fit_counts": {**result["fit_counts"], "baseline_reproducibility": 10},
               "elapsed_nested_seconds": result["elapsed_seconds"],
               "clean_summary": result["tables"]["clean_summary.csv"].to_dict("records"),
               "source_sha256": sha256(Path(__file__)),
               "versions": {pkg: version(pkg) for pkg in ("numpy", "pandas", "scikit-learn", "scipy")},
               "limitations": "Post-hoc exploratory development analysis; no significance claim, clinical validation or final global model/feature selection."}
    output.mkdir(parents=True, exist_ok=False)
    for filename, table in result["tables"].items():
        table.to_csv(output / filename, index=False, float_format="%.12g")
    for filename, content in (("outer_selections.json", result["selections"]), ("training_scopes.json", result["scopes"]),
                              ("evaluation_summary.json", summary)):
        (output / filename).write_text(ss.canonical_json(content), encoding="utf-8")
    print(result["tables"]["clean_summary.csv"][["model", "arm", "balanced_accuracy", "sensitivity", "specificity", "roc_auc", "mean_five_condition_ba"]].to_string(index=False))
    print(f"Original clean OOF: MATCHED | Held-out evaluation: NOT USED\nSaved: {output}", flush=True)
    return summary


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--clean", type=Path, default=ROOT / "data/processed/development_features.csv")
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--coherent", type=Path, default=ROOT / "data/processed/development_coherent_features.csv")
    parser.add_argument("--geometry", type=Path, default=ROOT / "results/robustness/coherent_geometry_development.csv")
    parser.add_argument("--design", type=Path, default=ROOT / "results/robustness/selection_step18")
    parser.add_argument("--modeling", type=Path, default=ROOT / "results/modeling")
    parser.add_argument("--output", type=Path, default=ROOT / "results/robustness/selection_step19")
    args = parser.parse_args()
    run(args.clean, args.manifest, args.coherent, args.geometry, args.design, args.modeling, args.output)


if __name__ == "__main__":
    main()
```

**`src/stable_selection.py`**

```python
"""Step 18: freeze and audit training-only stability feature selection.

No classifier fitting, candidate choice or performance evaluation in this step.
Step 19 must recompute rankings within each inner TRAINING partition.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

from .coherent_eval import load_coherent, sha256, validate_manifest_isolation
from .coherent_features import COHERENT_CONDITIONS, verify_coherent_geometry
from .features import FEATURE_COLUMNS
from .models import ROOT, inner_patient_folds, load_development

TOP_K = (5, 10, 15)
SCALE_EPS = 1e-12


def locked_protocol() -> dict:
    """Design choices fixed before Step 19, not a retrospective preregistration."""
    return {
        "step": 18, "version": "stability_selection_v1",
        "status": "posthoc_experiment3_design_frozen_before_new_performance_evaluation",
        "feature_columns": list(FEATURE_COLUMNS), "conditions": list(COHERENT_CONDITIONS),
        "arms": ["all_23_matched_tuning", "stability_topk"], "top_k_candidates": list(TOP_K),
        "outer_folds": "same five frozen patient folds", "inner_folds": 3,
        "inner_seed": "42 + outer_fold", "global_feature_list_for_CV": "forbidden",
        "ranking_scale": "clean training-only patient-weighted population SD, no diagnosis weights",
        "image_weight": "1 / number of training images for that patient",
        "drift_aggregation": "mean absolute image delta within patient, then equal mean across training patients",
        "stability_score": "maximum of four condition-specific mean drifts divided by clean weighted SD; lower is better",
        "constant_policy": "exclude from subset arms if SD <= 1e-12 * max(1, abs(weighted mean)); retain in all-23 comparator",
        "ranking_tie_break": "canonical FEATURE_COLUMNS order", "scale_epsilon": SCALE_EPS,
        "candidate_shortage": "skip k above eligible feature count; fail subset arm if no k available; never pad or substitute",
        "classifier_training": "clean fitting-partition images only, selected columns, patient-weighted scaling/classifier",
        "logreg": {"C": [0.01, 0.1, 1.0, 10.0, 100.0], "balance": [False, True]},
        "svm": {"C": [0.1, 1.0, 10.0], "gamma": ["scale", 0.01, 0.1], "balance": [False, True]},
        "thresholds": {"logreg_mean_probability": 0.5, "svm_mean_margin": 0.0},
        "inner_selection_objective": "mean over three inner folds of the equal mean of five patient BAs: clean plus four coherent conditions",
        "selection_tie_break": ["higher objective", "higher mean inner clean BA", "higher mean inner clean AUC",
                                "fewer features", "smaller C", "balance=False", "earlier gamma in fixed list"],
        "fair_comparator": "retune all-23 arm with identical inner folds, hyperparameter grids and objective; retain Steps 14/16 baselines separately",
        "outer_reports": ["clean patient metrics", "four coherent paired robustness metrics", "feature selections per outer fold", "fit time"],
        "inner_validation_perturbations": "allowed only to score inner candidates; never for ranking, scaling or classifier fit",
        "outer_validation": "excluded from ranking, scaling, classifier fitting and all parameter/candidate choices",
        "test": "held-out features/images/predictions forbidden until final frozen evaluation",
        "original_protocol": "preserved; not used for ranking or candidate tuning",
        "prohibited": ["dropping a named feature after viewing development results", "selecting k on outer OOF results",
                       "using Step 18 outer rankings inside inner CV", "global ranking across development patients"],
        "interpretation": "exploratory because earlier development results have been inspected; no guarantee of improvement or clinical realism",
    }


def canonical_json(value) -> str:
    return json.dumps(value, sort_keys=True, indent=2, allow_nan=False) + "\n"


def case_digest(cases) -> str:
    return hashlib.sha256(json.dumps(sorted(int(c) for c in cases)).encode()).hexdigest()


def rank_training_features(clean: pd.DataFrame, coherent: pd.DataFrame, *,
                           training_cases: set, forbidden_cases: set) -> pd.DataFrame:
    """Slice training patients FIRST: excluded feature values cannot affect fit."""
    training_cases, forbidden_cases = set(training_cases), set(forbidden_cases)
    if not training_cases or training_cases & forbidden_cases:
        raise ValueError("Empty training scope or overlap with forbidden patients")
    train = clean.loc[clean.Case.isin(training_cases)].copy()
    pert = coherent.loc[coherent.Case.isin(training_cases)].copy()
    if set(train.Case) != training_cases or set(pert.Case) != training_cases:
        raise ValueError("Missing training patients")
    if (train.ID.duplicated().any() or not train.split.eq("development").all()
            or not pert.split.eq("development").all()
            or pert.duplicated(["ID", "condition"]).any()
            or set(pert.condition) != set(COHERENT_CONDITIONS)
            or len(pert) != len(train) * 4 or not pert.valid.eq(True).all()):
        raise ValueError("Invalid training feature coverage/availability")
    train = train.sort_values("ID").reset_index(drop=True)
    matrix = train[list(FEATURE_COLUMNS)].to_numpy(float)
    if not np.isfinite(matrix).all():
        raise ValueError("Nonfinite clean training features")
    weights = (1.0 / train.groupby("Case").ID.transform("size")).to_numpy(float)
    mean = np.average(matrix, axis=0, weights=weights)
    sd = np.sqrt(np.average((matrix - mean) ** 2, axis=0, weights=weights))
    eligible = sd > SCALE_EPS * np.maximum(1.0, np.abs(mean))
    result = pd.DataFrame({"feature": FEATURE_COLUMNS, "canonical_order": range(len(FEATURE_COLUMNS)),
                           "clean_weighted_mean": mean, "clean_weighted_sd": sd, "eligible": eligible})
    condition_scores = []
    for condition in COHERENT_CONDITIONS:
        part = pert.loc[pert.condition.eq(condition)]
        if len(part) != len(train) or set(part.ID) != set(train.ID):
            raise ValueError("Missing training image-condition pairing")
        part = part.set_index("ID").loc[train.ID]
        for col in ("Case", "Pathology", "split", "dev_fold"):
            if not np.array_equal(part[col], train[col]):
                raise ValueError(f"Training perturbation metadata mismatch: {col}")
        alternative = part[list(FEATURE_COLUMNS)].to_numpy(float)
        if not np.isfinite(alternative).all():
            raise ValueError("Nonfinite coherent training features")
        delta = pd.DataFrame(np.abs(alternative - matrix), columns=FEATURE_COLUMNS)
        delta["Case"] = train.Case.to_numpy()
        patient_mean = delta.groupby("Case")[list(FEATURE_COLUMNS)].mean().mean().to_numpy(float)
        normalized = np.full(len(FEATURE_COLUMNS), np.nan)
        np.divide(patient_mean, sd, out=normalized, where=eligible)
        result[f"drift_{condition}"] = normalized
        condition_scores.append(normalized)
    result["stability_score"] = np.max(np.array(condition_scores), axis=0)
    ordered = result.loc[result.eligible].sort_values(["stability_score", "canonical_order"], kind="stable")
    result["rank"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    result.loc[ordered.index, "rank"] = np.arange(1, len(ordered) + 1)
    result["exclusion_reason"] = np.where(eligible, "", "constant_or_near_constant_clean_training_feature")
    return result.sort_values(["rank", "canonical_order"], na_position="last").reset_index(drop=True)


def selected_columns(ranking: pd.DataFrame, k: int) -> tuple[str, ...]:
    if k not in TOP_K:
        raise ValueError("k outside frozen candidate grid")
    eligible = ranking.loc[ranking.eligible].sort_values("rank")
    if len(eligible) < k:
        raise ValueError("Not enough eligible training features for k")
    selected = set(eligible.head(k).feature)
    # Model column order remains canonical, independent of score ordering.
    return tuple(feature for feature in FEATURE_COLUMNS if feature in selected)


def audit_scopes(dev: pd.DataFrame, coherent: pd.DataFrame) -> tuple[pd.DataFrame, list[dict], list[dict]]:
    ranks, scopes, candidates = [], [], []
    all_cases = set(dev.Case)
    for outer in range(5):
        outer_train = dev.loc[dev.dev_fold.ne(outer)]
        outer_val_cases = set(dev.loc[dev.dev_fold.eq(outer), "Case"])
        contexts = [("outer_refit_only", -1, outer_train, outer_val_cases)]
        for inner, (train, val) in enumerate(inner_patient_folds(outer_train, seed=42 + outer)):
            contexts.append(("inner_training", inner, train, outer_val_cases | set(val.Case)))
        for context, inner, train, forbidden in contexts:
            training_cases = set(train.Case)
            if training_cases | forbidden != all_cases:
                raise ValueError("Training and excluded scopes must partition development patients")
            ranking = rank_training_features(dev, coherent, training_cases=training_cases, forbidden_cases=forbidden)
            allowed = [k for k in TOP_K if int(ranking.eligible.sum()) >= k]
            if not allowed:
                raise ValueError("No frozen top-k candidate has sufficient eligible features")
            ranking.insert(0, "inner_fold", inner)
            ranking.insert(0, "outer_fold", outer)
            ranking.insert(0, "context", context)
            ranks.append(ranking)
            scopes.append({"context": context, "outer_fold": outer, "inner_fold": inner,
                           "training_patients": len(training_cases), "excluded_patients": len(forbidden),
                           "training_case_sha256": case_digest(training_cases), "excluded_case_sha256": case_digest(forbidden),
                           "training_cases": sorted(int(c) for c in training_cases),
                           "excluded_cases": sorted(int(c) for c in forbidden), "overlap": 0})
            for k in TOP_K:
                valid = k in allowed
                candidates.append({"context": context, "outer_fold": outer, "inner_fold": inner, "k": k,
                                   "valid": valid, "features": list(selected_columns(ranking, k)) if valid else [],
                                   "reason": "" if valid else "fewer_eligible_features_than_k"})
        print(f"Outer fold {outer}: outer-training scope and three inner-training scopes audited", flush=True)
    return pd.concat(ranks, ignore_index=True), scopes, candidates


def run(clean_path: Path, manifest: Path, coherent_path: Path, geometry_path: Path, output: Path) -> dict:
    if output.exists():
        raise FileExistsError("Step 18 output exists; choose a fresh --output directory")
    paths = {"clean_features": clean_path, "manifest": manifest, "coherent_features": coherent_path,
             "geometry": geometry_path}
    fingerprints = {key: {"path": str(path.resolve()), "sha256": sha256(path)} for key, path in paths.items()}
    dev = load_development(clean_path, manifest)
    validate_manifest_isolation(manifest, dev)
    coherent = load_coherent(coherent_path, dev)
    verify_coherent_geometry(coherent, pd.read_csv(geometry_path, dtype={"ID": str}), dev)
    protocol = locked_protocol()
    frozen_text = canonical_json(protocol)
    print("STEP 18 — TRAINING-ONLY STABILITY SELECTION DESIGN\n"
          f"Development: {len(dev)} images | {dev.Case.nunique()} patients | {len(FEATURE_COLUMNS)} features", flush=True)
    rankings, scopes, candidates = audit_scopes(dev, coherent)
    if len(rankings) != 20 * len(FEATURE_COLUMNS) or len(scopes) != 20 or len(candidates) != 60:
        raise RuntimeError("Incomplete training scope audit")
    if any(sha256(path) != fingerprints[key]["sha256"] for key, path in paths.items()):
        raise RuntimeError("Locked inputs changed during audit")
    result = {"step": 18, "status": "design_frozen_and_training_scopes_audited",
              "development_images": len(dev), "development_patients": int(dev.Case.nunique()),
              "training_scopes": len(scopes), "ranking_rows": len(rankings), "candidate_rows": len(candidates),
              "classifier_fits": 0, "performance_evaluated": False, "held_out_evaluation_used": False,
              "global_ranking_produced": False, "all_scopes_patient_disjoint": True,
              "protocol_sha256": hashlib.sha256(frozen_text.encode()).hexdigest(),
              "source_sha256": sha256(Path(__file__)), "inputs": fingerprints,
              "next_step": "19: nested training-only reranking, inner candidate selection, outer paired evaluation"}
    output.mkdir(parents=True, exist_ok=False)
    (output / "selection_protocol.json").write_bytes(frozen_text.encode("utf-8"))
    rankings.to_csv(output / "training_scope_rankings.csv", index=False, float_format="%.12g")
    for name, content in (("training_scopes.json", scopes), ("candidate_subsets.json", candidates), ("selection_design_summary.json", result)):
        (output / name).write_text(canonical_json(content), encoding="utf-8")
    print(f"Frozen top-k candidates: {TOP_K}; matched all-23 comparator\n"
          f"Patient-disjoint scopes: {len(scopes)} | Classifier fits: 0 | Held-out evaluation: NOT USED\nSaved: {output}")
    return result


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--clean", type=Path, default=ROOT / "data/processed/development_features.csv")
    parser.add_argument("--manifest", type=Path, default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--coherent", type=Path, default=ROOT / "data/processed/development_coherent_features.csv")
    parser.add_argument("--geometry", type=Path, default=ROOT / "results/robustness/coherent_geometry_development.csv")
    parser.add_argument("--output", type=Path, default=ROOT / "results/robustness/selection_step18")
    args = parser.parse_args()
    run(args.clean, args.manifest, args.coherent, args.geometry, args.output)


if __name__ == "__main__":
    main()
```

**`src/svm_model.py`**

```python
"""BUS-BRA RBF SVM with patient-level nested cross-validation.

Run from the repository root:
    .\\.venv\\Scripts\\python.exe -m src.svm_model

Uses the SAME 23 development-only features, locked outer patient folds and
patient-level evaluation as src.models (Logistic Regression). Tunable parameters
are chosen within three inner patient folds. The held-out test is never read.

IMPORTANT: SVC(probability=False) deliberately avoids libsvm's internal,
image-level probability cross-validation. Raw SVM decision values are NOT
calibrated malignancy probabilities. At this stage, 0 is a provisional decision
threshold; patient predictions average decision values across available views.
Probability calibration and threshold tuning are reserved for a later,
patient-group-safe evaluation protocol.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from .features import FEATURE_COLUMNS
from .models import (
    ROOT, SEED, NUM_OUTER_FOLDS, NUM_INNER_FOLDS,
    load_development, patient_image_weights, inner_patient_folds,
)

CS = (0.1, 1.0, 10.0)
GAMMAS = ("scale", 0.01, 0.1)
BALANCE_OPTIONS = (False, True)


def build_svm(C: float, gamma: str | float) -> Pipeline:
    """Training-fold weighted scaling followed by RBF SVM, no hidden CV."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(kernel="rbf", C=float(C), gamma=gamma)),
    ])


def fit_svm(train: pd.DataFrame, *, C: float, gamma: str | float,
            balance: bool) -> Pipeline:
    if train["Pathology"].nunique() != 2:
        raise ValueError("Training fold must contain both classes")
    weights = patient_image_weights(train, balance=balance)
    X = train[list(FEATURE_COLUMNS)].to_numpy(dtype=float)
    y = train["Pathology"].eq("malignant").astype(int).to_numpy()
    model = build_svm(C, gamma)
    model.fit(X, y, scaler__sample_weight=weights,
              classifier__sample_weight=weights)
    return model


def patient_svm_predictions(model: Pipeline, rows: pd.DataFrame,
                            outer_fold: int | None = None) -> pd.DataFrame:
    """One patient-level score: mean SVM decision margin over their images."""
    if len(rows) == 0:
        raise ValueError("Cannot predict an empty partition")
    scores = model.decision_function(rows[list(FEATURE_COLUMNS)].to_numpy(dtype=float))
    frame = pd.DataFrame({
        "Case": rows["Case"].to_numpy(),
        "actual": rows["Pathology"].eq("malignant").astype(int).to_numpy(),
        "decision_score": np.asarray(scores, dtype=float),
    })
    if (frame.groupby("Case")["actual"].nunique() != 1).any():
        raise ValueError("Inconsistent patient labels in prediction set")
    patients = frame.groupby("Case", as_index=False).agg(
        actual=("actual", "first"),
        decision_score=("decision_score", "mean"),
        n_images=("decision_score", "size"),
    )
    patients["predicted"] = (patients["decision_score"] >= 0.0).astype(int)
    if outer_fold is not None:
        patients["outer_fold"] = int(outer_fold)
    return patients


def score_svm_patients(patients: pd.DataFrame) -> dict[str, float | int]:
    y = patients["actual"].to_numpy(dtype=int)
    yhat = patients["predicted"].to_numpy(dtype=int)
    score = patients["decision_score"].to_numpy(dtype=float)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Validation patients must contain both classes")
    if not np.isfinite(score).all():
        raise ValueError("Nonfinite SVM decision scores")
    tn, fp, fn, tp = map(int, confusion_matrix(y, yhat, labels=[0, 1]).ravel())
    return {
        "patients": len(patients),
        "balanced_accuracy": float(balanced_accuracy_score(y, yhat)),
        "sensitivity": float(tp / (tp + fn)),
        "specificity": float(tn / (tn + fp)),
        "roc_auc": float(roc_auc_score(y, score)),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def choose_svm_settings(train: pd.DataFrame, *, seed: int = SEED,
                        cs=CS, gammas=GAMMAS,
                        balances=BALANCE_OPTIONS,
                        n_splits: int = NUM_INNER_FOLDS) -> dict:
    """Rank configurations on independent, inner-validation patients only."""
    inner = list(inner_patient_folds(train, seed=seed, n_splits=n_splits))
    ranking = []
    for C in cs:
        for gamma in gammas:
            for balance in balances:
                bas, aucs = [], []
                for fitting, validation in inner:
                    model = fit_svm(fitting, C=C, gamma=gamma, balance=balance)
                    metric = score_svm_patients(patient_svm_predictions(model, validation))
                    bas.append(metric["balanced_accuracy"])
                    aucs.append(metric["roc_auc"])
                ranking.append({
                    "C": float(C),
                    "gamma": gamma,
                    "balance": bool(balance),
                    "mean_inner_ba": float(np.mean(bas)),
                    "mean_inner_auc": float(np.mean(aucs)),
                })
    # Deterministic tie-break: BA, AUC, smaller C, 'scale' gamma, no balancing.
    return max(ranking, key=lambda r: (
        r["mean_inner_ba"], r["mean_inner_auc"],
        -r["C"], int(r["gamma"] == "scale"), -int(r["balance"]),
    ))


def nested_svm_cv(development: pd.DataFrame, *, cs=CS, gammas=GAMMAS,
                  balances=BALANCE_OPTIONS) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    predicted_folds, summaries = [], []
    begin = perf_counter()
    for outer in range(NUM_OUTER_FOLDS):
        train = development[development["dev_fold"].ne(outer)]
        validation = development[development["dev_fold"].eq(outer)]
        if not set(train["Case"]).isdisjoint(set(validation["Case"])):
            raise RuntimeError("Patient leakage across outer folds")
        selected = choose_svm_settings(
            train, seed=SEED + outer, cs=cs,
            gammas=gammas, balances=balances,
        )
        model = fit_svm(train, C=selected["C"], gamma=selected["gamma"],
                        balance=selected["balance"])
        predicted = patient_svm_predictions(model, validation, outer_fold=outer)
        stats = score_svm_patients(predicted)
        predicted_folds.append(predicted)
        summaries.append({"outer_fold": outer, **selected, **stats})
        print(f"Outer fold {outer}: {stats['patients']} patients | "
              f"C={selected['C']:g}, gamma={selected['gamma']}, "
              f"balanced={selected['balance']} | "
              f"BA={stats['balanced_accuracy']:.3f}, "
              f"AUC={stats['roc_auc']:.3f}, FN={stats['fn']}", flush=True)
    oof = pd.concat(predicted_folds, ignore_index=True).sort_values("Case").reset_index(drop=True)
    if oof["Case"].duplicated().any() or len(oof) != development["Case"].nunique():
        raise RuntimeError("OOF predictions must contain every development patient once")
    overall = score_svm_patients(oof)
    overall.update({
        "method": "nested patient-level CV (5 locked outer folds, 3 inner folds)",
        "features": len(FEATURE_COLUMNS),
        "threshold": 0.0,
        "threshold_status": "provisional raw margin (not calibrated probability)",
        "score_type": "SVC decision_function margin, patient-mean",
        "probability_calibration": "not performed; must use grouped CV later",
        "test_patients_used": 0,
        "elapsed_cv_seconds": round(perf_counter() - begin, 3),
    })
    return oof, pd.DataFrame(summaries), overall


def run(feature_csv: Path, manifest_csv: Path, output_dir: Path,
        model_dir: Path) -> dict:
    df = load_development(feature_csv, manifest_csv)
    print("BUS-BRA — RBF SVM CLASSIFIER")
    print("=" * 45)
    print(f"Development: {len(df)} images, {df['Case'].nunique()} patients, "
          f"{len(FEATURE_COLUMNS)} features")
    print("Protocol: 5 locked outer patient folds, 3 inner patient folds")
    print("SVM scores: raw decision margins; threshold 0 is provisional")
    print("Held-out test features and labels: NOT USED")
    oof, folds, result = nested_svm_cv(df)
    print(f"Overall OOF BA={result['balanced_accuracy']:.4f}, "
          f"sensitivity={result['sensitivity']:.4f}, "
          f"specificity={result['specificity']:.4f}, "
          f"AUC={result['roc_auc']:.4f}, FN={result['fn']}")

    # Select a provisional full-development reference model after honest OOF eval.
    chosen = choose_svm_settings(df, seed=SEED, n_splits=NUM_OUTER_FOLDS)
    reference = fit_svm(df, C=chosen["C"], gamma=chosen["gamma"],
                        balance=chosen["balance"])
    output_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    oof.to_csv(output_dir / "svm_nested_cv_predictions.csv", index=False)
    folds.to_csv(output_dir / "svm_nested_cv_folds.csv", index=False)
    result["development_reference_model_settings"] = chosen
    (output_dir / "svm_nested_cv_summary.json").write_text(
        json.dumps(result, indent=2) + "\n", encoding="utf-8")
    joblib.dump({
        "pipeline": reference,
        "features": list(FEATURE_COLUMNS),
        "settings": chosen,
        "status": "development_only_provisional_raw_decision_scores",
    }, model_dir / "svm_development_candidate.joblib")
    print(f"Saved evaluation results: {output_dir}")
    print(f"Saved provisional development-only model: "
          f"{model_dir / 'svm_development_candidate.joblib'}")
    return result


def main():
    parser = argparse.ArgumentParser(description="Patient-level nested CV RBF SVM")
    parser.add_argument("--features", type=Path,
                        default=ROOT / "data/processed/development_features.csv")
    parser.add_argument("--manifest", type=Path,
                        default=ROOT / "splits/image_splits.csv")
    parser.add_argument("--output", type=Path,
                        default=ROOT / "results/modeling")
    parser.add_argument("--models", type=Path, default=ROOT / "models")
    args = parser.parse_args()
    run(args.features, args.manifest, args.output, args.models)


if __name__ == "__main__":
    main()
```

**`src/visual_audit.py`**

```python
"""Generate a reproducible visual review sheet of BUS-BRA images and lesion masks.

Save as src/visual_audit.py and run from the project root:
    .\\.venv\\Scripts\\python.exe -m src.visual_audit

Reads data/raw/BUSBRA.zip directly and writes a contact sheet to
results/figures/busbra_visual_audit.png. Does not modify the source data.
"""
from __future__ import annotations

import io
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps

ROOT = Path(__file__).resolve().parents[1]
ZIP_PATH = ROOT / 'data' / 'raw' / 'BUSBRA.zip'
OUTPUT = ROOT / 'results' / 'figures' / 'busbra_visual_audit.png'

THUMB_W, THUMB_H = 288, 210
CELL_W, CELL_H = 310, 282
COLUMNS = 4


def choose_examples(metadata: pd.DataFrame) -> list[tuple[pd.Series, str]]:
    """Choose one image per case: 6 benign, 6 malignant, 4 metadata anomalies."""
    chosen: list[tuple[pd.Series, str]] = []
    seen_cases: set[str] = set()
    ordered = metadata.sort_values(['Case', 'ID'])
    for category in ('benign', 'malignant'):
        subset = ordered[ordered['Pathology'].astype(str).str.lower() == category]
        one_per_case = subset.drop_duplicates('Case')
        indices = np.linspace(0, len(one_per_case) - 1, 6, dtype=int)
        for row_idx in indices:
            row = one_per_case.iloc[int(row_idx)]
            case = str(row['Case'])
            if case not in seen_cases:
                chosen.append((row, 'routine sample'))
                seen_cases.add(case)
    with ZipFile(ZIP_PATH) as zf:
        for _, row in ordered.iterrows():
            image_path = f"BUSBRA/Images/{row['ID']}.png"
            with Image.open(io.BytesIO(zf.read(image_path))) as img:
                actual_w, actual_h = img.size
            if (actual_w, actual_h) != (int(row['Width']), int(row['Height'])):
                chosen.append((row, 'CSV dimension discrepancy'))
    return chosen


def make_tile(zf: ZipFile, row: pd.Series, reason: str) -> Image.Image:
    identifier = str(row['ID'])
    image_path = f'BUSBRA/Images/{identifier}.png'
    mask_path = f"BUSBRA/Masks/mask_{identifier.removeprefix('bus_')}.png"
    with Image.open(io.BytesIO(zf.read(image_path))) as f:
        img = f.convert('L')
    with Image.open(io.BytesIO(zf.read(mask_path))) as f:
        mask = f.convert('L')
    if img.size != mask.size:
        raise ValueError(f'Image/mask size mismatch for {identifier}')

    image_view = ImageOps.contain(img, (THUMB_W, THUMB_H), Image.Resampling.BILINEAR)
    mask_view = mask.resize(image_view.size, Image.Resampling.NEAREST)
    inside = np.asarray(mask_view) > 0
    a = np.asarray(image_view).copy()
    rgb = np.stack([a, a, a], axis=-1).astype(np.float32)
    # Transparent green fill inside the lesion; display-only, no changes to input data.
    rgb[inside] = 0.76 * rgb[inside] + 0.24 * np.array([40, 225, 80], dtype=np.float32)
    # Thin red outline of the lesion boundary.
    core = inside.copy()
    core[1:-1, 1:-1] = (
        inside[1:-1, 1:-1] & inside[:-2, 1:-1] & inside[2:, 1:-1]
        & inside[1:-1, :-2] & inside[1:-1, 2:]
    )
    border = inside & ~core
    rgb[border] = np.array([255, 55, 50], dtype=np.float32)
    overlay = Image.fromarray(np.clip(rgb, 0, 255).astype(np.uint8), 'RGB')

    tile = Image.new('RGB', (CELL_W, CELL_H), 'white')
    dx = (CELL_W - overlay.width) // 2
    tile.paste(overlay, (dx, 6))
    draw = ImageDraw.Draw(tile)
    y = THUMB_H + 12
    draw.text((10, y), f"{identifier} | Case {row['Case']} | {row['Pathology']}", fill='black')
    draw.text((10, y + 17), f"Mask area: {inside.mean():.1%}", fill='black')
    if reason == 'CSV dimension discrepancy':
        draw.text((10, y + 34), 'Metadata dimension discrepancy', fill=(155, 55, 15))
    else:
        draw.text((10, y + 34), 'Representative patient sample', fill=(80, 80, 80))
    return tile


def main() -> None:
    if not ZIP_PATH.exists():
        raise FileNotFoundError(f'Missing archive: {ZIP_PATH}')
    with ZipFile(ZIP_PATH) as zf:
        metadata = pd.read_csv(io.BytesIO(zf.read('BUSBRA/bus_data.csv')))
    selected = choose_examples(metadata)
    rows = (len(selected) + COLUMNS - 1) // COLUMNS
    sheet = Image.new('RGB', (CELL_W * COLUMNS, CELL_H * rows + 44), '#f0f1f3')
    draw = ImageDraw.Draw(sheet)
    draw.text((12, 14), 'BUS-BRA visual audit: green = mask interior; red = mask boundary', fill='black')
    with ZipFile(ZIP_PATH) as zf:
        for i, (row, reason) in enumerate(selected):
            tile = make_tile(zf, row, reason)
            sheet.paste(tile, ((i % COLUMNS) * CELL_W, 44 + (i // COLUMNS) * CELL_H))
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(OUTPUT)
    print(f'Generated visual audit for {len(selected)} image-mask pairs:')
    print(f'  {OUTPUT}')
    print('Review every tile: boundary placement, visible lesion, mask validity, possible quality issues.')
    print('Do not change labels or masks based on visual impression alone.')


if __name__ == '__main__':
    main()
```